In [ ]:
#por si se cachea el drive
from google.colab import drive
import os
import shutil

try:
    drive.flush_and_unmount()
    print("[OK] Drive desmontado.")
except Exception as exc:
    print("[INFO] No se pudo desmontar o no estaba montado:", exc)

if os.path.exists("/content/drive"):
    try:
        shutil.rmtree("/content/drive")
        print("[OK] Carpeta /content/drive eliminada.")
    except Exception as exc:
        print("[ADVERTENCIA] No se pudo eliminar /content/drive:", exc)

drive.mount("/content/drive", force_remount=True)

print("\nComprobaciones:")
print("Existe MyDrive:", os.path.exists("/content/drive/MyDrive"))
print("Existe carpeta clean:", os.path.exists("/content/drive/MyDrive/TFG_Glaucoma_CLEAN"))
print("Existe Refuge.zip:", os.path.exists("/content/drive/MyDrive/TFG_Glaucoma_CLEAN/Refuge.zip"))

print("\nContenido de TFG_Glaucoma_CLEAN:")
!ls -lah "/content/drive/MyDrive/TFG_Glaucoma_CLEAN"

# 1. Configuracion del entorno

Este bloque prepara el entorno de ejecucion, fija semillas y centraliza la configuracion del experimento. La variable `SM_FRAMEWORK` se define antes de importar `segmentation_models` para evitar inconsistencias de backend.

In [ ]:
# ============================================================
# 1. CONFIGURACIÓN DEL ENTORNO
# ============================================================

print("[INFO] Instalando dependencias principales...")
!pip install -q segmentation-models albumentations==1.3.1 openpyxl seaborn > /dev/null
print("[OK] Dependencias instaladas.")

# ------------------------------------------------------------
# Configuración previa obligatoria
# ------------------------------------------------------------

import os

# Debe definirse antes de importar segmentation_models.
os.environ["SM_FRAMEWORK"] = "tf.keras"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ------------------------------------------------------------
# Imports generales
# ------------------------------------------------------------

import gc
import glob
import random
import shutil
import zipfile
from dataclasses import dataclass
from datetime import datetime

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import segmentation_models as sm
import tensorflow as tf

from google.colab import drive

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from sklearn.model_selection import KFold

from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import (
    CSVLogger,
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    TensorBoard,
)

# ------------------------------------------------------------
# Funciones auxiliares de logging
# ------------------------------------------------------------

def log_info(message):
    print(f"[INFO] {message}")


def log_ok(message):
    print(f"[OK] {message}")


def log_warning(message):
    print(f"[ADVERTENCIA] {message}")


def log_error(message):
    print(f"[ERROR] {message}")


def log_critical(message):
    print(f"[CRITICO] {message}")


# ------------------------------------------------------------
# Configuración global del proyecto
# ------------------------------------------------------------

@dataclass(frozen=True)
class Config:
    SEED: int = 42
    IMG_SIZE: int = 512
    ROI_RADIUS: int = 200
    BATCH_SIZE: int = 8
    EPOCHS: int = 50
    BACKBONE: str = "inceptionresnetv2"
    N_SPLITS: int = 5
    CLASSES: int = 3
    LR_START: float = 1e-4

    # Carpeta principal limpia en Google Drive.
    DRIVE_BASE: str = "/content/drive/MyDrive/TFG_Glaucoma_CLEAN"

    # Dataset.
    DATASET_FOLDER_NAME: str = "Refuge"
    ZIP_NAME: str = "Refuge.zip"

    # Ruta local usada por TODO el notebook.
    # La celda 2 enlazará esta ruta a la carpeta Refuge de Drive.
    BASE_PATH: str = "/content/dataset_local/Refuge"
    EXTRACT_PATH: str = "/content/dataset_local"

    # Modelos y resultados.
    SAVE_PATH: str = "/content/drive/MyDrive/TFG_Glaucoma_CLEAN/Models_v5_TverskyCup"  # v5: reentreno Tversky-copa (v4 conservado en Models_vPro_Fixed)

    # Umbral inicial. Más adelante se calibra con validación externa.
    CLINICAL_THRESHOLD: float = 0.52

    # Mantener False. El ZIP solo se usa como fallback si no existe la carpeta Refuge.
    FORCE_REEXTRACT: bool = False

    @property
    def drive_dataset_folder_path(self):
        return os.path.join(self.DRIVE_BASE, self.DATASET_FOLDER_NAME)

    @property
    def drive_zip_path(self):
        return os.path.join(self.DRIVE_BASE, self.ZIP_NAME)

    @property
    def local_zip_path(self):
        return f"/content/{self.ZIP_NAME}"


CFG = Config()


# ------------------------------------------------------------
# Montaje robusto de Google Drive
# ------------------------------------------------------------

def mount_drive_safely():
    """
    Monta Google Drive antes de crear carpetas en SAVE_PATH.

    Esto evita crear accidentalmente una falsa carpeta local:
        /content/drive/MyDrive/...
    cuando Drive todavía no está montado.
    """

    expected_project_path = CFG.DRIVE_BASE

    # Caso correcto: Drive ya está montado y la carpeta del proyecto existe.
    if os.path.exists(expected_project_path):
        log_ok(f"Google Drive ya está montado. Proyecto detectado: {expected_project_path}")
        return

    mount_point = "/content/drive"

    # Si /content/drive existe con contenido pero no está montado correctamente,
    # se aparta para permitir el montaje real.
    if os.path.exists(mount_point) and len(os.listdir(mount_point)) > 0:
        backup_path = f"/content/drive_backup_before_mount_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        shutil.move(mount_point, backup_path)
        log_warning(f"/content/drive contenía archivos antes del montaje. Se ha movido a: {backup_path}")

    log_info("Montando Google Drive.")
    drive.mount(mount_point, force_remount=True)

    if not os.path.exists(expected_project_path):
        raise FileNotFoundError(
            f"Drive se ha montado, pero no se encuentra la carpeta del proyecto: {expected_project_path}"
        )

    log_ok(f"Google Drive montado correctamente. Proyecto detectado: {expected_project_path}")


mount_drive_safely()


# ------------------------------------------------------------
# Reproducibilidad y configuración de TensorFlow
# ------------------------------------------------------------

def set_environment(seed=CFG.SEED):
    """Fija semillas y configura TensorFlow para mejorar la reproducibilidad."""
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        policy = mixed_precision.Policy("mixed_float16")
        mixed_precision.set_global_policy(policy)
        log_ok(f"Mixed precision activado con compute_dtype={policy.compute_dtype}.")
    except Exception as exc:
        log_warning(f"No se pudo activar mixed precision. Se usará float32. Detalle: {exc}")

    gpus = tf.config.experimental.list_physical_devices("GPU")

    if len(gpus) == 0:
        log_warning("No se ha detectado GPU. El entrenamiento será mucho más lento.")
    else:
        log_ok(f"GPU detectada: {len(gpus)} dispositivo(s).")

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            log_warning(f"No se pudo configurar memory growth en GPU. Detalle: {exc}")

    # Ahora sí es seguro crear carpetas en Drive porque ya está montado.
    os.makedirs(CFG.SAVE_PATH, exist_ok=True)
    os.makedirs(os.path.join(CFG.SAVE_PATH, "logs"), exist_ok=True)

    log_ok(f"Semilla global fijada: {seed}.")
    log_ok(f"Ruta base en Drive: {CFG.DRIVE_BASE}")
    log_ok(f"Carpeta del dataset en Drive: {CFG.drive_dataset_folder_path}")
    log_ok(f"Ruta local del dataset: {CFG.BASE_PATH}")
    log_ok(f"Ruta de modelos: {CFG.SAVE_PATH}")


set_environment()

# Preprocesamiento específico del backbone seleccionado.
preprocess_input = sm.get_preprocessing(CFG.BACKBONE)

log_ok("Configuración del entorno completada.")

# 2. Carga y preparacion del dataset

Se mantiene la estructura original en Google Drive y en REFUGE. Antes de indexar imagenes y mascaras se validan las carpetas criticas para detectar problemas de montaje o descompresion de forma temprana.

In [ ]:
# ============================================================
# 2. CARGA Y PREPARACIÓN DEL DATASET
# ============================================================

def count_files_by_extensions(root_dir, extensions):
    """Cuenta archivos dentro de una carpeta usando varias extensiones."""
    files = []

    for ext in extensions:
        files.extend(
            glob.glob(
                os.path.join(root_dir, "**", f"*{ext}"),
                recursive=True
            )
        )
        files.extend(
            glob.glob(
                os.path.join(root_dir, "**", f"*{ext.upper()}"),
                recursive=True
            )
        )

    return len(set(files))


def get_required_dataset_paths():
    """Devuelve las rutas que usará el resto del notebook."""
    return {
        "train_images": os.path.join(
            CFG.BASE_PATH,
            "REFUGE-Training400",
            "Training400"
        ),
        "train_masks": os.path.join(
            CFG.BASE_PATH,
            "Annotation-Training400",
            "Disc_Cup_Masks"
        ),
        "val_images": os.path.join(
            CFG.BASE_PATH,
            "REFUGE-Validation400",
            "REFUGE-Validation400"
        ),
        "val_masks": os.path.join(
            CFG.BASE_PATH,
            "REFUGE-Validation400-GT",
            "REFUGE-Validation400-GT",
            "Disc_Cup_Masks"
        ),
        "val_labels": os.path.join(
            CFG.BASE_PATH,
            "REFUGE-Validation400-GT",
            "REFUGE-Validation400-GT",
            "Fovea_locations.xlsx"
        ),
        "test_images": os.path.join(
            CFG.BASE_PATH,
            "Test400"
        ),
        "test_masks": os.path.join(
            CFG.BASE_PATH,
            "REFUGE-Test-GT",
            "Disc_Cup_Masks"
        ),
        "test_labels": os.path.join(
            CFG.BASE_PATH,
            "REFUGE-Test-GT",
            "Glaucoma_label_and_Fovea_location.xlsx"
        ),
    }


def is_refuge_complete_folder(path):
    """
    Comprueba si una carpeta tiene la estructura REFUGE completa esperada.

    Estructura principal:
        REFUGE-Training400/Training400
        Annotation-Training400/Disc_Cup_Masks
        REFUGE-Validation400/REFUGE-Validation400
        REFUGE-Validation400-GT/REFUGE-Validation400-GT/Disc_Cup_Masks
        REFUGE-Validation400-GT/REFUGE-Validation400-GT/Fovea_locations.xlsx
    """

    required = [
        os.path.join(path, "REFUGE-Training400", "Training400"),
        os.path.join(path, "Annotation-Training400", "Disc_Cup_Masks"),
        os.path.join(path, "REFUGE-Validation400", "REFUGE-Validation400"),
        os.path.join(
            path,
            "REFUGE-Validation400-GT",
            "REFUGE-Validation400-GT",
            "Disc_Cup_Masks"
        ),
        os.path.join(
            path,
            "REFUGE-Validation400-GT",
            "REFUGE-Validation400-GT",
            "Fovea_locations.xlsx"
        ),
    ]

    return all(os.path.exists(p) for p in required)


def remove_local_base_path_if_needed():
    """
    Limpia CFG.BASE_PATH si contiene una extracción local antigua.

    Seguridad:
        - Solo borra dentro de /content/dataset_local.
        - No borra nada de Google Drive.
    """

    # Un symlink se elimina siempre de forma segura: solo borra el enlace, nunca el destino.
    # Se comprueba ANTES del guard de realpath para que los re-runs sean idempotentes:
    # al reiniciar el kernel /content sobrevive y el enlace a Drive seguiria presente.
    if os.path.islink(CFG.BASE_PATH):
        os.unlink(CFG.BASE_PATH)
        log_ok(f"Enlace local anterior eliminado: {CFG.BASE_PATH}")
        return

    if not os.path.exists(CFG.BASE_PATH):
        return

    if not os.path.realpath(CFG.BASE_PATH).startswith("/content/dataset_local"):
        raise RuntimeError(
            f"Por seguridad no se borra CFG.BASE_PATH porque no está dentro de /content/dataset_local: {CFG.BASE_PATH}"
        )

    shutil.rmtree(CFG.BASE_PATH)
    log_ok(f"Extracción local anterior eliminada: {CFG.BASE_PATH}")


def link_drive_refuge_folder_to_local():
    """
    Crea un enlace simbólico local:

        /content/dataset_local/Refuge
        → /content/drive/MyDrive/TFG_Glaucoma_CLEAN/Refuge

    Así el resto del notebook puede seguir usando CFG.BASE_PATH.
    """

    drive_refuge_path = CFG.drive_dataset_folder_path

    if not os.path.exists(drive_refuge_path):
        raise FileNotFoundError(
            f"No existe la carpeta Refuge en Drive: {drive_refuge_path}"
        )

    if not is_refuge_complete_folder(drive_refuge_path):
        raise FileNotFoundError(
            "La carpeta Refuge existe, pero no tiene la estructura completa esperada."
        )

    os.makedirs(os.path.dirname(CFG.BASE_PATH), exist_ok=True)

    remove_local_base_path_if_needed()

    os.symlink(drive_refuge_path, CFG.BASE_PATH)

    log_ok(f"Dataset enlazado localmente:")
    log_ok(f"{CFG.BASE_PATH} -> {drive_refuge_path}")


def unzip_nested_training_masks_if_needed(base_path=CFG.BASE_PATH):
    """
    Compatibilidad con estructuras donde las máscaras de entrenamiento
    vienen en Disc_Cup_Masks.zip.
    """

    masks_zip_path = os.path.join(
        base_path,
        "Annotation-Training400",
        "Disc_Cup_Masks.zip"
    )

    masks_target_dir = os.path.join(
        base_path,
        "Annotation-Training400",
        "Disc_Cup_Masks"
    )

    if os.path.exists(masks_zip_path) and not os.path.exists(masks_target_dir):
        log_info("Descomprimiendo máscaras anidadas de entrenamiento.")
        with zipfile.ZipFile(masks_zip_path, "r") as zip_ref:
            zip_ref.extractall(os.path.dirname(masks_zip_path))
        log_ok("Máscaras anidadas descomprimidas.")


def extract_zip_fallback():
    """
    Fallback: usa Refuge.zip solo si no existe la carpeta Refuge ya descomprimida.

    Este flujo queda como respaldo, no como vía principal.
    """

    if not os.path.exists(CFG.drive_zip_path):
        raise FileNotFoundError(
            f"No existe ni la carpeta Refuge ni el archivo {CFG.ZIP_NAME}. "
            f"Buscado en: {CFG.DRIVE_BASE}"
        )

    if not os.path.exists(CFG.local_zip_path):
        log_info(f"Copiando {CFG.ZIP_NAME} al almacenamiento local.")
        shutil.copy(CFG.drive_zip_path, CFG.local_zip_path)

    os.makedirs(CFG.EXTRACT_PATH, exist_ok=True)

    if CFG.FORCE_REEXTRACT:
        remove_local_base_path_if_needed()

    log_info(f"Descomprimiendo {CFG.ZIP_NAME} en {CFG.EXTRACT_PATH}.")
    with zipfile.ZipFile(CFG.local_zip_path, "r") as zip_ref:
        zip_ref.extractall(CFG.EXTRACT_PATH)

    if not is_refuge_complete_folder(CFG.BASE_PATH):
        # Buscar una carpeta Refuge válida dentro de EXTRACT_PATH.
        detected_root = None

        for current_root, dirs, files in os.walk(CFG.EXTRACT_PATH):
            depth = current_root.replace(CFG.EXTRACT_PATH, "").count(os.sep)

            if depth > 4:
                dirs[:] = []
                continue

            if is_refuge_complete_folder(current_root):
                detected_root = current_root
                break

        if detected_root is None:
            raise FileNotFoundError(
                "El ZIP se ha descomprimido, pero no se ha encontrado una estructura REFUGE completa."
            )

        remove_local_base_path_if_needed()
        os.symlink(detected_root, CFG.BASE_PATH)
        log_ok(f"Raíz REFUGE detectada y enlazada: {CFG.BASE_PATH} -> {detected_root}")


def validate_dataset_structure():
    """
    Comprueba que las rutas principales existen y contienen archivos.
    """

    paths = get_required_dataset_paths()

    required_for_training_and_validation = [
        "train_images",
        "train_masks",
        "val_images",
        "val_masks",
        "val_labels",
    ]

    missing = []

    for key in required_for_training_and_validation:
        if not os.path.exists(paths[key]):
            missing.append((key, paths[key]))

    if missing:
        for key, path in missing:
            log_error(f"Ruta crítica no encontrada ({key}): {path}")
        raise FileNotFoundError("La estructura del dataset REFUGE no está completa.")

    for key in required_for_training_and_validation:
        log_ok(f"Ruta verificada ({key}): {paths[key]}")

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
    mask_extensions = [".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"]

    n_train_images = count_files_by_extensions(paths["train_images"], image_extensions)
    n_train_masks = count_files_by_extensions(paths["train_masks"], mask_extensions)
    n_val_images = count_files_by_extensions(paths["val_images"], image_extensions)
    n_val_masks = count_files_by_extensions(paths["val_masks"], mask_extensions)

    log_info(f"Training interno - imágenes: {n_train_images} | máscaras: {n_train_masks}")
    log_info(f"Validación externa - imágenes: {n_val_images} | máscaras: {n_val_masks}")

    if n_train_images == 0 or n_train_masks == 0:
        raise FileNotFoundError("No se han encontrado imágenes o máscaras de entrenamiento.")

    if n_val_images == 0 or n_val_masks == 0:
        raise FileNotFoundError("No se han encontrado imágenes o máscaras de validación externa.")

    # Comprobación específica del archivo de etiquetas.
    try:
        labels_df = pd.read_excel(paths["val_labels"])
        log_ok(f"Archivo de etiquetas de validación leído correctamente: {paths['val_labels']}")
        log_info(f"Columnas de etiquetas de validación: {list(labels_df.columns)}")

        if "Glaucoma Label" not in labels_df.columns:
            log_warning("El archivo de validación no contiene una columna llamada exactamente 'Glaucoma Label'.")
        else:
            label_counts = labels_df["Glaucoma Label"].value_counts(dropna=False).to_dict()
            log_info(f"Distribución Glaucoma Label: {label_counts}")

    except Exception as exc:
        raise RuntimeError(
            f"No se pudo leer el archivo de etiquetas de validación: {paths['val_labels']}. Detalle: {exc}"
        )

    return {
        "n_train_images": n_train_images,
        "n_train_masks": n_train_masks,
        "n_val_images": n_val_images,
        "n_val_masks": n_val_masks,
        "val_labels_path": paths["val_labels"],
    }


def setup_data():
    """
    Prepara el dataset para el notebook.

    Prioridad:
        1. Usar carpeta ya descomprimida:
           /content/drive/MyDrive/TFG_Glaucoma_CLEAN/Refuge

        2. Usar Refuge.zip solo como fallback.

    No reentrena modelos ni modifica Models_vPro_Fixed.
    """

    if not os.path.exists("/content/drive/MyDrive"):
        log_info("Montando Google Drive.")
        drive.mount("/content/drive")
    else:
        log_info("Google Drive ya está montado.")

    if not os.path.exists(CFG.DRIVE_BASE):
        raise FileNotFoundError(
            f"No se encuentra la carpeta base del proyecto: {CFG.DRIVE_BASE}"
        )

    os.makedirs(CFG.SAVE_PATH, exist_ok=True)
    os.makedirs(os.path.join(CFG.SAVE_PATH, "logs"), exist_ok=True)

    drive_refuge_path = CFG.drive_dataset_folder_path

    if os.path.exists(drive_refuge_path) and is_refuge_complete_folder(drive_refuge_path):
        log_ok(f"Carpeta Refuge completa detectada en Drive: {drive_refuge_path}")
        link_drive_refuge_folder_to_local()
    else:
        log_warning("No se ha detectado una carpeta Refuge completa en Drive. Se usará Refuge.zip como fallback.")
        extract_zip_fallback()

    unzip_nested_training_masks_if_needed(CFG.BASE_PATH)

    stats = validate_dataset_structure()

    log_ok("Carga y verificación inicial del dataset completadas.")

    return stats


dataset_stats = setup_data()
dataset_stats

# 3. Protocolo experimental y preprocesamiento

El protocolo separa estrictamente entrenamiento interno y validacion externa. `REFUGE-Training400` se utiliza para entrenamiento y validacion interna mediante K-Fold. `REFUGE-Validation400` queda reservado para la validacion externa final.

In [ ]:
# ============================================================
# 3. PROTOCOLO EXPERIMENTAL Y PREPROCESAMIENTO
# ============================================================

import re

# ------------------------------------------------------------
# 3.1. Definicion estricta de conjuntos experimentales
# ------------------------------------------------------------

DATASET_PATHS = get_required_dataset_paths()

TRAIN_DIRS = [
    (
        DATASET_PATHS["train_images"],
        DATASET_PATHS["train_masks"],
    )
]

EXTERNAL_VAL_DIRS = [
    (
        DATASET_PATHS["val_images"],
        DATASET_PATHS["val_masks"],
    )
]


# ------------------------------------------------------------
# 3.2. Indexacion robusta de pares imagen-mascara
# ------------------------------------------------------------

def list_files_with_extensions(root_dir, extensions):
    """Lista archivos recursivamente filtrando por extensiones."""
    files = []

    for ext in extensions:
        files.extend(
            glob.glob(
                os.path.join(root_dir, "**", f"*{ext}"),
                recursive=True
            )
        )
        files.extend(
            glob.glob(
                os.path.join(root_dir, "**", f"*{ext.upper()}"),
                recursive=True
            )
        )

    return sorted(list(set(files)))


def normalize_stem(path):
    """
    Normaliza el nombre base de un archivo para facilitar el emparejamiento
    entre imagenes y mascaras.
    """
    stem = os.path.splitext(os.path.basename(path))[0].lower()

    replacements = [
        "_mask",
        "-mask",
        "_seg",
        "-seg",
        "_disc_cup",
        "-disc-cup",
        "_disc",
        "-disc",
        "_cup",
        "-cup",
    ]

    for rep in replacements:
        stem = stem.replace(rep, "")

    stem = stem.strip()
    return stem


def numeric_key(path):
    """
    Extrae una clave numerica del nombre del archivo.
    Sirve como fallback cuando imagen y mascara no tienen exactamente
    el mismo nombre pero comparten numeracion.
    """
    stem = os.path.splitext(os.path.basename(path))[0].lower()
    nums = re.findall(r"\d+", stem)

    if len(nums) == 0:
        return None

    return nums[-1].lstrip("0") or "0"


def build_mask_index(mask_paths):
    """Crea indices de mascaras por nombre normalizado y por clave numerica."""
    by_stem = {}
    by_num = {}

    for mask_path in mask_paths:
        stem = normalize_stem(mask_path)
        num = numeric_key(mask_path)

        by_stem[stem] = mask_path

        if num is not None and num not in by_num:
            by_num[num] = mask_path

    return by_stem, by_num


def get_all_pairs_robust(dirs_list):
    """
    Indexa pares imagen-mascara.

    Devuelve:
        np.array con tuplas (image_path, mask_path)
    """
    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
    mask_extensions = [".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"]

    all_pairs = []

    for img_dir, mask_dir in dirs_list:
        log_info(f"Indexando imagenes en: {img_dir}")
        log_info(f"Indexando mascaras en: {mask_dir}")

        image_paths = list_files_with_extensions(img_dir, image_extensions)
        mask_paths = list_files_with_extensions(mask_dir, mask_extensions)

        log_info(f"Imagenes encontradas: {len(image_paths)}")
        log_info(f"Mascaras encontradas: {len(mask_paths)}")

        if len(image_paths) == 0:
            raise FileNotFoundError(f"No se encontraron imagenes en: {img_dir}")

        if len(mask_paths) == 0:
            raise FileNotFoundError(f"No se encontraron mascaras en: {mask_dir}")

        masks_by_stem, masks_by_num = build_mask_index(mask_paths)

        unmatched = []

        for image_path in image_paths:
            img_stem = normalize_stem(image_path)
            img_num = numeric_key(image_path)

            mask_path = None

            if img_stem in masks_by_stem:
                mask_path = masks_by_stem[img_stem]
            elif img_num is not None and img_num in masks_by_num:
                mask_path = masks_by_num[img_num]

            if mask_path is not None:
                all_pairs.append((image_path, mask_path))
            else:
                unmatched.append(image_path)

        if len(unmatched) > 0:
            log_warning(f"Imagenes sin mascara emparejada: {len(unmatched)}")
            for example in unmatched[:5]:
                log_warning(f"Ejemplo sin mascara: {os.path.basename(example)}")

    return np.array(all_pairs, dtype=object)


train_data = get_all_pairs_robust(TRAIN_DIRS)
external_val_data = get_all_pairs_robust(EXTERNAL_VAL_DIRS)

log_ok(f"Training interno indexado: {len(train_data)} pares.")
log_ok(f"Validacion externa reservada: {len(external_val_data)} pares.")


# ------------------------------------------------------------
# 3.3. Comprobacion de fuga de datos
# ------------------------------------------------------------

import hashlib


def file_md5(path, chunk_size=1024 * 1024):
    """
    Calcula el hash MD5 de un archivo.

    Se usa para comprobar fuga de datos real, no solo coincidencias
    de numeracion entre carpetas.
    """
    md5 = hashlib.md5()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            md5.update(chunk)

    return md5.hexdigest()


def verify_no_data_leakage(train_pairs, external_pairs, use_hash=True):
    """
    Comprueba que no exista fuga de datos entre entrenamiento y validacion externa.

    No se usa solo el numero del archivo porque en REFUGE puede haber numeraciones
    repetidas entre particiones distintas.

    Comprobaciones:
        1. que no se reutilice exactamente la misma ruta;
        2. opcionalmente, que no haya archivos de imagen identicos por hash.
    """

    train_image_paths = [pair[0] for pair in train_pairs]
    external_image_paths = [pair[0] for pair in external_pairs]

    # 1. Comprobacion por ruta absoluta
    train_realpaths = set(os.path.realpath(path) for path in train_image_paths)
    external_realpaths = set(os.path.realpath(path) for path in external_image_paths)

    path_overlap = train_realpaths.intersection(external_realpaths)

    if len(path_overlap) > 0:
        examples = sorted(list(path_overlap))[:10]
        raise ValueError(
            "Se ha detectado fuga de datos por rutas identicas entre Training y Validation. "
            f"Ejemplos: {examples}"
        )

    log_ok("No hay solapamiento de rutas entre entrenamiento y validacion externa.")

    # 2. Comprobacion por contenido de imagen
    if use_hash:
        log_info("Calculando hashes de imagenes para comprobar duplicados reales.")

        train_hashes = {}
        for path in train_image_paths:
            train_hashes[file_md5(path)] = path

        duplicated = []

        for path in external_image_paths:
            h = file_md5(path)

            if h in train_hashes:
                duplicated.append(
                    {
                        "train_path": train_hashes[h],
                        "external_path": path,
                        "hash": h,
                    }
                )

        if len(duplicated) > 0:
            examples = duplicated[:5]

            for item in examples:
                log_error(f"Duplicado detectado:")
                log_error(f"  Train: {item['train_path']}")
                log_error(f"  Val:   {item['external_path']}")

            raise ValueError(
                f"Se han detectado {len(duplicated)} imagenes identicas entre Training y Validation."
            )

        log_ok("No se detectan imagenes duplicadas por contenido entre entrenamiento y validacion externa.")

    log_ok("Comprobacion de fuga de datos superada correctamente.")


verify_no_data_leakage(train_data, external_val_data, use_hash=True)


if len(train_data) == 0:
    raise ValueError("No hay pares de entrenamiento. Revisa la estructura del dataset.")

if len(external_val_data) == 0:
    raise ValueError("No hay pares de validacion externa. Revisa la estructura del dataset.")


# ------------------------------------------------------------
# 3.4. Lectura de imagenes y mascaras
# ------------------------------------------------------------

def read_rgb_image(image_path):
    """Lee una imagen en RGB."""
    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {image_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return image_rgb


def read_mask_grayscale(mask_path):
    """Lee una mascara en escala de grises."""
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    if mask is None:
        raise FileNotFoundError(f"No se pudo leer la mascara: {mask_path}")

    if len(mask.shape) == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    return mask


# ------------------------------------------------------------
# 3.5. Localizacion automatica de la region papilar
# ------------------------------------------------------------

def _legacy_locate_roi_center(image_rgb):
    """
    Algoritmo de localizacion ROI original (canal verde + CLAHE + percentil 99.3
    + mayor componente brillante + centroide).

    Se conserva intacto y se usa como referencia: si su resultado es plausible
    (dentro del FOV y no pegado al borde), se respeta para NO alterar el recorte
    de las imagenes que ya estaban bien. Solo se sustituye en los casos en los que
    falla. Asi el cambio no introduce distribution shift respecto a los modelos ya
    entrenados (no es necesario reentrenar).
    """
    green = image_rgb[:, :, 1]

    green = cv2.GaussianBlur(green, (9, 9), 0)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(green)

    threshold = np.percentile(enhanced, 99.3)
    bright = (enhanced >= threshold).astype(np.uint8) * 255

    kernel = np.ones((15, 15), np.uint8)
    bright = cv2.morphologyEx(bright, cv2.MORPH_CLOSE, kernel)
    bright = cv2.morphologyEx(bright, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        bright,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = green.shape

    if len(contours) == 0:
        return w // 2, h // 2

    largest_contour = max(contours, key=cv2.contourArea)
    moments = cv2.moments(largest_contour)

    if moments["m00"] == 0:
        return w // 2, h // 2

    cx = int(moments["m10"] / moments["m00"])
    cy = int(moments["m01"] / moments["m00"])

    return cx, cy


def _largest_component_mask(binary):
    """Devuelve la mascara del mayor componente conectado (ignora el fondo)."""
    binary = (binary > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    if num <= 1:
        return binary

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return (labels == largest).astype(np.uint8)


def estimate_fov_mask(image_rgb):
    """
    Estima la mascara del campo de vision (FOV) del fondo de ojo, es decir el
    circulo iluminado de la camara de retinografia. Solo usa la imagen.

    Sirve para:
        - acotar la busqueda de la papila al interior del FOV;
        - penalizar candidatos pegados al borde del FOV (reflejos del anillo);
        - descartar artefactos fuera del FOV.

    Devuelve:
        (fov_mask uint8, (fx, fy) centro del FOV, fov_radius aproximado)
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    # Fuera del FOV el fondo es practicamente negro. Umbral bajo y robusto.
    fov = (gray > 12).astype(np.uint8)

    # Si casi nada supera el umbral, asumimos que todo el frame es FOV.
    if fov.sum() < 0.05 * h * w:
        fov = np.ones((h, w), dtype=np.uint8)

    fov = _largest_component_mask(fov)

    # Cerrar huecos internos (vasos oscuros, reflejos) para obtener un FOV solido.
    fov = cv2.morphologyEx(
        fov,
        cv2.MORPH_CLOSE,
        np.ones((25, 25), np.uint8)
    )

    ys, xs = np.where(fov > 0)

    if len(xs) == 0:
        return (
            np.ones((h, w), dtype=np.uint8),
            (w / 2.0, h / 2.0),
            min(h, w) / 2.0,
        )

    fx = float(xs.mean())
    fy = float(ys.mean())

    # Radio equivalente a partir del area del FOV.
    fov_radius = float(np.sqrt(fov.sum() / np.pi))

    return fov, (fx, fy), fov_radius


def is_plausible_center(cx, cy, fov_info, image_shape):
    """
    Comprueba si un centro candidato es anatomicamente plausible:
        1. cae dentro de la imagen;
        2. cae dentro del FOV;
        3. no esta pegado al borde del FOV.

    Se usa como guarda de compatibilidad: si el centro del algoritmo original
    es plausible, se respeta.
    """
    fov_mask, (fx, fy), fov_radius = fov_info
    h, w = image_shape[:2]

    cx_i = int(round(cx))
    cy_i = int(round(cy))

    if not (0 <= cx_i < w and 0 <= cy_i < h):
        return False

    if fov_mask[cy_i, cx_i] == 0:
        return False

    dist_to_fov_center = np.hypot(cx_i - fx, cy_i - fy)

    if dist_to_fov_center > 0.92 * fov_radius:
        return False

    return True


def locate_roi_center_robust(image_rgb):
    """
    Localizacion robusta de la papila optica usando SOLO la imagen.

    A diferencia del algoritmo original, no se queda con el mayor componente
    brillante, sino que:
        1. detecta multiples candidatos brillantes a varios percentiles;
        2. los puntua combinando brillo, compacidad/circularidad y tamano plausible;
        3. penaliza candidatos demasiado perifericos o pegados al borde del FOV.

    Devuelve:
        (cx, cy, diagnostics)
    """
    green = image_rgb[:, :, 1]
    green = cv2.GaussianBlur(green, (9, 9), 0)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    h, w = enhanced.shape

    fov_mask, (fx, fy), fov_radius = estimate_fov_mask(image_rgb)
    enhanced_fov = np.where(fov_mask > 0, enhanced, 0).astype(np.uint8)

    fov_values = enhanced[fov_mask > 0]

    kernel = np.ones((15, 15), np.uint8)
    candidates = []

    # Varios percentiles, no solo 99.3, para no depender de un unico umbral.
    for pct in (99.5, 99.0, 98.5, 97.5):
        if fov_values.size == 0:
            break

        threshold = np.percentile(fov_values, pct)
        bright = ((enhanced_fov >= threshold) & (fov_mask > 0)).astype(np.uint8) * 255
        bright = cv2.morphologyEx(bright, cv2.MORPH_CLOSE, kernel)
        bright = cv2.morphologyEx(bright, cv2.MORPH_OPEN, kernel)

        contours, _ = cv2.findContours(
            bright,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            area = cv2.contourArea(contour)

            if area < 50:
                continue

            moments = cv2.moments(contour)
            if moments["m00"] == 0:
                continue

            ccx = moments["m10"] / moments["m00"]
            ccy = moments["m01"] / moments["m00"]

            perimeter = cv2.arcLength(contour, True)
            circularity = (4.0 * np.pi * area / (perimeter ** 2)) if perimeter > 0 else 0.0
            circularity = float(np.clip(circularity, 0.0, 1.0))

            contour_mask = np.zeros((h, w), dtype=np.uint8)
            cv2.drawContours(contour_mask, [contour], -1, 255, thickness=-1)
            mean_bright = float(cv2.mean(enhanced, mask=contour_mask)[0])

            candidates.append(
                {
                    "cx": ccx,
                    "cy": ccy,
                    "area": float(area),
                    "circularity": circularity,
                    "mean_bright": mean_bright,
                }
            )

    if len(candidates) == 0:
        diagnostics = {
            "center": (int(round(fx)), int(round(fy))),
            "score": float("nan"),
            "n_candidates": 0,
            "fov_center": (fx, fy),
            "fov_radius": fov_radius,
            "used_fallback": True,
        }
        return int(round(fx)), int(round(fy)), diagnostics

    # Deduplicar candidatos muy cercanos (mismo blob detectado a varios percentiles),
    # conservando el de mayor area.
    dedup = []
    for cand in sorted(candidates, key=lambda c: c["area"], reverse=True):
        is_duplicate = False
        for kept in dedup:
            if np.hypot(cand["cx"] - kept["cx"], cand["cy"] - kept["cy"]) < 25:
                is_duplicate = True
                break
        if not is_duplicate:
            dedup.append(cand)

    # Tamano de disco esperado: el diametro del disco ronda ~0.2 del diametro del FOV.
    expected_disc_radius = 0.20 * fov_radius

    for cand in dedup:
        eq_radius = np.sqrt(cand["area"] / np.pi)
        size_ratio = eq_radius / max(expected_disc_radius, 1e-6)
        size_score = float(np.exp(-0.5 * (np.log(max(size_ratio, 1e-6)) / 0.6) ** 2))

        dist = np.hypot(cand["cx"] - fx, cand["cy"] - fy)
        periph = dist / max(fov_radius, 1e-6)
        periph_penalty = float(np.clip((periph - 0.85) / 0.15, 0.0, 1.0))
        border_penalty = 1.0 if periph > 0.95 else 0.0

        bright_norm = cand["mean_bright"] / 255.0

        cand["score"] = (
            0.45 * bright_norm
            + 0.30 * cand["circularity"]
            + 0.25 * size_score
            - 0.50 * periph_penalty
            - 0.50 * border_penalty
        )

    best = max(dedup, key=lambda c: c["score"])

    diagnostics = {
        "center": (int(round(best["cx"])), int(round(best["cy"]))),
        "score": float(best["score"]),
        "n_candidates": len(dedup),
        "fov_center": (fx, fy),
        "fov_radius": fov_radius,
        "used_fallback": False,
    }

    return int(round(best["cx"])), int(round(best["cy"])), diagnostics


def locate_roi_center(image_rgb):
    """
    Localiza el centro de la papila optica usando SOLO la imagen (aplicable en
    inferencia real, nunca usa la mascara).

    Estrategia de OVERRIDE CON GUARDA:
        1. se calcula el centro con el algoritmo original (`_legacy_locate_roi_center`);
        2. si ese centro es plausible (dentro del FOV y no pegado al borde), se respeta,
           de modo que las imagenes ya correctas conservan exactamente su recorte;
        3. solo si el centro original NO es plausible se sustituye por el localizador
           robusto (`locate_roi_center_robust`).

    De esta forma se corrigen los casos en los que la ROI fallaba (papila fuera del
    recorte) sin alterar la distribucion de recortes del resto de imagenes, por lo que
    los modelos ya entrenados siguen siendo validos sin reentrenar.
    """
    cx_old, cy_old = _legacy_locate_roi_center(image_rgb)

    fov_info = estimate_fov_mask(image_rgb)

    if is_plausible_center(cx_old, cy_old, fov_info, image_rgb.shape):
        return cx_old, cy_old

    cx_new, cy_new, _ = locate_roi_center_robust(image_rgb)
    return cx_new, cy_new


def crop_square_with_padding(array, cx, cy, radius, is_mask=False):
    """
    Recorta una region cuadrada centrada en (cx, cy).

    Si la region se sale de la imagen, rellena con padding.
    Para imagenes se rellena con 0.
    Para mascaras se rellena con 255, que en REFUGE suele representar fondo.
    """
    h, w = array.shape[:2]

    x1 = cx - radius
    x2 = cx + radius
    y1 = cy - radius
    y2 = cy + radius

    src_x1 = max(0, x1)
    src_x2 = min(w, x2)
    src_y1 = max(0, y1)
    src_y2 = min(h, y2)

    pad_left = max(0, -x1)
    pad_right = max(0, x2 - w)
    pad_top = max(0, -y1)
    pad_bottom = max(0, y2 - h)

    cropped = array[src_y1:src_y2, src_x1:src_x2]

    if is_mask:
        pad_value = 255
    else:
        pad_value = 0

    if len(array.shape) == 3:
        cropped = cv2.copyMakeBorder(
            cropped,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            cv2.BORDER_CONSTANT,
            value=[pad_value, pad_value, pad_value]
        )
    else:
        cropped = cv2.copyMakeBorder(
            cropped,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            cv2.BORDER_CONSTANT,
            value=pad_value
        )

    return cropped


# ------------------------------------------------------------
# 3.6. Conversion de mascaras a clases semanticas
# ------------------------------------------------------------

def decode_refuge_mask(mask_gray):
    """
    Convierte la mascara original de REFUGE a clases semanticas:

        0 = fondo
        1 = disco optico / anillo neuroretiniano
        2 = copa optica

    En REFUGE es habitual encontrar:
        255 = fondo
        128 = disco
        0   = copa

    La funcion usa asignacion por valor mas cercano para ser robusta.
    """
    mask_gray = mask_gray.astype(np.float32)

    # Caso en el que la mascara ya venga como clases 0, 1, 2.
    unique_values = np.unique(mask_gray)

    if mask_gray.max() <= 2 and len(unique_values) <= 3:
        mask_class = mask_gray.astype(np.uint8)
        return mask_class

    dist_to_cup = np.abs(mask_gray - 0)
    dist_to_disc = np.abs(mask_gray - 128)
    dist_to_background = np.abs(mask_gray - 255)

    stacked = np.stack(
        [
            dist_to_background,
            dist_to_disc,
            dist_to_cup,
        ],
        axis=-1
    )

    mask_class = np.argmin(stacked, axis=-1).astype(np.uint8)

    return mask_class


def one_hot_encode_mask(mask_class):
    """Convierte mascara de clases a one-hot encoding."""
    mask_onehot = tf.keras.utils.to_categorical(
        mask_class,
        num_classes=CFG.CLASSES
    )

    return mask_onehot.astype(np.float32)


# ------------------------------------------------------------
# 3.7. Mejora de contraste y normalizacion
# ------------------------------------------------------------

def apply_clahe_rgb(image_rgb):
    """
    Aplica CLAHE sobre el canal L del espacio LAB.
    Mejora el contraste local sin alterar directamente las clases de la mascara.
    """
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge((l_channel, a_channel, b_channel))
    enhanced_rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    return enhanced_rgb


def preprocess_image_and_mask(image_path, mask_path):
    """
    Preprocesa una imagen y su mascara asociada.

    Pasos:
        1. lectura RGB;
        2. lectura mascara;
        3. localizacion automatica ROI;
        4. recorte cuadrado con padding;
        5. resize a CFG.IMG_SIZE;
        6. conversion mascara a clases;
        7. CLAHE;
        8. preprocesamiento del backbone;
        9. one-hot encoding de mascara.
    """
    image_rgb = read_rgb_image(image_path)
    mask_gray = read_mask_grayscale(mask_path)

    cx, cy = locate_roi_center(image_rgb)

    image_crop = crop_square_with_padding(
        image_rgb,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=False
    )

    mask_crop = crop_square_with_padding(
        mask_gray,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=True
    )

    image_resized = cv2.resize(
        image_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    mask_class = decode_refuge_mask(mask_resized)

    image_enhanced = apply_clahe_rgb(image_resized)

    image_preprocessed = preprocess_input(
        image_enhanced.astype(np.float32)
    )

    mask_onehot = one_hot_encode_mask(mask_class)

    return image_preprocessed.astype(np.float32), mask_onehot.astype(np.float32)


def preprocess_image_only(image_path):
    """
    Preprocesa una imagen sin mascara.
    Se usara posteriormente en inferencia real.
    """
    image_rgb = read_rgb_image(image_path)

    cx, cy = locate_roi_center(image_rgb)

    image_crop = crop_square_with_padding(
        image_rgb,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=False
    )

    image_resized = cv2.resize(
        image_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    image_enhanced = apply_clahe_rgb(image_resized)

    image_preprocessed = preprocess_input(
        image_enhanced.astype(np.float32)
    )

    metadata = {
        "cx": cx,
        "cy": cy,
        "roi_radius": CFG.ROI_RADIUS,
        "original_shape": image_rgb.shape,
    }

    return image_preprocessed.astype(np.float32), image_resized, metadata


# ------------------------------------------------------------
# 3.8. Comprobacion visual del preprocesamiento
# ------------------------------------------------------------

def visualize_preprocessing_sample(pairs, index=0):
    """Muestra una muestra preprocesada para verificar imagen y mascara."""
    image_path, mask_path = pairs[index]

    image_rgb = read_rgb_image(image_path)
    mask_gray = read_mask_grayscale(mask_path)

    cx, cy = locate_roi_center(image_rgb)

    image_crop = crop_square_with_padding(
        image_rgb,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=False
    )

    mask_crop = crop_square_with_padding(
        mask_gray,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=True
    )

    image_resized = cv2.resize(
        image_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    mask_class = decode_refuge_mask(mask_resized)
    image_enhanced = apply_clahe_rgb(image_resized)

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image_rgb)
    plt.scatter([cx], [cy], c="red", s=30)
    plt.title("Imagen original + ROI")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(image_resized)
    plt.title("ROI redimensionada")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(image_enhanced)
    plt.title("ROI con CLAHE")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(mask_class, cmap="viridis", vmin=0, vmax=2)
    plt.title("Mascara: fondo/disco/copa")
    plt.axis("off")

    plt.suptitle(
        f"Muestra: {os.path.basename(image_path)}",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()

    log_info(f"Imagen: {image_path}")
    log_info(f"Mascara: {mask_path}")
    log_info(f"Valores originales mascara: {np.unique(mask_gray)[:20]}")
    log_info(f"Valores mascara procesada: {np.unique(mask_class)}")


# Ejecutar una comprobacion visual rapida.
visualize_preprocessing_sample(train_data, index=0)

log_ok("Protocolo experimental y preprocesamiento definidos correctamente.")

# 4. Construccion del pipeline de entrenamiento

Las funciones de preprocesamiento son compartidas por entrenamiento, validacion, inferencia y visualizacion. Esto evita diferencias artificiales entre lo que el modelo aprende y lo que recibe durante la evaluacion.

In [ ]:
# ============================================================
# 4. CONSTRUCCIÓN DEL PIPELINE DE ENTRENAMIENTO
# ============================================================

# ------------------------------------------------------------
# 4.1. Data augmentation
# ------------------------------------------------------------

def get_training_augmentation():
    """
    Define aumentos de datos para segmentacion.

    Se aplican transformaciones moderadas para no destruir la estructura anatomica
    del fondo de ojo. La mascara se transforma de forma sincronizada con la imagen.
    """

    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),

            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=0.6
            ),

            A.RandomBrightnessContrast(
                brightness_limit=0.08,
                contrast_limit=0.08,
                p=0.3
            ),

            A.GaussNoise(
                var_limit=(5.0, 20.0),
                p=0.2
            ),
        ]
    )


train_augmentation = get_training_augmentation()


# ------------------------------------------------------------
# 4.2. Funciones auxiliares para augmentation
# ------------------------------------------------------------

def mask_onehot_to_class(mask_onehot):
    """Convierte una mascara one-hot a mascara de clases."""
    return np.argmax(mask_onehot, axis=-1).astype(np.uint8)


def mask_class_to_onehot(mask_class):
    """Convierte una mascara de clases a one-hot."""
    return tf.keras.utils.to_categorical(
        mask_class,
        num_classes=CFG.CLASSES
    ).astype(np.float32)


def apply_augmentation_numpy(image, mask_onehot):
    """
    Aplica augmentation usando Albumentations.

    Entrada:
        image: imagen ya preprocesada para el backbone.
        mask_onehot: mascara one-hot.

    Salida:
        image_aug: imagen aumentada.
        mask_aug_onehot: mascara aumentada one-hot.
    """

    mask_class = mask_onehot_to_class(mask_onehot)

    augmented = train_augmentation(
        image=image.astype(np.float32),
        mask=mask_class.astype(np.uint8)
    )

    image_aug = augmented["image"].astype(np.float32)
    mask_aug_class = augmented["mask"].astype(np.uint8)

    # Seguridad: evitar valores fuera de rango en la mascara.
    mask_aug_class = np.clip(mask_aug_class, 0, CFG.CLASSES - 1)

    mask_aug_onehot = mask_class_to_onehot(mask_aug_class)

    return image_aug.astype(np.float32), mask_aug_onehot.astype(np.float32)


# ------------------------------------------------------------
# 4.3. Carga Python/Numpy para tf.data
# ------------------------------------------------------------

def load_sample_numpy(image_path, mask_path, augment=False):
    """
    Carga y preprocesa una muestra individual.

    Esta funcion se ejecuta dentro de tf.numpy_function.
    """

    image_path = image_path.decode("utf-8")
    mask_path = mask_path.decode("utf-8")

    image, mask = preprocess_image_and_mask(image_path, mask_path)

    if augment:
        image, mask = apply_augmentation_numpy(image, mask)

    image = image.astype(np.float32)
    mask = mask.astype(np.float32)

    return image, mask


def tf_load_sample(image_path, mask_path, augment=False):
    """
    Wrapper TensorFlow para cargar una muestra usando funciones Python/Numpy.
    """

    image, mask = tf.numpy_function(
        func=lambda img_p, msk_p: load_sample_numpy(
            img_p,
            msk_p,
            augment=augment
        ),
        inp=[image_path, mask_path],
        Tout=[tf.float32, tf.float32]
    )

    image.set_shape((CFG.IMG_SIZE, CFG.IMG_SIZE, 3))
    mask.set_shape((CFG.IMG_SIZE, CFG.IMG_SIZE, CFG.CLASSES))

    return image, mask


# ------------------------------------------------------------
# 4.4. Creacion del pipeline tf.data
# ------------------------------------------------------------

def create_dataset(pairs, augment=False, shuffle=False, batch_size=CFG.BATCH_SIZE):
    """
    Crea un dataset tf.data a partir de pares imagen-mascara.

    Args:
        pairs:
            np.array/list de tuplas (image_path, mask_path).
        augment:
            aplica data augmentation si True.
        shuffle:
            baraja las muestras si True.
        batch_size:
            tamano de batch.

    Returns:
        tf.data.Dataset
    """

    if len(pairs) == 0:
        raise ValueError("No se puede crear un dataset vacio.")

    image_paths = [pair[0] for pair in pairs]
    mask_paths = [pair[1] for pair in pairs]

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            tf.constant(image_paths),
            tf.constant(mask_paths)
        )
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(pairs),
            seed=CFG.SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        lambda img_p, msk_p: tf_load_sample(
            img_p,
            msk_p,
            augment=augment
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False
    )

    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


# ------------------------------------------------------------
# 4.5. Datasets de prueba del pipeline
# ------------------------------------------------------------

# Pequeño subconjunto para verificar que el pipeline funciona antes del entrenamiento.
sample_train_pairs = train_data[:min(16, len(train_data))]
sample_val_pairs = train_data[min(16, len(train_data)):min(32, len(train_data))]

if len(sample_val_pairs) == 0:
    sample_val_pairs = train_data[:min(16, len(train_data))]

debug_train_ds = create_dataset(
    sample_train_pairs,
    augment=True,
    shuffle=True,
    batch_size=4
)

debug_val_ds = create_dataset(
    sample_val_pairs,
    augment=False,
    shuffle=False,
    batch_size=4
)

log_ok("Datasets de prueba creados correctamente.")


# ------------------------------------------------------------
# 4.6. Verificacion de shapes y tipos
# ------------------------------------------------------------

for batch_images, batch_masks in debug_train_ds.take(1):
    log_info(f"Batch imagenes shape: {batch_images.shape}")
    log_info(f"Batch mascaras shape: {batch_masks.shape}")
    log_info(f"Batch imagenes dtype: {batch_images.dtype}")
    log_info(f"Batch mascaras dtype: {batch_masks.dtype}")

    unique_mask_values = tf.unique(
        tf.reshape(
            tf.argmax(batch_masks, axis=-1),
            [-1]
        )
    )[0]

    log_info(f"Clases presentes en batch de mascaras: {unique_mask_values.numpy()}")


# ------------------------------------------------------------
# 4.7. Visualizacion de batch aumentado
# ------------------------------------------------------------

def denormalize_for_display(image):
    """
    Convierte una imagen preprocesada a un rango aproximado visible.

    No se usa para entrenamiento, solo para visualizacion.
    """
    image = image.astype(np.float32)

    img_min = image.min()
    img_max = image.max()

    if img_max - img_min < 1e-6:
        return np.zeros_like(image)

    image = (image - img_min) / (img_max - img_min)

    return np.clip(image, 0, 1)


def visualize_training_batch(dataset, n=4):
    """Visualiza imagenes y mascaras de un batch del pipeline."""
    batch_images, batch_masks = next(iter(dataset))

    batch_images = batch_images.numpy()
    batch_masks = batch_masks.numpy()

    n = min(n, batch_images.shape[0])

    plt.figure(figsize=(4 * n, 8))

    for i in range(n):
        image_display = denormalize_for_display(batch_images[i])
        mask_class = np.argmax(batch_masks[i], axis=-1)

        plt.subplot(2, n, i + 1)
        plt.imshow(image_display)
        plt.title(f"Imagen {i+1}")
        plt.axis("off")

        plt.subplot(2, n, n + i + 1)
        plt.imshow(mask_class, cmap="viridis", vmin=0, vmax=2)
        plt.title("Mascara")
        plt.axis("off")

    plt.suptitle("Comprobacion del pipeline de entrenamiento", fontsize=14)
    plt.tight_layout()
    plt.show()


visualize_training_batch(debug_train_ds, n=4)

log_ok("Pipeline de entrenamiento construido y verificado correctamente.")

# 5. Definicion del modelo

Se mantiene la arquitectura U-Net con backbone `inceptionresnetv2`. Las metricas historicas `dice_coef_disc` y `dice_coef_cup` se conservan por compatibilidad, pero calculan IoU por clase mediante `segmentation_models.metrics.IOUScore`.

> **v5 (reentreno dirigido):** la arquitectura U-Net + `inceptionresnetv2` no cambia. La pérdida **definitiva de entrenamiento** se define en la sección 6 y en v5 sustituye la Dice por una **Tversky multiclase** que penaliza más la sobre-segmentación de la copa (β>α), para corregir en origen el sesgo de vCDR (+0.074) detectado en la auditoría de v4.


In [ ]:
# ============================================================
# 5. DEFINICIÓN DEL MODELO
# ============================================================

# ------------------------------------------------------------
# 5.1. Utilidades para pérdidas y métricas
# ------------------------------------------------------------

SMOOTH = 1e-6


def cast_for_metric(y_true, y_pred):
    """
    Convierte tensores a float32 para evitar problemas con mixed precision.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return y_true, y_pred


def clip_predictions(y_pred):
    """
    Evita valores extremos exactos en softmax para estabilizar logaritmos.
    """
    y_pred = tf.clip_by_value(y_pred, SMOOTH, 1.0 - SMOOTH)
    return y_pred


# ------------------------------------------------------------
# 5.2. Dice loss multiclase
# ------------------------------------------------------------

def multiclass_dice_loss(y_true, y_pred):
    """
    Dice loss multiclase.

    Penaliza la falta de solapamiento entre la máscara real y la predicha.
    Es especialmente útil en segmentación médica, donde las clases pueden estar
    desbalanceadas.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)
    y_pred = clip_predictions(y_pred)

    axes = [1, 2]

    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true + y_pred, axis=axes)

    dice = (2.0 * intersection + SMOOTH) / (denominator + SMOOTH)

    # Media entre clases y batch
    dice_loss = 1.0 - tf.reduce_mean(dice)

    return dice_loss


# ------------------------------------------------------------
# 5.3. Categorical focal loss
# ------------------------------------------------------------

def categorical_focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """
    Focal loss multiclase.

    Reduce el peso de ejemplos fáciles y aumenta la contribución relativa
    de regiones más difíciles, como la copa óptica.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)
    y_pred = clip_predictions(y_pred)

    cross_entropy = -y_true * tf.math.log(y_pred)
    focal_weight = alpha * tf.pow(1.0 - y_pred, gamma)

    loss = focal_weight * cross_entropy
    loss = tf.reduce_sum(loss, axis=-1)

    return tf.reduce_mean(loss)


# ------------------------------------------------------------
# 5.4. Pérdida híbrida final
# ------------------------------------------------------------

def hybrid_loss(y_true, y_pred):
    """
    Pérdida híbrida usada para entrenar la segmentación.

    Combina:
        - Dice loss: favorece el solapamiento espacial.
        - Focal loss: ayuda en clases pequeñas o difíciles.
    """
    dice = multiclass_dice_loss(y_true, y_pred)
    focal = categorical_focal_loss(y_true, y_pred)

    return dice + focal


# ------------------------------------------------------------
# 5.5. Métricas de segmentación
# ------------------------------------------------------------

def class_iou(y_true, y_pred, class_id):
    """
    Calcula IoU para una clase concreta usando probabilidades softmax.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)

    y_true_c = y_true[..., class_id]
    y_pred_c = y_pred[..., class_id]

    intersection = tf.reduce_sum(y_true_c * y_pred_c, axis=[1, 2])
    union = (
        tf.reduce_sum(y_true_c, axis=[1, 2])
        + tf.reduce_sum(y_pred_c, axis=[1, 2])
        - intersection
    )

    iou = (intersection + SMOOTH) / (union + SMOOTH)

    return tf.reduce_mean(iou)


def class_dice(y_true, y_pred, class_id):
    """
    Calcula Dice para una clase concreta usando probabilidades softmax.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)

    y_true_c = y_true[..., class_id]
    y_pred_c = y_pred[..., class_id]

    intersection = tf.reduce_sum(y_true_c * y_pred_c, axis=[1, 2])
    denominator = (
        tf.reduce_sum(y_true_c, axis=[1, 2])
        + tf.reduce_sum(y_pred_c, axis=[1, 2])
    )

    dice = (2.0 * intersection + SMOOTH) / (denominator + SMOOTH)

    return tf.reduce_mean(dice)


def global_iou(y_true, y_pred):
    """
    IoU media de las clases anatómicas principales.

    Se excluye el fondo y se calcula sobre:
        1 = disco óptico
        2 = copa óptica
    """
    disc_iou = class_iou(y_true, y_pred, class_id=1)
    cup_iou = class_iou(y_true, y_pred, class_id=2)

    return (disc_iou + cup_iou) / 2.0


def iou_disc(y_true, y_pred):
    """IoU del disco óptico."""
    return class_iou(y_true, y_pred, class_id=1)


def iou_cup(y_true, y_pred):
    """IoU de la copa óptica."""
    return class_iou(y_true, y_pred, class_id=2)


def dice_disc(y_true, y_pred):
    """Dice del disco óptico."""
    return class_dice(y_true, y_pred, class_id=1)


def dice_cup(y_true, y_pred):
    """Dice de la copa óptica."""
    return class_dice(y_true, y_pred, class_id=2)


# ------------------------------------------------------------
# 5.6. Construcción del modelo
# ------------------------------------------------------------

def build_model():
    """
    Construye y compila el modelo U-Net.

    Arquitectura:
        - U-Net
        - Backbone: inceptionresnetv2
        - Salida: 3 clases con softmax
    """
    model = sm.Unet(
        CFG.BACKBONE,
        encoder_weights="imagenet",
        classes=CFG.CLASSES,
        activation="softmax"
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=CFG.LR_START
    )

    model.compile(
        optimizer=optimizer,
        loss=hybrid_loss,
        metrics=[
            global_iou,
            iou_disc,
            iou_cup,
            dice_disc,
            dice_cup,
        ]
    )

    return model


# ------------------------------------------------------------
# 5.7. Objetos personalizados para carga posterior de modelos
# ------------------------------------------------------------

CUSTOM_OBJECTS = {
    "hybrid_loss": hybrid_loss,
    "multiclass_dice_loss": multiclass_dice_loss,
    "categorical_focal_loss": categorical_focal_loss,
    "global_iou": global_iou,
    "iou_disc": iou_disc,
    "iou_cup": iou_cup,
    "dice_disc": dice_disc,
    "dice_cup": dice_cup,
}


# ------------------------------------------------------------
# 5.8. Prueba rápida de construcción del modelo
# ------------------------------------------------------------

test_model = build_model()

log_ok("Modelo construido y compilado correctamente.")
log_info(f"Backbone: {CFG.BACKBONE}")
log_info(f"Tamaño de entrada: {CFG.IMG_SIZE}x{CFG.IMG_SIZE}x3")
log_info(f"Número de clases: {CFG.CLASSES}")
log_info(f"Learning rate inicial: {CFG.LR_START}")

test_model.summary()

# Liberar memoria antes del entrenamiento real.
del test_model
tf.keras.backend.clear_session()
gc.collect()

log_ok("Definición del modelo completada.")

# 6. Entrenamiento K-Fold del modelo de segmentación disco-copa
Este bloque entrena cinco modelos mediante K-Fold usando únicamente REFUGE-Training400. Prioriza métricas clínicamente relevantes para glaucoma, como disco óptico completo, copa óptica y error de vCDR, y guarda el mejor modelo de cada fold.

**Cambio v5 — pérdida dirigida a la copa.** La pérdida pasa de `Dice ponderada + 0.5·focal` a `Tversky ponderada + 0.5·focal`. La Tversky usa α=β=0.5 para fondo y disco (equivale a la Dice anterior, el disco no se altera) y **α=0.3 < β=0.7 para la copa**, penalizando más los falsos positivos (sobre-segmentación) que los falsos negativos. Es la intervención principal de la auditoría: ataca en origen la sobre-estimación sistemática del vCDR (+0.074) y la baja especificidad. Todo lo demás del entrenamiento (backbone, 512×512, batch 8, LR 1e-4, 50 épocas, augmentation y la métrica de selección de checkpoint) se mantiene idéntico para que la mejora sea atribuible a este cambio. Los modelos v5 se guardan en `Models_v5_TverskyCup`, conservando los de v4 para comparar.


In [ ]:
# ============================================================
# 6. ENTRENAMIENTO K-FOLD
# ============================================================

import json
import time

# ------------------------------------------------------------
# 6.1. Configuración del entrenamiento
# ------------------------------------------------------------

TRAIN_ONLY_FIRST_FOLD = False
RETRAIN_EXISTING_FOLDS = False

# Métrica principal para guardar checkpoints.
# Combina segmentación del disco clínico, copa y error de vCDR.
MONITOR_METRIC = "val_clinical_selection_score"
MONITOR_MODE = "max"

TRAINING_SUMMARY_PATH = os.path.join(CFG.SAVE_PATH, "training_summary.csv")
SPLITS_DIR = os.path.join(CFG.SAVE_PATH, "fold_splits")
BACKUP_DIR = os.path.join(CFG.SAVE_PATH, "training_backups")

os.makedirs(CFG.SAVE_PATH, exist_ok=True)
os.makedirs(os.path.join(CFG.SAVE_PATH, "logs"), exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

log_info("Configuración de entrenamiento:")
log_info(f"Folds: {CFG.N_SPLITS}")
log_info(f"Epochs máximos: {CFG.EPOCHS}")
log_info(f"Batch size: {CFG.BATCH_SIZE}")
log_info(f"Learning rate inicial: {CFG.LR_START}")
log_info(f"Métrica monitorizada: {MONITOR_METRIC}")
log_info(f"Ruta de guardado: {CFG.SAVE_PATH}")


# ------------------------------------------------------------
# 6.2. Augmentation prudente para segmentación clínica
# ------------------------------------------------------------

def get_training_augmentation_clinical():
    """
    Data augmentation moderado para fondo de ojo.

    Decisiones clínicas:
        - HorizontalFlip se mantiene porque ayuda a robustez izquierda/derecha.
        - VerticalFlip se elimina porque invierte superior/inferior y puede
          distorsionar la semántica anatómica relevante para ISNT.
        - Rotaciones y traslaciones son moderadas.
        - No se aplican cambios fotométricos fuertes porque ya existe CLAHE
          y preprocess_input del backbone.
    """

    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),

            A.ShiftScaleRotate(
                shift_limit=0.04,
                scale_limit=0.08,
                rotate_limit=12,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=0.60
            ),
        ]
    )


# Sobrescribe el augmentation global usado por apply_augmentation_numpy()
# definido en la celda 4.
train_augmentation = get_training_augmentation_clinical()

log_ok("Augmentation clínico ajustado para entrenamiento.")


# ------------------------------------------------------------
# 6.3. Métricas clínicas derivadas de disco completo y copa
# ------------------------------------------------------------

def soft_binary_iou(y_true_bin, y_pred_bin):
    """
    IoU para máscaras binarias suaves.

    Se usa con probabilidades, no solo con argmax.
    """
    y_true_bin = tf.cast(y_true_bin, tf.float32)
    y_pred_bin = tf.cast(y_pred_bin, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin, axis=[1, 2])
    union = (
        tf.reduce_sum(y_true_bin, axis=[1, 2])
        + tf.reduce_sum(y_pred_bin, axis=[1, 2])
        - intersection
    )

    iou = (intersection + SMOOTH) / (union + SMOOTH)

    return tf.reduce_mean(iou)


def soft_binary_dice(y_true_bin, y_pred_bin):
    """
    Dice para máscaras binarias suaves.
    """
    y_true_bin = tf.cast(y_true_bin, tf.float32)
    y_pred_bin = tf.cast(y_pred_bin, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin, axis=[1, 2])
    denominator = (
        tf.reduce_sum(y_true_bin, axis=[1, 2])
        + tf.reduce_sum(y_pred_bin, axis=[1, 2])
    )

    dice = (2.0 * intersection + SMOOTH) / (denominator + SMOOTH)

    return tf.reduce_mean(dice)


def optic_disc_iou(y_true, y_pred):
    """
    IoU del disco óptico clínico completo.

    Importante:
        En máscaras semánticas tipo REFUGE:
            clase 1 = disco/anillo
            clase 2 = copa

        Para el cálculo clínico del disco óptico completo:
            disco completo = clase 1 + clase 2
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)

    y_true_disc = tf.clip_by_value(y_true[..., 1] + y_true[..., 2], 0.0, 1.0)
    y_pred_disc = tf.clip_by_value(y_pred[..., 1] + y_pred[..., 2], 0.0, 1.0)

    return soft_binary_iou(y_true_disc, y_pred_disc)


def optic_disc_dice(y_true, y_pred):
    """
    Dice del disco óptico clínico completo.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)

    y_true_disc = tf.clip_by_value(y_true[..., 1] + y_true[..., 2], 0.0, 1.0)
    y_pred_disc = tf.clip_by_value(y_pred[..., 1] + y_pred[..., 2], 0.0, 1.0)

    return soft_binary_dice(y_true_disc, y_pred_disc)


def rim_iou(y_true, y_pred):
    """
    IoU del anillo/disco sin copa.

    Esta métrica evalúa la clase 1, pero no debe confundirse con el disco
    óptico clínico completo usado para vCDR.
    """
    return class_iou(y_true, y_pred, class_id=1)


def clinical_global_iou(y_true, y_pred):
    """
    IoU clínico medio.

    Promedia:
        - disco óptico completo;
        - copa óptica.

    Es más adecuado para este TFG que una media genérica de clases,
    porque el biomarcador final depende de disco completo y copa.
    """
    disc = optic_disc_iou(y_true, y_pred)
    cup = iou_cup(y_true, y_pred)

    return (disc + cup) / 2.0


def vertical_diameter_from_bool_mask(mask_bool):
    """
    Calcula el diámetro vertical de una máscara binaria por batch.

    Entrada:
        mask_bool: tensor booleano de forma (B, H, W)

    Salida:
        diámetro vertical por imagen, forma (B,)
    """
    mask_bool = tf.cast(mask_bool, tf.bool)

    row_has_pixels = tf.reduce_any(mask_bool, axis=2)

    height = tf.shape(mask_bool)[1]
    rows = tf.cast(tf.range(height), tf.float32)
    rows = tf.reshape(rows, [1, -1])

    big = tf.constant(1e6, dtype=tf.float32)

    min_row = tf.reduce_min(
        tf.where(row_has_pixels, rows, big),
        axis=1
    )

    max_row = tf.reduce_max(
        tf.where(row_has_pixels, rows, -big),
        axis=1
    )

    has_pixels = tf.reduce_any(row_has_pixels, axis=1)

    diameter = tf.where(
        has_pixels,
        max_row - min_row + 1.0,
        tf.zeros_like(min_row)
    )

    return diameter


def hard_vcdr_from_prediction(y):
    """
    Calcula vCDR a partir de una máscara one-hot/probabilística.

    Se usa argmax porque el vCDR clínico se calcula sobre la segmentación final.
    """
    mask_class = tf.argmax(y, axis=-1, output_type=tf.int32)

    # Disco clínico completo = anillo/disco + copa.
    disc_mask = tf.logical_or(
        tf.equal(mask_class, 1),
        tf.equal(mask_class, 2)
    )

    cup_mask = tf.equal(mask_class, 2)

    disc_diameter = vertical_diameter_from_bool_mask(disc_mask)
    cup_diameter = vertical_diameter_from_bool_mask(cup_mask)

    vcdr = tf.where(
        disc_diameter > 0,
        cup_diameter / (disc_diameter + SMOOTH),
        tf.zeros_like(disc_diameter)
    )

    vcdr = tf.clip_by_value(vcdr, 0.0, 1.5)

    return vcdr


def hard_vcdr_mae(y_true, y_pred):
    """
    Error absoluto medio del vCDR estimado frente al vCDR derivado
    de la máscara ground truth.

    No es una loss diferenciable; se usa como métrica clínica de validación.
    """
    true_vcdr = hard_vcdr_from_prediction(y_true)
    pred_vcdr = hard_vcdr_from_prediction(y_pred)

    valid = tf.cast(true_vcdr > 0, tf.float32)

    error = tf.abs(true_vcdr - pred_vcdr)

    mae = tf.reduce_sum(error * valid) / (tf.reduce_sum(valid) + SMOOTH)

    return mae


def clinical_selection_score(y_true, y_pred):
    """
    Métrica compuesta para seleccionar el mejor checkpoint.

    Combina:
        - IoU del disco óptico completo;
        - IoU de la copa óptica;
        - error de vCDR.

    La copa recibe más peso que el disco porque es más pequeña,
    más difícil de segmentar y más determinante para el cálculo del vCDR.
    """
    disc_score = optic_disc_iou(y_true, y_pred)
    cup_score = iou_cup(y_true, y_pred)
    vcdr_error = hard_vcdr_mae(y_true, y_pred)

    vcdr_score = tf.clip_by_value(1.0 - vcdr_error, 0.0, 1.0)

    score = (
        0.30 * disc_score
        + 0.45 * cup_score
        + 0.25 * vcdr_score
    )

    return score


log_ok("Métricas clínicas de segmentación definidas.")


# ------------------------------------------------------------
# 6.4. Pérdida ponderada orientada a anillo y copa
# ------------------------------------------------------------

# Fondo, anillo/disco sin copa, copa.
# La copa se pondera más porque pequeños errores alteran mucho el vCDR.
# El fondo se pondera menos porque domina espacialmente la imagen.
CLASS_WEIGHTS = tf.constant([0.10, 0.35, 0.55], dtype=tf.float32)

# Coeficientes de Tversky por clase [fondo, disco/anillo, copa].
# alpha pondera los falsos positivos; beta pondera los falsos negativos.
# Fondo y disco: alpha = beta = 0.5 (equivalente a Dice -> disco SIN cambios).
# Copa: alpha = 0.3 < beta = 0.7 -> penaliza MAS la sobre-segmentacion (FP),
# que es la causa del sesgo sistematico +0.074 en vCDR diagnosticado en v4.
TVERSKY_ALPHA = tf.constant([0.5, 0.5, 0.3], dtype=tf.float32)
TVERSKY_BETA = tf.constant([0.5, 0.5, 0.7], dtype=tf.float32)


def weighted_multiclass_tversky_loss(y_true, y_pred):
    """
    Tversky loss multiclase ponderada (generaliza la Dice).

    Para la copa (clase 2) se usa beta > alpha para penalizar mas los
    falsos positivos (sobre-segmentacion) que los falsos negativos,
    reduciendo en origen la sobre-estimacion del vCDR. Para fondo y disco
    alpha = beta = 0.5, por lo que se comporta como la Dice anterior.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)
    y_pred = clip_predictions(y_pred)

    axes = [1, 2]

    true_pos = tf.reduce_sum(y_true * y_pred, axis=axes)
    false_pos = tf.reduce_sum((1.0 - y_true) * y_pred, axis=axes)
    false_neg = tf.reduce_sum(y_true * (1.0 - y_pred), axis=axes)

    tversky = (true_pos + SMOOTH) / (
        true_pos
        + TVERSKY_ALPHA * false_pos
        + TVERSKY_BETA * false_neg
        + SMOOTH
    )
    tversky_loss_per_class = 1.0 - tversky

    weighted_loss = tversky_loss_per_class * CLASS_WEIGHTS
    weighted_loss = tf.reduce_sum(weighted_loss, axis=-1) / tf.reduce_sum(CLASS_WEIGHTS)

    return tf.reduce_mean(weighted_loss)


def weighted_multiclass_dice_loss(y_true, y_pred):
    """
    Dice loss multiclase ponderada.

    Penaliza con más fuerza errores en estructuras clínicas pequeñas,
    especialmente la copa óptica.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)
    y_pred = clip_predictions(y_pred)

    axes = [1, 2]

    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true + y_pred, axis=axes)

    dice = (2.0 * intersection + SMOOTH) / (denominator + SMOOTH)
    dice_loss_per_class = 1.0 - dice

    weighted_loss = dice_loss_per_class * CLASS_WEIGHTS
    weighted_loss = tf.reduce_sum(weighted_loss, axis=-1) / tf.reduce_sum(CLASS_WEIGHTS)

    return tf.reduce_mean(weighted_loss)


def weighted_categorical_focal_loss(y_true, y_pred, gamma=2.0):
    """
    Focal loss multiclase ponderada.

    Aumenta la importancia relativa de píxeles difíciles y clases pequeñas.
    """
    y_true, y_pred = cast_for_metric(y_true, y_pred)
    y_pred = clip_predictions(y_pred)

    cross_entropy = -y_true * tf.math.log(y_pred)
    focal_factor = tf.pow(1.0 - y_pred, gamma)

    class_weights = tf.reshape(
        CLASS_WEIGHTS,
        [1, 1, 1, CFG.CLASSES]
    )

    loss = class_weights * focal_factor * cross_entropy
    loss = tf.reduce_sum(loss, axis=-1)

    return tf.reduce_mean(loss)


def clinical_hybrid_loss(y_true, y_pred):
    """
    Pérdida final de entrenamiento.

    Combina:
        - Tversky ponderado: solapamiento anatómico (copa con beta>alpha para penalizar la sobre-segmentación).
        - Focal ponderado: énfasis en clases difíciles.

    Se pondera la focal con 0.5 para no sobrecastigar ruido pequeño.
    """
    tversky_loss = weighted_multiclass_tversky_loss(y_true, y_pred)
    focal_loss = weighted_categorical_focal_loss(y_true, y_pred)

    return tversky_loss + 0.5 * focal_loss


CUSTOM_OBJECTS.update(
    {
        "optic_disc_iou": optic_disc_iou,
        "optic_disc_dice": optic_disc_dice,
        "rim_iou": rim_iou,
        "clinical_global_iou": clinical_global_iou,
        "hard_vcdr_mae": hard_vcdr_mae,
        "clinical_selection_score": clinical_selection_score,
        "weighted_multiclass_dice_loss": weighted_multiclass_dice_loss,
        "weighted_multiclass_tversky_loss": weighted_multiclass_tversky_loss,
        "weighted_categorical_focal_loss": weighted_categorical_focal_loss,
        "clinical_hybrid_loss": clinical_hybrid_loss,
    }
)

log_ok("Pérdida clínica ponderada definida.")


# ------------------------------------------------------------
# 6.5. Construcción del modelo de entrenamiento
# ------------------------------------------------------------

def build_training_model():
    """
    Construye y compila el modelo usado en el entrenamiento K-Fold.

    Arquitectura:
        - U-Net
        - Backbone: inceptionresnetv2
        - Salida: 3 clases semánticas
    """

    model = sm.Unet(
        CFG.BACKBONE,
        encoder_weights="imagenet",
        classes=CFG.CLASSES,
        activation="softmax"
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=CFG.LR_START
    )

    model.compile(
        optimizer=optimizer,
        loss=clinical_hybrid_loss,
        metrics=[
            clinical_selection_score,
            clinical_global_iou,
            optic_disc_iou,
            iou_cup,
            rim_iou,
            optic_disc_dice,
            dice_cup,
            hard_vcdr_mae,
        ]
    )

    return model


# ------------------------------------------------------------
# 6.6. Utilidades de entrenamiento
# ------------------------------------------------------------

def reset_fold_seed(fold_number):
    """Fija semillas específicas por fold para reproducibilidad."""
    fold_seed = CFG.SEED + fold_number

    random.seed(fold_seed)
    np.random.seed(fold_seed)
    tf.random.set_seed(fold_seed)

    log_info(f"Semilla del fold {fold_number}: {fold_seed}")


def get_model_path(fold_number):
    """Ruta del modelo principal del fold."""
    return os.path.join(
        CFG.SAVE_PATH,
        f"model_fold_{fold_number}.keras"
    )


def get_log_path(fold_number):
    """Ruta del CSVLogger del fold."""
    return os.path.join(
        CFG.SAVE_PATH,
        f"log_fold_{fold_number}.csv"
    )


def save_fold_split(fold_number, fold_train_pairs, fold_val_pairs):
    """
    Guarda los splits internos del fold para reproducibilidad.
    """
    train_split_path = os.path.join(
        SPLITS_DIR,
        f"fold_{fold_number}_train.csv"
    )

    val_split_path = os.path.join(
        SPLITS_DIR,
        f"fold_{fold_number}_val.csv"
    )

    pd.DataFrame(
        fold_train_pairs,
        columns=["image_path", "mask_path"]
    ).to_csv(train_split_path, index=False)

    pd.DataFrame(
        fold_val_pairs,
        columns=["image_path", "mask_path"]
    ).to_csv(val_split_path, index=False)

    log_ok(f"Split del fold {fold_number} guardado.")


def extract_best_metric(history, metric_name, mode="max"):
    """
    Extrae el mejor valor de una métrica desde el historial de entrenamiento.
    """
    values = history.history.get(metric_name, [])

    if len(values) == 0:
        return None

    if mode == "min":
        return float(np.min(values))

    return float(np.max(values))


def extract_last_metric(history, metric_name):
    """
    Extrae el último valor de una métrica desde el historial.
    """
    values = history.history.get(metric_name, [])

    if len(values) == 0:
        return None

    return float(values[-1])


def make_callbacks(fold_number):
    """
    Define callbacks del entrenamiento.

    Incluye:
        - ModelCheckpoint
        - EarlyStopping
        - ReduceLROnPlateau
        - CSVLogger
        - TensorBoard
        - BackupAndRestore
        - TerminateOnNaN
    """

    model_path = get_model_path(fold_number)
    log_path = get_log_path(fold_number)

    tensorboard_path = os.path.join(
        CFG.SAVE_PATH,
        "logs",
        f"fold_{fold_number}"
    )

    backup_path = os.path.join(
        BACKUP_DIR,
        f"fold_{fold_number}"
    )

    callbacks = [
        ModelCheckpoint(
            filepath=model_path,
            monitor=MONITOR_METRIC,
            mode=MONITOR_MODE,
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        EarlyStopping(
            monitor=MONITOR_METRIC,
            mode=MONITOR_MODE,
            patience=14,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor=MONITOR_METRIC,
            mode=MONITOR_MODE,
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),

        CSVLogger(
            filename=log_path,
            append=False
        ),

        TensorBoard(
            log_dir=tensorboard_path
        ),

        tf.keras.callbacks.BackupAndRestore(
            backup_dir=backup_path
        ),

        tf.keras.callbacks.TerminateOnNaN(),
    ]

    return callbacks


# ------------------------------------------------------------
# 6.7. Comprobaciones previas al entrenamiento
# ------------------------------------------------------------

if len(train_data) < CFG.N_SPLITS:
    raise ValueError(
        f"No hay suficientes muestras para {CFG.N_SPLITS} folds. "
        f"Muestras disponibles: {len(train_data)}"
    )

if len(external_val_data) == 0:
    raise ValueError(
        "La validación externa no está disponible. "
        "No se debe entrenar sin protocolo completo."
    )

log_ok(f"Muestras disponibles para K-Fold interno: {len(train_data)}")
log_ok(f"Muestras reservadas para validación externa final: {len(external_val_data)}")

# Comprobación defensiva: Validation400 no entra en el K-Fold.
verify_no_data_leakage(
    train_data,
    external_val_data,
    use_hash=False
)


# ------------------------------------------------------------
# 6.8. Entrenamiento K-Fold
# ------------------------------------------------------------

kfold = KFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED
)

training_results = []

start_global_time = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(train_data), start=1):

    if TRAIN_ONLY_FIRST_FOLD and fold_idx > 1:
        log_warning("TRAIN_ONLY_FIRST_FOLD=True. Se detiene el entrenamiento tras el primer fold.")
        break

    print("\n" + "=" * 80)
    print(f"FOLD {fold_idx}/{CFG.N_SPLITS}")
    print("=" * 80)

    fold_start_time = time.time()

    model_path = get_model_path(fold_idx)

    if os.path.exists(model_path) and not RETRAIN_EXISTING_FOLDS:
        log_warning(f"El modelo del fold {fold_idx} ya existe y no se reentrena: {model_path}")

        training_results.append(
            {
                "fold": fold_idx,
                "status": "skipped_existing",
                "model_path": model_path,
                "best_val_clinical_selection_score": None,
                "best_val_clinical_global_iou": None,
                "best_val_optic_disc_iou": None,
                "best_val_iou_cup": None,
                "best_val_hard_vcdr_mae": None,
                "epochs_ran": 0,
                "duration_min": 0,
            }
        )

        continue

    reset_fold_seed(fold_idx)

    fold_train_pairs = train_data[train_idx]
    fold_val_pairs = train_data[val_idx]

    log_info(f"Imágenes de entrenamiento interno: {len(fold_train_pairs)}")
    log_info(f"Imágenes de validación interna: {len(fold_val_pairs)}")

    save_fold_split(
        fold_number=fold_idx,
        fold_train_pairs=fold_train_pairs,
        fold_val_pairs=fold_val_pairs
    )

    tf.keras.backend.clear_session()
    gc.collect()

    train_ds = create_dataset(
        fold_train_pairs,
        augment=True,
        shuffle=True,
        batch_size=CFG.BATCH_SIZE
    )

    val_ds = create_dataset(
        fold_val_pairs,
        augment=False,
        shuffle=False,
        batch_size=CFG.BATCH_SIZE
    )

    model = build_training_model()

    callbacks = make_callbacks(fold_idx)

    log_info("Comenzando entrenamiento del fold.")

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG.EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    fold_duration_min = (time.time() - fold_start_time) / 60

    best_val_clinical_selection_score = extract_best_metric(
        history,
        "val_clinical_selection_score",
        mode="max"
    )

    best_val_clinical_global_iou = extract_best_metric(
        history,
        "val_clinical_global_iou",
        mode="max"
    )

    best_val_optic_disc_iou = extract_best_metric(
        history,
        "val_optic_disc_iou",
        mode="max"
    )

    best_val_iou_cup = extract_best_metric(
        history,
        "val_iou_cup",
        mode="max"
    )

    best_val_rim_iou = extract_best_metric(
        history,
        "val_rim_iou",
        mode="max"
    )

    best_val_optic_disc_dice = extract_best_metric(
        history,
        "val_optic_disc_dice",
        mode="max"
    )

    best_val_dice_cup = extract_best_metric(
        history,
        "val_dice_cup",
        mode="max"
    )

    best_val_hard_vcdr_mae = extract_best_metric(
        history,
        "val_hard_vcdr_mae",
        mode="min"
    )

    epochs_ran = len(history.history.get("loss", []))

    fold_result = {
        "fold": fold_idx,
        "status": "trained",
        "model_path": model_path,
        "best_val_clinical_selection_score": best_val_clinical_selection_score,
        "best_val_clinical_global_iou": best_val_clinical_global_iou,
        "best_val_optic_disc_iou": best_val_optic_disc_iou,
        "best_val_iou_cup": best_val_iou_cup,
        "best_val_rim_iou": best_val_rim_iou,
        "best_val_optic_disc_dice": best_val_optic_disc_dice,
        "best_val_dice_cup": best_val_dice_cup,
        "best_val_hard_vcdr_mae": best_val_hard_vcdr_mae,
        "last_train_loss": extract_last_metric(history, "loss"),
        "last_val_loss": extract_last_metric(history, "val_loss"),
        "epochs_ran": epochs_ran,
        "duration_min": round(fold_duration_min, 2),
    }

    training_results.append(fold_result)

    with open(
        os.path.join(CFG.SAVE_PATH, f"fold_{fold_idx}_summary.json"),
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(fold_result, file, indent=4, ensure_ascii=False)

    log_ok(f"Fold {fold_idx} completado en {fold_duration_min:.2f} minutos.")
    log_ok(f"Mejor val_clinical_selection_score: {best_val_clinical_selection_score}")
    log_ok(f"Mejor val_clinical_global_iou: {best_val_clinical_global_iou}")
    log_ok(f"Mejor val_optic_disc_iou: {best_val_optic_disc_iou}")
    log_ok(f"Mejor val_iou_cup: {best_val_iou_cup}")
    log_ok(f"Mejor val_hard_vcdr_mae: {best_val_hard_vcdr_mae}")

    del model, train_ds, val_ds, history
    tf.keras.backend.clear_session()
    gc.collect()


# ------------------------------------------------------------
# 6.9. Resumen del entrenamiento
# ------------------------------------------------------------

total_duration_min = (time.time() - start_global_time) / 60

training_summary = pd.DataFrame(training_results)
training_summary.to_csv(TRAINING_SUMMARY_PATH, index=False)

log_ok("Entrenamiento K-Fold finalizado.")
log_ok(f"Tiempo total aproximado: {total_duration_min:.2f} minutos.")
log_ok(f"Resumen guardado en: {TRAINING_SUMMARY_PATH}")

display(training_summary)

print("\nModelos esperados:")
for fold_number in range(1, CFG.N_SPLITS + 1):
    model_path = get_model_path(fold_number)
    status = "OK" if os.path.exists(model_path) else "NO ENCONTRADO"
    print(f"Fold {fold_number}: {status} -> {model_path}")

# 7. Carga y verificacion de modelos entrenados

Este bloque comprueba y carga los modelos finales `model_fold_1`.keras a `model_fold_5.keras` desde `Models_vPro_Fixed`. No reconstruye datasets mezclados ni modifica el protocolo experimental. Su función es verificar que los cinco modelos existen, que pueden cargarse correctamente, que generan predicciones válidas y que el ensemble está preparado para la inferencia posterior.



In [ ]:
# ============================================================
# 7. CARGA Y VERIFICACIÓN DE MODELOS ENTRENADOS
# ============================================================

# ------------------------------------------------------------
# 7.1. Rutas de modelos y resumen de entrenamiento
# ------------------------------------------------------------

def get_model_path(fold_number):
    """Ruta del modelo principal del fold."""
    return os.path.join(
        CFG.SAVE_PATH,
        f"model_fold_{fold_number}.keras"
    )


MODEL_PATHS = [
    get_model_path(fold)
    for fold in range(1, CFG.N_SPLITS + 1)
]

TRAINING_SUMMARY_PATH = os.path.join(
    CFG.SAVE_PATH,
    "training_summary.csv"
)

MODEL_VERIFICATION_PATH = os.path.join(
    CFG.SAVE_PATH,
    "model_verification_summary.csv"
)


def get_file_size_mb(path):
    """Devuelve el tamaño de un archivo en MB."""
    if not os.path.exists(path):
        return 0.0

    return os.path.getsize(path) / (1024 ** 2)


def verify_model_files(model_paths):
    """
    Comprueba que existen los modelos esperados y que no están vacíos.
    """

    records = []

    for fold, model_path in enumerate(model_paths, start=1):
        exists = os.path.exists(model_path)
        size_mb = get_file_size_mb(model_path)

        status = "OK" if exists and size_mb > 1 else "ERROR"

        records.append(
            {
                "fold": fold,
                "model_path": model_path,
                "exists": exists,
                "size_mb": round(size_mb, 2),
                "status": status,
            }
        )

    verification_df = pd.DataFrame(records)

    display(verification_df)

    missing = verification_df[verification_df["status"] != "OK"]

    if len(missing) > 0:
        raise FileNotFoundError(
            "Hay modelos ausentes o corruptos. Revisa la tabla de verificación."
        )

    log_ok("Todos los modelos esperados existen y tienen tamaño válido.")

    return verification_df


model_files_df = verify_model_files(MODEL_PATHS)


# ------------------------------------------------------------
# 7.2. Lectura del resumen de entrenamiento
# ------------------------------------------------------------

if os.path.exists(TRAINING_SUMMARY_PATH):
    training_summary_loaded = pd.read_csv(TRAINING_SUMMARY_PATH)
    log_ok(f"Resumen de entrenamiento cargado desde: {TRAINING_SUMMARY_PATH}")
    display(training_summary_loaded)
else:
    log_warning("No se ha encontrado training_summary.csv. Se continúa solo con verificación de modelos.")
    training_summary_loaded = None


# ------------------------------------------------------------
# 7.3. Visualización rápida de métricas internas
# ------------------------------------------------------------

def plot_training_metric(summary_df, metric_name, title=None):
    """
    Grafica una métrica por fold.
    """

    if summary_df is None:
        return

    if metric_name not in summary_df.columns:
        log_warning(f"La métrica {metric_name} no existe en el resumen.")
        return

    plt.figure(figsize=(8, 4))
    plt.bar(summary_df["fold"], summary_df[metric_name])
    plt.xlabel("Fold")
    plt.ylabel(metric_name)
    plt.title(title or metric_name)
    plt.xticks(summary_df["fold"])
    plt.grid(axis="y", alpha=0.3)
    plt.show()


if training_summary_loaded is not None:
    plot_training_metric(
        training_summary_loaded,
        "best_val_clinical_selection_score",
        "Score clínico de selección por fold"
    )

    plot_training_metric(
        training_summary_loaded,
        "best_val_iou_cup",
        "IoU de copa óptica por fold"
    )

    plot_training_metric(
        training_summary_loaded,
        "best_val_hard_vcdr_mae",
        "MAE de vCDR por fold"
    )


# ------------------------------------------------------------
# 7.4. Carga segura de modelos
# ------------------------------------------------------------

def load_single_model(fold_number, compile_model=False):
    """
    Carga un modelo individual.

    Se usa compile=False porque en esta fase solo necesitamos inferencia.
    Así evitamos depender de pérdidas y métricas personalizadas del entrenamiento.
    """

    model_path = get_model_path(fold_number)

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"No se encuentra el modelo del fold {fold_number}: {model_path}"
        )

    custom_objects = globals().get("CUSTOM_OBJECTS", {})

    model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        compile=compile_model,
        safe_mode=False
    )

    return model


# ------------------------------------------------------------
# 7.5. Batch de prueba sin usar create_dataset()
# ------------------------------------------------------------

def create_smoke_test_batch_without_dataset(n_samples=2):
    """
    Crea un batch pequeño directamente desde train_data.

    No usa create_dataset(), por lo que este punto puede ejecutarse
    aunque no se haya ejecutado el punto 4.
    """

    smoke_pairs = train_data[:min(n_samples, len(train_data))]

    images = []
    masks = []

    for image_path, mask_path in smoke_pairs:
        image, mask = preprocess_image_and_mask(
            image_path,
            mask_path
        )

        images.append(image)
        masks.append(mask)

    images = np.stack(images).astype(np.float32)
    masks = np.stack(masks).astype(np.float32)

    return images, masks, smoke_pairs


smoke_images, smoke_masks, smoke_pairs = create_smoke_test_batch_without_dataset(
    n_samples=2
)

log_info(f"Smoke batch imágenes: {smoke_images.shape}")
log_info(f"Smoke batch máscaras: {smoke_masks.shape}")


# ------------------------------------------------------------
# 7.6. Verificación secuencial de predicciones
# ------------------------------------------------------------

def check_prediction_tensor(prediction, fold_number):
    """
    Comprueba propiedades básicas de una predicción softmax.
    """

    if prediction.shape[-1] != CFG.CLASSES:
        raise ValueError(
            f"El fold {fold_number} devuelve {prediction.shape[-1]} clases, "
            f"pero se esperaban {CFG.CLASSES}."
        )

    if prediction.shape[1] != CFG.IMG_SIZE or prediction.shape[2] != CFG.IMG_SIZE:
        raise ValueError(
            f"El fold {fold_number} devuelve tamaño {prediction.shape[1:3]}, "
            f"pero se esperaba {(CFG.IMG_SIZE, CFG.IMG_SIZE)}."
        )

    softmax_sum = np.sum(prediction, axis=-1)

    mean_sum = float(np.mean(softmax_sum))
    min_sum = float(np.min(softmax_sum))
    max_sum = float(np.max(softmax_sum))

    pred_class = np.argmax(prediction, axis=-1)
    unique_classes = np.unique(pred_class)

    log_info(
        f"Fold {fold_number} | softmax sum media={mean_sum:.4f}, "
        f"min={min_sum:.4f}, max={max_sum:.4f} | clases={unique_classes}"
    )

    if not np.isfinite(prediction).all():
        raise ValueError(
            f"El fold {fold_number} ha generado NaN o Inf en la predicción."
        )

    return {
        "fold": fold_number,
        "prediction_shape": str(prediction.shape),
        "softmax_sum_mean": mean_sum,
        "softmax_sum_min": min_sum,
        "softmax_sum_max": max_sum,
        "predicted_classes": ",".join(map(str, unique_classes.tolist())),
    }


def verify_models_sequentially():
    """
    Carga cada modelo de forma secuencial, predice un batch pequeño
    y libera memoria después.
    """

    records = []

    for fold_number in range(1, CFG.N_SPLITS + 1):
        log_info(f"Verificando modelo fold {fold_number}.")

        model = load_single_model(
            fold_number,
            compile_model=False
        )

        prediction = model.predict(
            smoke_images,
            verbose=0
        )

        record = check_prediction_tensor(
            prediction,
            fold_number
        )

        records.append(record)

        del model, prediction
        tf.keras.backend.clear_session()
        gc.collect()

        log_ok(f"Modelo fold {fold_number} verificado.")

    verification_df = pd.DataFrame(records)

    verification_df.to_csv(
        MODEL_VERIFICATION_PATH,
        index=False
    )

    log_ok(f"Resumen de verificación guardado en: {MODEL_VERIFICATION_PATH}")

    return verification_df


model_prediction_verification_df = verify_models_sequentially()

display(model_prediction_verification_df)


# ------------------------------------------------------------
# 7.7. Función ensemble secuencial
# ------------------------------------------------------------

def predict_ensemble_sequential(images, use_tta=False):
    """
    Predice con el ensemble cargando los modelos de forma secuencial.

    Args:
        images:
            array de forma (B, H, W, 3).
        use_tta:
            si True, aplica test-time augmentation horizontal.

    Returns:
        predicción media del ensemble.
    """

    if isinstance(images, tf.Tensor):
        images_np = images.numpy().astype(np.float32)
    else:
        images_np = images.astype(np.float32)

    ensemble_prediction = None

    for fold_number in range(1, CFG.N_SPLITS + 1):
        log_info(f"Predicción ensemble: fold {fold_number}.")

        model = load_single_model(
            fold_number,
            compile_model=False
        )

        pred = model.predict(
            images_np,
            verbose=0
        ).astype(np.float32)

        if use_tta:
            images_flip = images_np[:, :, ::-1, :]

            pred_flip = model.predict(
                images_flip,
                verbose=0
            ).astype(np.float32)

            pred_flip = pred_flip[:, :, ::-1, :]

            pred = (pred + pred_flip) / 2.0

        if ensemble_prediction is None:
            ensemble_prediction = pred
        else:
            ensemble_prediction += pred

        del model, pred
        tf.keras.backend.clear_session()
        gc.collect()

    ensemble_prediction = ensemble_prediction / CFG.N_SPLITS

    return ensemble_prediction.astype(np.float32)


ensemble_smoke_prediction = predict_ensemble_sequential(
    smoke_images,
    use_tta=True
)

log_info(f"Predicción ensemble shape: {ensemble_smoke_prediction.shape}")

ensemble_classes = np.argmax(
    ensemble_smoke_prediction,
    axis=-1
)

log_info(f"Clases predichas por ensemble: {np.unique(ensemble_classes)}")

softmax_sum = np.sum(
    ensemble_smoke_prediction,
    axis=-1
)

log_info(f"Softmax ensemble media: {np.mean(softmax_sum):.4f}")
log_info(f"Softmax ensemble min: {np.min(softmax_sum):.4f}")
log_info(f"Softmax ensemble max: {np.max(softmax_sum):.4f}")


# ------------------------------------------------------------
# 7.8. Visualización rápida del ensemble
# ------------------------------------------------------------

def denormalize_for_display(image):
    """
    Convierte una imagen preprocesada a un rango visible aproximado.
    Solo se usa para visualización.
    """

    image = image.astype(np.float32)

    img_min = image.min()
    img_max = image.max()

    if img_max - img_min < 1e-6:
        return np.zeros_like(image)

    image = (image - img_min) / (img_max - img_min)

    return np.clip(image, 0.0, 1.0)


def visualize_ensemble_smoke_test(images, masks, predictions, n=2):
    """
    Visualiza imagen, máscara real y predicción ensemble.
    """

    n = min(n, images.shape[0])

    plt.figure(figsize=(5 * n, 10))

    for i in range(n):
        image_display = denormalize_for_display(images[i])
        true_mask = np.argmax(masks[i], axis=-1)
        pred_mask = np.argmax(predictions[i], axis=-1)

        plt.subplot(3, n, i + 1)
        plt.imshow(image_display)
        plt.title(f"Imagen {i+1}")
        plt.axis("off")

        plt.subplot(3, n, n + i + 1)
        plt.imshow(true_mask, cmap="viridis", vmin=0, vmax=2)
        plt.title("Máscara real")
        plt.axis("off")

        plt.subplot(3, n, 2 * n + i + 1)
        plt.imshow(pred_mask, cmap="viridis", vmin=0, vmax=2)
        plt.title("Predicción ensemble")
        plt.axis("off")

    plt.suptitle("Smoke test visual del ensemble", fontsize=14)
    plt.tight_layout()
    plt.show()


visualize_ensemble_smoke_test(
    smoke_images,
    smoke_masks,
    ensemble_smoke_prediction,
    n=2
)


# ------------------------------------------------------------
# 7.9. Estado final del punto 7
# ------------------------------------------------------------

log_ok("Carga y verificación de modelos entrenados completada.")
log_ok("El ensemble secuencial está preparado para inferencia.")

# 8. Inferencia y segmentación

Este bloque define el sistema de inferencia final a partir del ensemble de los cinco modelos entrenados. Aplica el mismo preprocesamiento usado en entrenamiento, realiza predicción con test-time augmentation horizontal, combina las salidas del ensemble y genera una máscara final anatómicamente coherente. No calcula todavía métricas clínicas ni valida sobre `REFUGE-Validation400`; solo prepara la segmentación final que luego alimentará el cálculo de biomarcadores.

In [ ]:
# ============================================================
# 8. INFERENCIA Y SEGMENTACIÓN
# ============================================================

# ------------------------------------------------------------
# 8.1. Configuración de inferencia
# ------------------------------------------------------------

USE_TTA_INFERENCE = True
INFERENCE_BATCH_SIZE = 4

# Postprocesamiento anatómico básico.
MIN_DISC_AREA_PIXELS = int(0.0025 * CFG.IMG_SIZE * CFG.IMG_SIZE)
MIN_CUP_AREA_PIXELS = int(0.0005 * CFG.IMG_SIZE * CFG.IMG_SIZE)

log_info("Configuración de inferencia:")
log_info(f"Use TTA: {USE_TTA_INFERENCE}")
log_info(f"Batch size inferencia: {INFERENCE_BATCH_SIZE}")
log_info(f"Área mínima disco: {MIN_DISC_AREA_PIXELS} px")
log_info(f"Área mínima copa: {MIN_CUP_AREA_PIXELS} px")


# ------------------------------------------------------------
# 8.2. Utilidades de máscaras clínicas
# ------------------------------------------------------------

def class_mask_to_clinical_masks(mask_class):
    """
    Convierte una máscara semántica 0/1/2 en máscaras clínicas binarias.

    Convención:
        0 = fondo
        1 = anillo/disco sin copa
        2 = copa

    Clínicamente:
        disco óptico completo = clase 1 + clase 2
        copa óptica = clase 2
        anillo neurorretiniano = disco completo - copa
    """

    mask_class = mask_class.astype(np.uint8)

    optic_disc_mask = np.logical_or(
        mask_class == 1,
        mask_class == 2
    )

    cup_mask = mask_class == 2

    rim_mask = np.logical_and(
        optic_disc_mask,
        np.logical_not(cup_mask)
    )

    return {
        "optic_disc": optic_disc_mask.astype(np.uint8),
        "cup": cup_mask.astype(np.uint8),
        "rim": rim_mask.astype(np.uint8),
    }


def clinical_masks_to_class_mask(optic_disc_mask, cup_mask):
    """
    Reconstruye una máscara semántica 0/1/2 desde máscaras clínicas.

    La copa sobrescribe al disco/anillo porque está dentro del disco.
    """

    optic_disc_mask = optic_disc_mask.astype(bool)
    cup_mask = cup_mask.astype(bool)

    final_mask = np.zeros(
        optic_disc_mask.shape,
        dtype=np.uint8
    )

    final_mask[optic_disc_mask] = 1
    final_mask[cup_mask] = 2

    return final_mask


# ------------------------------------------------------------
# 8.3. Postprocesamiento anatómico
# ------------------------------------------------------------

def largest_connected_component(binary_mask, min_area=0):
    """
    Conserva el mayor componente conectado de una máscara binaria.

    Si no hay componentes válidos o el mayor componente es demasiado pequeño,
    devuelve una máscara vacía.
    """

    binary_mask = binary_mask.astype(np.uint8)

    if binary_mask.max() == 0:
        return np.zeros_like(binary_mask, dtype=np.uint8), {
            "found": False,
            "area": 0,
            "num_components": 0,
        }

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask,
        connectivity=8
    )

    if num_labels <= 1:
        return np.zeros_like(binary_mask, dtype=np.uint8), {
            "found": False,
            "area": 0,
            "num_components": 0,
        }

    # Ignorar fondo, etiqueta 0.
    component_areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = int(np.argmax(component_areas)) + 1
    largest_area = int(stats[largest_idx, cv2.CC_STAT_AREA])

    if largest_area < min_area:
        return np.zeros_like(binary_mask, dtype=np.uint8), {
            "found": False,
            "area": largest_area,
            "num_components": int(num_labels - 1),
        }

    largest_mask = (labels == largest_idx).astype(np.uint8)

    return largest_mask, {
        "found": True,
        "area": largest_area,
        "num_components": int(num_labels - 1),
    }


def fill_binary_holes(binary_mask):
    """
    Rellena huecos internos de una máscara binaria.
    """

    binary_mask = binary_mask.astype(np.uint8)

    if binary_mask.max() == 0:
        return binary_mask

    h, w = binary_mask.shape

    flood = binary_mask.copy()
    mask = np.zeros((h + 2, w + 2), np.uint8)

    cv2.floodFill(
        flood,
        mask,
        seedPoint=(0, 0),
        newVal=1
    )

    flood_inv = 1 - flood
    filled = np.logical_or(binary_mask, flood_inv).astype(np.uint8)

    return filled


def postprocess_segmentation_mask(mask_class):
    """
    Aplica restricciones anatómicas simples a la segmentación final.

    Reglas:
        1. El disco clínico completo debe ser un componente principal compacto.
        2. La copa debe ser un componente principal compacto.
        3. La copa debe quedar dentro del disco.
        4. La máscara final mantiene la convención 0/1/2.
    """

    clinical_masks = class_mask_to_clinical_masks(mask_class)

    raw_disc = clinical_masks["optic_disc"]
    raw_cup = clinical_masks["cup"]

    disc_lcc, disc_info = largest_connected_component(
        raw_disc,
        min_area=MIN_DISC_AREA_PIXELS
    )

    disc_lcc = fill_binary_holes(disc_lcc)

    cup_lcc, cup_info = largest_connected_component(
        raw_cup,
        min_area=MIN_CUP_AREA_PIXELS
    )

    cup_lcc = fill_binary_holes(cup_lcc)

    # La copa no puede estar fuera del disco.
    cup_area_before = int(cup_lcc.sum())

    cup_lcc = np.logical_and(
        cup_lcc.astype(bool),
        disc_lcc.astype(bool)
    ).astype(np.uint8)

    cup_area_after = int(cup_lcc.sum())

    if cup_area_after < MIN_CUP_AREA_PIXELS:
        cup_lcc = np.zeros_like(cup_lcc, dtype=np.uint8)

    final_mask = clinical_masks_to_class_mask(
        optic_disc_mask=disc_lcc,
        cup_mask=cup_lcc
    )

    diagnostics = {
        "disc_found": disc_info["found"],
        "disc_area": int(disc_lcc.sum()),
        "disc_components_raw": disc_info["num_components"],
        "cup_found": cup_info["found"] and int(cup_lcc.sum()) >= MIN_CUP_AREA_PIXELS,
        "cup_area_before_intersection": cup_area_before,
        "cup_area": int(cup_lcc.sum()),
        "cup_components_raw": cup_info["num_components"],
        "cup_area_removed_outside_disc": int(cup_area_before - cup_area_after),
        "valid_anatomy": bool(disc_lcc.sum() > 0 and cup_lcc.sum() > 0),
    }

    return final_mask, diagnostics


# ------------------------------------------------------------
# 8.4. Incertidumbre y confianza de la predicción
# ------------------------------------------------------------

def compute_confidence_maps(probability_map):
    """
    Calcula mapas de confianza e incertidumbre desde la salida softmax.

    confidence:
        probabilidad máxima por píxel.

    entropy:
        entropía normalizada por píxel.
        0 = muy seguro
        1 = máxima incertidumbre
    """

    probability_map = probability_map.astype(np.float32)

    confidence = np.max(probability_map, axis=-1)

    eps = 1e-7
    entropy = -np.sum(
        probability_map * np.log(probability_map + eps),
        axis=-1
    )

    entropy = entropy / np.log(CFG.CLASSES)

    return {
        "confidence": confidence.astype(np.float32),
        "entropy": entropy.astype(np.float32),
    }


def summarize_prediction_quality(probability_map, postprocess_diagnostics):
    """
    Resume la calidad interna de una predicción sin usar etiqueta real.
    """

    confidence_maps = compute_confidence_maps(probability_map)

    confidence = confidence_maps["confidence"]
    entropy = confidence_maps["entropy"]

    summary = {
        "mean_confidence": float(np.mean(confidence)),
        "median_confidence": float(np.median(confidence)),
        "mean_entropy": float(np.mean(entropy)),
        "median_entropy": float(np.median(entropy)),
        "disc_found": postprocess_diagnostics["disc_found"],
        "cup_found": postprocess_diagnostics["cup_found"],
        "valid_anatomy": postprocess_diagnostics["valid_anatomy"],
    }

    return summary


# ------------------------------------------------------------
# 8.5. Predicción ensemble sobre batch
# ------------------------------------------------------------

def predict_model_with_tta(model, images_np, use_tta=True):
    """
    Predice con un modelo individual.

    TTA utilizado:
        - imagen original;
        - flip horizontal;
        - deshacer flip en la predicción;
        - promedio de ambas salidas.

    No se usa flip vertical para preservar la semántica superior/inferior.
    """

    pred = model.predict(
        images_np,
        verbose=0
    ).astype(np.float32)

    if use_tta:
        images_flip = images_np[:, :, ::-1, :]

        pred_flip = model.predict(
            images_flip,
            verbose=0
        ).astype(np.float32)

        pred_flip = pred_flip[:, :, ::-1, :]

        pred = (pred + pred_flip) / 2.0

    return pred


def predict_ensemble_batch_sequential(images, use_tta=USE_TTA_INFERENCE):
    """
    Predice un batch con el ensemble cargando cada modelo de forma secuencial.

    Esta estrategia es más lenta que mantener los cinco modelos en memoria,
    pero reduce riesgo de quedarse sin RAM/GPU en Colab.
    """

    if isinstance(images, tf.Tensor):
        images_np = images.numpy().astype(np.float32)
    else:
        images_np = images.astype(np.float32)

    ensemble_prediction = None

    for fold_number in range(1, CFG.N_SPLITS + 1):
        log_info(f"Inferencia ensemble batch | fold {fold_number}")

        model = load_single_model(
            fold_number,
            compile_model=False
        )

        pred = predict_model_with_tta(
            model=model,
            images_np=images_np,
            use_tta=use_tta
        )

        if ensemble_prediction is None:
            ensemble_prediction = pred
        else:
            ensemble_prediction += pred

        del model, pred
        tf.keras.backend.clear_session()
        gc.collect()

    ensemble_prediction = ensemble_prediction / CFG.N_SPLITS

    return ensemble_prediction.astype(np.float32)


# ------------------------------------------------------------
# 8.6. Inferencia sobre una imagen individual
# ------------------------------------------------------------

def infer_single_image(image_path, use_tta=USE_TTA_INFERENCE, apply_postprocess=True):
    """
    Ejecuta inferencia completa sobre una imagen individual.

    Devuelve:
        - imagen preprocesada para visualización;
        - mapa de probabilidades;
        - máscara raw por argmax;
        - máscara postprocesada;
        - diagnósticos anatómicos;
        - mapas de confianza/incertidumbre.
    """

    image_preprocessed, image_resized, metadata = preprocess_image_only(
        image_path
    )

    batch = np.expand_dims(
        image_preprocessed,
        axis=0
    ).astype(np.float32)

    probability_batch = predict_ensemble_batch_sequential(
        batch,
        use_tta=use_tta
    )

    probability_map = probability_batch[0]

    raw_mask = np.argmax(
        probability_map,
        axis=-1
    ).astype(np.uint8)

    if apply_postprocess:
        final_mask, postprocess_diagnostics = postprocess_segmentation_mask(
            raw_mask
        )
    else:
        final_mask = raw_mask
        postprocess_diagnostics = {
            "disc_found": True,
            "cup_found": True,
            "valid_anatomy": True,
            "disc_area": int(np.logical_or(raw_mask == 1, raw_mask == 2).sum()),
            "cup_area": int((raw_mask == 2).sum()),
        }

    confidence_maps = compute_confidence_maps(probability_map)

    quality_summary = summarize_prediction_quality(
        probability_map=probability_map,
        postprocess_diagnostics=postprocess_diagnostics
    )

    result = {
        "image_path": image_path,
        "image_resized": image_resized,
        "image_preprocessed": image_preprocessed,
        "probability_map": probability_map,
        "raw_mask": raw_mask,
        "final_mask": final_mask,
        "metadata": metadata,
        "postprocess_diagnostics": postprocess_diagnostics,
        "confidence_maps": confidence_maps,
        "quality_summary": quality_summary,
    }

    return result


# ------------------------------------------------------------
# 8.7. Inferencia sobre pares imagen-máscara
# ------------------------------------------------------------

def infer_pair(pair, use_tta=USE_TTA_INFERENCE, apply_postprocess=True):
    """
    Ejecuta inferencia sobre un par imagen-máscara.

    La máscara real se conserva solo para visualización o validación posterior.
    No calcula métricas todavía.
    """

    image_path, mask_path = pair

    result = infer_single_image(
        image_path=image_path,
        use_tta=use_tta,
        apply_postprocess=apply_postprocess
    )

    mask_gray = read_mask_grayscale(mask_path)

    image_rgb = read_rgb_image(image_path)
    cx, cy = locate_roi_center(image_rgb)

    mask_crop = crop_square_with_padding(
        mask_gray,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=True
    )

    mask_resized = cv2.resize(
        mask_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    true_mask = decode_refuge_mask(mask_resized)

    result["mask_path"] = mask_path
    result["true_mask"] = true_mask

    return result


# ------------------------------------------------------------
# 8.8. Visualización de inferencia
# ------------------------------------------------------------

def create_overlay(image_rgb, mask_class, alpha=0.35):
    """
    Crea una superposición simple de máscara sobre imagen.

    Colores:
        clase 1 = disco/anillo
        clase 2 = copa
    """

    image = image_rgb.astype(np.float32)

    if image.max() > 1.0:
        image = image / 255.0

    overlay = image.copy()

    # Disco/anillo: verde
    disc_color = np.array([0.0, 1.0, 0.0], dtype=np.float32)

    # Copa: rojo
    cup_color = np.array([1.0, 0.0, 0.0], dtype=np.float32)

    disc_region = mask_class == 1
    cup_region = mask_class == 2

    overlay[disc_region] = (
        (1.0 - alpha) * overlay[disc_region]
        + alpha * disc_color
    )

    overlay[cup_region] = (
        (1.0 - alpha) * overlay[cup_region]
        + alpha * cup_color
    )

    overlay = np.clip(overlay, 0.0, 1.0)

    return overlay


def visualize_inference_result(result, show_true_mask=True):
    """
    Visualiza una inferencia individual.
    """

    image_resized = result["image_resized"]
    raw_mask = result["raw_mask"]
    final_mask = result["final_mask"]
    entropy = result["confidence_maps"]["entropy"]
    confidence = result["confidence_maps"]["confidence"]

    has_true = show_true_mask and "true_mask" in result

    n_cols = 6 if has_true else 5

    plt.figure(figsize=(4 * n_cols, 5))

    plt.subplot(1, n_cols, 1)
    plt.imshow(image_resized)
    plt.title("Imagen ROI")
    plt.axis("off")

    col = 2

    if has_true:
        plt.subplot(1, n_cols, col)
        plt.imshow(result["true_mask"], cmap="viridis", vmin=0, vmax=2)
        plt.title("Máscara real")
        plt.axis("off")
        col += 1

    plt.subplot(1, n_cols, col)
    plt.imshow(raw_mask, cmap="viridis", vmin=0, vmax=2)
    plt.title("Predicción raw")
    plt.axis("off")
    col += 1

    plt.subplot(1, n_cols, col)
    plt.imshow(final_mask, cmap="viridis", vmin=0, vmax=2)
    plt.title("Postprocesada")
    plt.axis("off")
    col += 1

    overlay = create_overlay(
        image_resized,
        final_mask,
        alpha=0.35
    )

    plt.subplot(1, n_cols, col)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")
    col += 1

    plt.subplot(1, n_cols, col)
    plt.imshow(entropy, cmap="magma", vmin=0, vmax=1)
    plt.title("Incertidumbre")
    plt.axis("off")

    plt.suptitle(
        os.path.basename(result["image_path"]),
        fontsize=12
    )

    plt.tight_layout()
    plt.show()

    log_info("Diagnóstico interno de segmentación:")
    for key, value in result["postprocess_diagnostics"].items():
        log_info(f"{key}: {value}")

    log_info("Resumen de confianza:")
    for key, value in result["quality_summary"].items():
        log_info(f"{key}: {value}")


# ------------------------------------------------------------
# 8.9. Smoke test de inferencia
# ------------------------------------------------------------

# Usamos una muestra interna para verificar inferencia.
# La validación externa cuantitativa se hará más adelante.
test_pair = train_data[0]

inference_test_result = infer_pair(
    test_pair,
    use_tta=USE_TTA_INFERENCE,
    apply_postprocess=True
)

visualize_inference_result(
    inference_test_result,
    show_true_mask=True
)

log_ok("Inferencia y segmentación configuradas correctamente.")

# 9. Extracción de biomarcadores clínicos

Este bloque transforma la segmentación final en variables clínicas interpretables: `vCDR`, `hCDR`, `area_CDR`, `rCDR`, `área del anillo neurorretiniano` e `indicadores sectoriales` inspirados en la regla ISNT. No clasifica todavía glaucoma ni calcula AUC; su objetivo es convertir la máscara disco-copa en biomarcadores estructurales medibles y reutilizables en la validación externa. Este enfoque es coherente con la prioridad clínica del proyecto: segmentación de disco/copa y estimación robusta de vCDR como biomarcador principal.

In [ ]:
# ============================================================
# 9. EXTRACCIÓN DE BIOMARCADORES CLÍNICOS
# ============================================================

# ------------------------------------------------------------
# 9.1. Utilidades geométricas básicas
# ------------------------------------------------------------

def safe_divide(numerator, denominator, default=np.nan):
    """División segura para evitar errores por denominador cero."""
    if denominator is None or denominator == 0:
        return default

    return float(numerator) / float(denominator)


def binary_mask_area(mask):
    """Calcula el área en píxeles de una máscara binaria."""
    return int(np.sum(mask.astype(bool)))


def get_binary_bbox(mask):
    """
    Devuelve bounding box de una máscara binaria.

    Returns:
        dict con x_min, x_max, y_min, y_max, width, height.
        Si la máscara está vacía, devuelve None.
    """
    mask = mask.astype(bool)

    if not np.any(mask):
        return None

    ys, xs = np.where(mask)

    x_min = int(xs.min())
    x_max = int(xs.max())
    y_min = int(ys.min())
    y_max = int(ys.max())

    width = x_max - x_min + 1
    height = y_max - y_min + 1

    return {
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "width": int(width),
        "height": int(height),
    }


def get_mask_centroid(mask):
    """
    Calcula el centroide de una máscara binaria.

    Returns:
        (cx, cy) o (nan, nan) si la máscara está vacía.
    """
    mask_uint8 = mask.astype(np.uint8)

    if mask_uint8.sum() == 0:
        return np.nan, np.nan

    moments = cv2.moments(mask_uint8)

    if moments["m00"] == 0:
        return np.nan, np.nan

    cx = moments["m10"] / moments["m00"]
    cy = moments["m01"] / moments["m00"]

    return float(cx), float(cy)


# ------------------------------------------------------------
# 9.2. Conversión de máscara semántica a estructuras clínicas
# ------------------------------------------------------------

def get_clinical_masks_from_class_mask(mask_class):
    """
    Convierte una máscara 0/1/2 en estructuras clínicas.

    Convención semántica:
        0 = fondo
        1 = anillo/disco sin copa
        2 = copa

    Convención clínica:
        disco óptico completo = clase 1 + clase 2
        copa óptica = clase 2
        anillo neurorretiniano = disco completo - copa
    """

    mask_class = mask_class.astype(np.uint8)

    optic_disc_mask = np.logical_or(
        mask_class == 1,
        mask_class == 2
    )

    cup_mask = mask_class == 2

    rim_mask = np.logical_and(
        optic_disc_mask,
        np.logical_not(cup_mask)
    )

    return {
        "optic_disc": optic_disc_mask.astype(np.uint8),
        "cup": cup_mask.astype(np.uint8),
        "rim": rim_mask.astype(np.uint8),
    }


# ------------------------------------------------------------
# 9.3. Cálculo de CDR vertical, horizontal y de área
# ------------------------------------------------------------

def compute_cdr_metrics(optic_disc_mask, cup_mask):
    """
    Calcula relaciones copa/disco.

    Métricas:
        vCDR:
            diámetro vertical de copa / diámetro vertical de disco.

        hCDR:
            diámetro horizontal de copa / diámetro horizontal de disco.

        area_CDR:
            área de copa / área de disco.

        rCDR:
            raíz cuadrada de area_CDR.
            Representa una relación lineal equivalente derivada del área.
    """

    disc_bbox = get_binary_bbox(optic_disc_mask)
    cup_bbox = get_binary_bbox(cup_mask)

    disc_area = binary_mask_area(optic_disc_mask)
    cup_area = binary_mask_area(cup_mask)

    if disc_bbox is None:
        disc_vertical_diameter = 0
        disc_horizontal_diameter = 0
    else:
        disc_vertical_diameter = disc_bbox["height"]
        disc_horizontal_diameter = disc_bbox["width"]

    if cup_bbox is None:
        cup_vertical_diameter = 0
        cup_horizontal_diameter = 0
    else:
        cup_vertical_diameter = cup_bbox["height"]
        cup_horizontal_diameter = cup_bbox["width"]

    vcdr = safe_divide(
        cup_vertical_diameter,
        disc_vertical_diameter,
        default=np.nan
    )

    hcdr = safe_divide(
        cup_horizontal_diameter,
        disc_horizontal_diameter,
        default=np.nan
    )

    area_cdr = safe_divide(
        cup_area,
        disc_area,
        default=np.nan
    )

    if np.isnan(area_cdr):
        rcdr = np.nan
    else:
        rcdr = float(np.sqrt(max(area_cdr, 0.0)))

    return {
        "disc_vertical_diameter": int(disc_vertical_diameter),
        "cup_vertical_diameter": int(cup_vertical_diameter),
        "disc_horizontal_diameter": int(disc_horizontal_diameter),
        "cup_horizontal_diameter": int(cup_horizontal_diameter),
        "vCDR": float(vcdr) if not np.isnan(vcdr) else np.nan,
        "hCDR": float(hcdr) if not np.isnan(hcdr) else np.nan,
        "area_CDR": float(area_cdr) if not np.isnan(area_cdr) else np.nan,
        "rCDR": float(rcdr) if not np.isnan(rcdr) else np.nan,
    }


# ------------------------------------------------------------
# 9.4. Cálculo de áreas y anillo neurorretiniano
# ------------------------------------------------------------

def compute_area_metrics(optic_disc_mask, cup_mask, rim_mask):
    """
    Calcula áreas estructurales básicas.

    El anillo neurorretiniano se deriva como:
        anillo = disco óptico completo - copa óptica
    """

    disc_area = binary_mask_area(optic_disc_mask)
    cup_area = binary_mask_area(cup_mask)
    rim_area = binary_mask_area(rim_mask)

    rim_to_disc_ratio = safe_divide(
        rim_area,
        disc_area,
        default=np.nan
    )

    cup_to_rim_ratio = safe_divide(
        cup_area,
        rim_area,
        default=np.nan
    )

    return {
        "disc_area": int(disc_area),
        "cup_area": int(cup_area),
        "rim_area": int(rim_area),
        "rim_to_disc_ratio": float(rim_to_disc_ratio) if not np.isnan(rim_to_disc_ratio) else np.nan,
        "cup_to_rim_ratio": float(cup_to_rim_ratio) if not np.isnan(cup_to_rim_ratio) else np.nan,
    }


# ------------------------------------------------------------
# 9.5. Indicador sectorial inspirado en ISNT
# ------------------------------------------------------------

def compute_sector_masks(shape, center_x, center_y):
    """
    Divide la imagen en cuatro sectores angulares alrededor del centro del disco.

    Nota clínica:
        Sin conocer lateralidad ocular, izquierda/derecha de imagen no se puede
        traducir de forma estricta a nasal/temporal. Por tanto, este bloque
        calcula un indicador ISNT-like aproximado, no una regla ISNT clínica completa.
    """

    h, w = shape

    yy, xx = np.indices((h, w))

    dx = xx - center_x
    dy = yy - center_y

    angles = np.degrees(np.arctan2(dy, dx))

    superior = np.logical_and(
        angles >= -135,
        angles < -45
    )

    inferior = np.logical_and(
        angles >= 45,
        angles < 135
    )

    right = np.logical_and(
        angles >= -45,
        angles < 45
    )

    left = np.logical_or(
        angles >= 135,
        angles < -135
    )

    return {
        "superior": superior,
        "inferior": inferior,
        "left": left,
        "right": right,
    }


def compute_rim_sector_ratios(optic_disc_mask, rim_mask):
    """
    Calcula proporción de anillo por sector.

    Para cada sector:
        rim_sector_ratio = área de anillo en sector / área de disco en sector

    Este valor aproxima cuánto tejido de anillo queda en cada región.
    """

    disc_cx, disc_cy = get_mask_centroid(optic_disc_mask)

    if np.isnan(disc_cx) or np.isnan(disc_cy):
        return {
            "rim_ratio_superior": np.nan,
            "rim_ratio_inferior": np.nan,
            "rim_ratio_left": np.nan,
            "rim_ratio_right": np.nan,
            "vertical_rim_ratio_mean": np.nan,
            "horizontal_rim_ratio_mean": np.nan,
            "vertical_to_horizontal_rim_ratio": np.nan,
            "isnt_like_violation_count": np.nan,
            "isnt_like_risk": np.nan,
        }

    sector_masks = compute_sector_masks(
        shape=optic_disc_mask.shape,
        center_x=disc_cx,
        center_y=disc_cy
    )

    ratios = {}

    for sector_name, sector_mask in sector_masks.items():
        disc_sector = np.logical_and(
            optic_disc_mask.astype(bool),
            sector_mask
        )

        rim_sector = np.logical_and(
            rim_mask.astype(bool),
            sector_mask
        )

        disc_sector_area = binary_mask_area(disc_sector)
        rim_sector_area = binary_mask_area(rim_sector)

        ratio = safe_divide(
            rim_sector_area,
            disc_sector_area,
            default=np.nan
        )

        ratios[f"rim_ratio_{sector_name}"] = float(ratio) if not np.isnan(ratio) else np.nan

    superior = ratios["rim_ratio_superior"]
    inferior = ratios["rim_ratio_inferior"]
    left = ratios["rim_ratio_left"]
    right = ratios["rim_ratio_right"]

    vertical_mean = np.nanmean([superior, inferior])
    horizontal_mean = np.nanmean([left, right])

    vertical_to_horizontal = safe_divide(
        vertical_mean,
        horizontal_mean,
        default=np.nan
    )

    # Indicador simplificado:
    # En un patrón sano se espera que inferior/superior sean relativamente robustos.
    # Penalizamos si inferior no supera superior, si superior queda por debajo del
    # sector lateral más delgado o si el promedio vertical cae por debajo del horizontal.
    violations = []

    if not np.isnan(inferior) and not np.isnan(superior):
        violations.append(int(inferior <= superior))

    lateral_min = np.nanmin([left, right])

    if not np.isnan(superior) and not np.isnan(lateral_min):
        violations.append(int(superior <= lateral_min))

    if not np.isnan(vertical_mean) and not np.isnan(horizontal_mean):
        violations.append(int(vertical_mean <= horizontal_mean))

    if len(violations) == 0:
        violation_count = np.nan
        isnt_like_risk = np.nan
    else:
        violation_count = int(np.sum(violations))
        isnt_like_risk = float(np.mean(violations))

    ratios.update(
        {
            "vertical_rim_ratio_mean": float(vertical_mean) if not np.isnan(vertical_mean) else np.nan,
            "horizontal_rim_ratio_mean": float(horizontal_mean) if not np.isnan(horizontal_mean) else np.nan,
            "vertical_to_horizontal_rim_ratio": float(vertical_to_horizontal) if not np.isnan(vertical_to_horizontal) else np.nan,
            "isnt_like_violation_count": violation_count,
            "isnt_like_risk": isnt_like_risk,
        }
    )

    return ratios


# ------------------------------------------------------------
# 9.6. Desplazamiento copa-disco
# ------------------------------------------------------------

def compute_centroid_metrics(optic_disc_mask, cup_mask):
    """
    Calcula el desplazamiento relativo entre centroide de disco y copa.

    Puede ayudar a detectar excavaciones excéntricas o segmentaciones anómalas.
    """

    disc_cx, disc_cy = get_mask_centroid(optic_disc_mask)
    cup_cx, cup_cy = get_mask_centroid(cup_mask)

    disc_bbox = get_binary_bbox(optic_disc_mask)

    if disc_bbox is None:
        norm_factor = np.nan
    else:
        norm_factor = max(
            disc_bbox["width"],
            disc_bbox["height"]
        )

    if (
        np.isnan(disc_cx)
        or np.isnan(disc_cy)
        or np.isnan(cup_cx)
        or np.isnan(cup_cy)
        or np.isnan(norm_factor)
        or norm_factor == 0
    ):
        centroid_offset = np.nan
        centroid_offset_x = np.nan
        centroid_offset_y = np.nan
    else:
        centroid_offset_x = (cup_cx - disc_cx) / norm_factor
        centroid_offset_y = (cup_cy - disc_cy) / norm_factor

        centroid_offset = np.sqrt(
            centroid_offset_x ** 2
            + centroid_offset_y ** 2
        )

    return {
        "disc_centroid_x": float(disc_cx) if not np.isnan(disc_cx) else np.nan,
        "disc_centroid_y": float(disc_cy) if not np.isnan(disc_cy) else np.nan,
        "cup_centroid_x": float(cup_cx) if not np.isnan(cup_cx) else np.nan,
        "cup_centroid_y": float(cup_cy) if not np.isnan(cup_cy) else np.nan,
        "cup_disc_centroid_offset_x": float(centroid_offset_x) if not np.isnan(centroid_offset_x) else np.nan,
        "cup_disc_centroid_offset_y": float(centroid_offset_y) if not np.isnan(centroid_offset_y) else np.nan,
        "cup_disc_centroid_offset": float(centroid_offset) if not np.isnan(centroid_offset) else np.nan,
    }


# ------------------------------------------------------------
# 9.7. Función principal de biomarcadores
# ------------------------------------------------------------

def compute_biomarkers_from_mask(mask_class):
    """
    Calcula todos los biomarcadores clínicos derivados de una máscara 0/1/2.

    Returns:
        dict con variables geométricas y clínicas.
    """

    clinical_masks = get_clinical_masks_from_class_mask(mask_class)

    optic_disc_mask = clinical_masks["optic_disc"]
    cup_mask = clinical_masks["cup"]
    rim_mask = clinical_masks["rim"]

    area_metrics = compute_area_metrics(
        optic_disc_mask=optic_disc_mask,
        cup_mask=cup_mask,
        rim_mask=rim_mask
    )

    cdr_metrics = compute_cdr_metrics(
        optic_disc_mask=optic_disc_mask,
        cup_mask=cup_mask
    )

    sector_metrics = compute_rim_sector_ratios(
        optic_disc_mask=optic_disc_mask,
        rim_mask=rim_mask
    )

    centroid_metrics = compute_centroid_metrics(
        optic_disc_mask=optic_disc_mask,
        cup_mask=cup_mask
    )

    biomarkers = {}

    biomarkers.update(area_metrics)
    biomarkers.update(cdr_metrics)
    biomarkers.update(sector_metrics)
    biomarkers.update(centroid_metrics)

    biomarkers["valid_disc"] = bool(area_metrics["disc_area"] > 0)
    biomarkers["valid_cup"] = bool(area_metrics["cup_area"] > 0)
    biomarkers["valid_biomarkers"] = bool(
        biomarkers["valid_disc"]
        and biomarkers["valid_cup"]
        and not np.isnan(biomarkers["vCDR"])
    )

    return biomarkers


def extract_biomarkers_from_inference_result(result):
    """
    Extrae biomarcadores desde un resultado de inferencia del punto 8.

    Calcula:
        - biomarcadores de la máscara predicha postprocesada;
        - biomarcadores de la máscara raw;
        - biomarcadores de la máscara real, si existe.
    """

    output = {
        "image_path": result.get("image_path", None),
    }

    final_biomarkers = compute_biomarkers_from_mask(
        result["final_mask"]
    )

    raw_biomarkers = compute_biomarkers_from_mask(
        result["raw_mask"]
    )

    for key, value in final_biomarkers.items():
        output[f"pred_{key}"] = value

    for key, value in raw_biomarkers.items():
        output[f"raw_{key}"] = value

    if "true_mask" in result:
        true_biomarkers = compute_biomarkers_from_mask(
            result["true_mask"]
        )

        for key, value in true_biomarkers.items():
            output[f"true_{key}"] = value

        if (
            not np.isnan(output.get("pred_vCDR", np.nan))
            and not np.isnan(output.get("true_vCDR", np.nan))
        ):
            output["abs_error_vCDR"] = abs(
                output["pred_vCDR"] - output["true_vCDR"]
            )
        else:
            output["abs_error_vCDR"] = np.nan

    if "quality_summary" in result:
        for key, value in result["quality_summary"].items():
            output[f"quality_{key}"] = value

    return output


# ------------------------------------------------------------
# 9.8. Visualización de biomarcadores
# ------------------------------------------------------------

def draw_bbox_on_image(image, mask, color=(255, 255, 255), thickness=2):
    """
    Dibuja bounding box de una máscara sobre una imagen.
    """
    image_out = image.copy()

    if image_out.max() <= 1.0:
        image_out = (image_out * 255).astype(np.uint8)
    else:
        image_out = image_out.astype(np.uint8)

    bbox = get_binary_bbox(mask)

    if bbox is not None:
        cv2.rectangle(
            image_out,
            (bbox["x_min"], bbox["y_min"]),
            (bbox["x_max"], bbox["y_max"]),
            color,
            thickness
        )

    return image_out


def visualize_biomarkers(result, biomarker_row=None):
    """
    Visualiza imagen, segmentación y geometría usada para biomarcadores.
    """

    if biomarker_row is None:
        biomarker_row = extract_biomarkers_from_inference_result(result)

    image = result["image_resized"]
    final_mask = result["final_mask"]

    clinical_masks = get_clinical_masks_from_class_mask(final_mask)

    optic_disc_mask = clinical_masks["optic_disc"]
    cup_mask = clinical_masks["cup"]
    rim_mask = clinical_masks["rim"]

    overlay = create_overlay(
        image,
        final_mask,
        alpha=0.35
    )

    bbox_image = image.copy()

    if bbox_image.max() <= 1.0:
        bbox_image = (bbox_image * 255).astype(np.uint8)
    else:
        bbox_image = bbox_image.astype(np.uint8)

    disc_bbox = get_binary_bbox(optic_disc_mask)
    cup_bbox = get_binary_bbox(cup_mask)

    if disc_bbox is not None:
        cv2.rectangle(
            bbox_image,
            (disc_bbox["x_min"], disc_bbox["y_min"]),
            (disc_bbox["x_max"], disc_bbox["y_max"]),
            (0, 255, 0),
            2
        )

    if cup_bbox is not None:
        cv2.rectangle(
            bbox_image,
            (cup_bbox["x_min"], cup_bbox["y_min"]),
            (cup_bbox["x_max"], cup_bbox["y_max"]),
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title("Imagen ROI")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(overlay)
    plt.title("Overlay disco/copa")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(bbox_image)
    plt.title("Diámetros para CDR")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(rim_mask, cmap="gray")
    plt.title("Anillo neurorretiniano")
    plt.axis("off")

    plt.suptitle(
        f"vCDR={biomarker_row.get('pred_vCDR', np.nan):.3f} | "
        f"hCDR={biomarker_row.get('pred_hCDR', np.nan):.3f} | "
        f"area_CDR={biomarker_row.get('pred_area_CDR', np.nan):.3f} | "
        f"ISNT-like risk={biomarker_row.get('pred_isnt_like_risk', np.nan):.3f}",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 9.9. Smoke test sobre la inferencia del punto 8
# ------------------------------------------------------------

biomarker_test_row = extract_biomarkers_from_inference_result(
    inference_test_result
)

biomarker_test_df = pd.DataFrame([biomarker_test_row])

selected_columns = [
    "pred_disc_area",
    "pred_cup_area",
    "pred_rim_area",
    "pred_vCDR",
    "pred_hCDR",
    "pred_area_CDR",
    "pred_rCDR",
    "pred_rim_to_disc_ratio",
    "pred_rim_ratio_inferior",
    "pred_rim_ratio_superior",
    "pred_rim_ratio_left",
    "pred_rim_ratio_right",
    "pred_vertical_to_horizontal_rim_ratio",
    "pred_isnt_like_risk",
    "true_vCDR",
    "abs_error_vCDR",
]

available_selected_columns = [
    col for col in selected_columns
    if col in biomarker_test_df.columns
]

display(biomarker_test_df[available_selected_columns])

visualize_biomarkers(
    inference_test_result,
    biomarker_row=biomarker_test_row
)

log_ok("Extracción de biomarcadores clínicos configurada correctamente.")

# 10. Validación externa
Este bloque evalúa el sistema final sobre `REFUGE-Validation400`, que no se ha usado durante el entrenamiento. Aplica el ensemble a todas las imágenes externas, calcula métricas de segmentación disco/copa, error de vCDR, biomarcadores clínicos y, si encuentra las etiquetas de glaucoma, calcula `AUC`, `curva ROC`, `umbral óptimo`, `sensibilidad`, `especificidad`, `precisión`, `F1` y `matriz de confusión`. No reentrena modelos ni modifica los folds.

In [ ]:
# ============================================================
# 10. VALIDACIÓN EXTERNA
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ------------------------------------------------------------
# 10.1. Configuración de validación externa
# ------------------------------------------------------------

EXTERNAL_RESULTS_PATH = os.path.join(
    CFG.SAVE_PATH,
    "external_validation_results.csv"
)

EXTERNAL_SUMMARY_PATH = os.path.join(
    CFG.SAVE_PATH,
    "external_validation_summary.csv"
)

PREDICTION_MEMMAP_PATH = "/content/external_val_ensemble_predictions.dat"

VALIDATION_BATCH_SIZE = INFERENCE_BATCH_SIZE

log_info("Configuración de validación externa:")
log_info(f"Dataset externo: {len(external_val_data)} imágenes")
log_info(f"Batch size: {VALIDATION_BATCH_SIZE}")
log_info(f"Ruta resultados: {EXTERNAL_RESULTS_PATH}")


# ------------------------------------------------------------
# 10.2. Métricas binarias de segmentación
# ------------------------------------------------------------

def binary_iou_np(y_true, y_pred, smooth=1e-6):
    """IoU binario en NumPy."""
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()

    return float((intersection + smooth) / (union + smooth))


def binary_dice_np(y_true, y_pred, smooth=1e-6):
    """Dice binario en NumPy."""
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    intersection = np.logical_and(y_true, y_pred).sum()
    denominator = y_true.sum() + y_pred.sum()

    return float((2.0 * intersection + smooth) / (denominator + smooth))


def compute_segmentation_metrics_np(true_mask, pred_mask):
    """
    Calcula métricas de segmentación sobre disco clínico completo y copa.

    Convención:
        disco clínico completo = clase 1 + clase 2
        copa = clase 2
    """

    true_clinical = get_clinical_masks_from_class_mask(true_mask)
    pred_clinical = get_clinical_masks_from_class_mask(pred_mask)

    true_disc = true_clinical["optic_disc"]
    pred_disc = pred_clinical["optic_disc"]

    true_cup = true_clinical["cup"]
    pred_cup = pred_clinical["cup"]

    metrics = {
        "seg_disc_iou": binary_iou_np(true_disc, pred_disc),
        "seg_disc_dice": binary_dice_np(true_disc, pred_disc),
        "seg_cup_iou": binary_iou_np(true_cup, pred_cup),
        "seg_cup_dice": binary_dice_np(true_cup, pred_cup),
    }

    metrics["seg_clinical_global_iou"] = (
        metrics["seg_disc_iou"]
        + metrics["seg_cup_iou"]
    ) / 2.0

    metrics["seg_clinical_global_dice"] = (
        metrics["seg_disc_dice"]
        + metrics["seg_cup_dice"]
    ) / 2.0

    return metrics


# ------------------------------------------------------------
# 10.3. Carga de etiquetas clínicas de glaucoma
# ------------------------------------------------------------

def find_external_label_file():
    """
    Busca archivo con etiquetas clínicas dentro de REFUGE-Validation400-GT.

    En tu notebook anterior se usaba:
        Fovea_locations.xlsx

    y la columna:
        Glaucoma Label
    """

    val_gt_root = os.path.join(
        CFG.BASE_PATH,
        "REFUGE-Validation400-GT",
        "REFUGE-Validation400-GT"
    )

    candidate_names = [
        "Fovea_locations.xlsx",
        "Fovea_locations.xls",
        "Fovea_locations.csv",
        "Glaucoma_label.xlsx",
        "Glaucoma_label.csv",
        "glaucoma_label.xlsx",
        "glaucoma_label.csv",
        "labels.xlsx",
        "labels.csv",
    ]

    for name in candidate_names:
        candidate_path = os.path.join(val_gt_root, name)

        if os.path.exists(candidate_path):
            return candidate_path

    files = []

    for ext in ["*.xlsx", "*.xls", "*.csv"]:
        files.extend(
            glob.glob(
                os.path.join(val_gt_root, "**", ext),
                recursive=True
            )
        )

    if len(files) > 0:
        return files[0]

    return None


def normalize_label_key(value):
    """
    Normaliza identificadores de imagen para cruzar etiquetas con rutas.
    """
    value = str(value)
    value = os.path.basename(value)
    value = os.path.splitext(value)[0]
    value = value.strip().lower()

    return value


def load_external_glaucoma_labels():
    """
    Carga etiquetas clínicas de glaucoma si están disponibles.

    Devuelve:
        label_dict:
            dict {image_id_normalizado: 0/1}
        label_file:
            ruta del archivo usado o None
    """

    label_file = find_external_label_file()

    if label_file is None:
        log_warning("No se ha encontrado archivo de etiquetas clínicas de glaucoma.")
        return {}, None

    log_info(f"Archivo de etiquetas detectado: {label_file}")

    if label_file.endswith(".csv"):
        df = pd.read_csv(label_file)
    else:
        df = pd.read_excel(label_file)

    log_info(f"Columnas disponibles en etiquetas: {list(df.columns)}")

    possible_name_cols = [
        col for col in df.columns
        if (
            "img" in str(col).lower()
            or "image" in str(col).lower()
            or "name" in str(col).lower()
            or "filename" in str(col).lower()
            or "file" in str(col).lower()
        )
    ]

    possible_label_cols = [
        col for col in df.columns
        if (
            "glaucoma" in str(col).lower()
            or "label" in str(col).lower()
            or "diagnosis" in str(col).lower()
        )
    ]

    if len(possible_name_cols) == 0:
        raise ValueError(
            "No se ha podido identificar la columna de nombre de imagen "
            f"en el archivo: {label_file}"
        )

    if len(possible_label_cols) == 0:
        raise ValueError(
            "No se ha podido identificar la columna de etiqueta de glaucoma "
            f"en el archivo: {label_file}"
        )

    name_col = possible_name_cols[0]

    # Preferir explícitamente Glaucoma Label si existe.
    glaucoma_label_candidates = [
        col for col in possible_label_cols
        if "glaucoma" in str(col).lower()
    ]

    if len(glaucoma_label_candidates) > 0:
        label_col = glaucoma_label_candidates[0]
    else:
        label_col = possible_label_cols[0]

    log_ok(f"Columna de imagen: {name_col}")
    log_ok(f"Columna de etiqueta: {label_col}")

    label_dict = {}

    for _, row in df.iterrows():
        image_key = normalize_label_key(row[name_col])

        try:
            label = int(row[label_col])
        except Exception:
            continue

        label_dict[image_key] = label

    log_ok(f"Etiquetas clínicas cargadas: {len(label_dict)}")

    return label_dict, label_file


external_label_dict, external_label_file = load_external_glaucoma_labels()


def get_label_for_image_path(image_path, label_dict):
    """
    Recupera etiqueta clínica para una imagen.

    Intenta primero por nombre normalizado y después por clave numérica.
    """

    stem_key = normalize_label_key(image_path)

    if stem_key in label_dict:
        return label_dict[stem_key]

    num_key = numeric_key(image_path)

    if num_key is not None:
        # Intento por número exacto y por variantes con ceros.
        if num_key in label_dict:
            return label_dict[num_key]

        for key, value in label_dict.items():
            if numeric_key(key) == num_key:
                return value

    return np.nan


# ------------------------------------------------------------
# 10.4. Preprocesamiento externo sin create_dataset()
# ------------------------------------------------------------

EXTERNAL_IMAGES_MEMMAP_PATH = "/content/external_val_images_preprocessed.dat"
EXTERNAL_TRUE_MASKS_MEMMAP_PATH = "/content/external_val_true_masks.dat"


def preprocess_external_validation_to_memmap(force_rebuild=False):
    """
    Preprocesa REFUGE-Validation400 una sola vez y guarda:
        - imágenes preprocesadas en memmap;
        - máscaras reales en formato clase 0/1/2 en memmap.

    No usa create_dataset().
    """

    n_samples = len(external_val_data)

    if force_rebuild:
        for path in [
            EXTERNAL_IMAGES_MEMMAP_PATH,
            EXTERNAL_TRUE_MASKS_MEMMAP_PATH,
        ]:
            if os.path.exists(path):
                os.remove(path)

    images_ready = os.path.exists(EXTERNAL_IMAGES_MEMMAP_PATH)
    masks_ready = os.path.exists(EXTERNAL_TRUE_MASKS_MEMMAP_PATH)

    images_memmap = np.memmap(
        EXTERNAL_IMAGES_MEMMAP_PATH,
        dtype="float32",
        mode="w+" if not images_ready else "r+",
        shape=(n_samples, CFG.IMG_SIZE, CFG.IMG_SIZE, 3)
    )

    masks_memmap = np.memmap(
        EXTERNAL_TRUE_MASKS_MEMMAP_PATH,
        dtype="uint8",
        mode="w+" if not masks_ready else "r+",
        shape=(n_samples, CFG.IMG_SIZE, CFG.IMG_SIZE)
    )

    if images_ready and masks_ready and not force_rebuild:
        log_info("Preprocesamiento externo ya existe en memmap. Se reutiliza.")
        return images_memmap, masks_memmap

    log_info("Preprocesando validación externa sin create_dataset().")

    for idx, (image_path, mask_path) in enumerate(external_val_data):
        if idx % 50 == 0:
            log_info(f"Preprocesando imagen externa {idx}/{n_samples}")

        image_preprocessed, mask_onehot = preprocess_image_and_mask(
            image_path,
            mask_path
        )

        mask_class = np.argmax(
            mask_onehot,
            axis=-1
        ).astype(np.uint8)

        images_memmap[idx] = image_preprocessed.astype(np.float32)
        masks_memmap[idx] = mask_class

    images_memmap.flush()
    masks_memmap.flush()

    log_ok("Preprocesamiento externo guardado en memmap.")

    return images_memmap, masks_memmap


# force_rebuild=True: tras cambiar locate_roi_center hay que regenerar el memmap para
# no reutilizar recortes de la ROI antigua dentro de la misma sesion de Colab.
external_images_memmap, external_true_masks_memmap = preprocess_external_validation_to_memmap(
    force_rebuild=True
)


# ------------------------------------------------------------
# 10.5. Predicción eficiente del ensemble sin create_dataset()
# ------------------------------------------------------------

def iter_memmap_batches(images_memmap, batch_size):
    """
    Iterador de batches sobre el memmap de imágenes externas.
    """

    n_samples = images_memmap.shape[0]

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)

        batch = np.array(
            images_memmap[start:end],
            dtype=np.float32
        )

        yield start, end, batch


def predict_external_ensemble_to_memmap():
    """
    Predice REFUGE-Validation400 con el ensemble.

    No usa create_dataset().
    Carga un modelo por fold y recorre las imágenes preprocesadas en batches.
    """

    n_samples = len(external_val_data)

    if os.path.exists(PREDICTION_MEMMAP_PATH):
        os.remove(PREDICTION_MEMMAP_PATH)

    pred_sum = np.memmap(
        PREDICTION_MEMMAP_PATH,
        dtype="float32",
        mode="w+",
        shape=(n_samples, CFG.IMG_SIZE, CFG.IMG_SIZE, CFG.CLASSES)
    )

    pred_sum[:] = 0.0
    pred_sum.flush()

    for fold_number in range(1, CFG.N_SPLITS + 1):
        print("\n" + "-" * 80)
        log_info(f"Predicción externa con fold {fold_number}/{CFG.N_SPLITS}")
        print("-" * 80)

        model = load_single_model(
            fold_number,
            compile_model=False
        )

        for start, end, batch_images in iter_memmap_batches(
            external_images_memmap,
            VALIDATION_BATCH_SIZE
        ):
            pred = predict_model_with_tta(
                model=model,
                images_np=batch_images,
                use_tta=USE_TTA_INFERENCE
            )

            pred_sum[start:end] += pred.astype(np.float32)

        pred_sum.flush()

        del model
        tf.keras.backend.clear_session()
        gc.collect()

        log_ok(f"Fold {fold_number} acumulado correctamente.")

    pred_sum[:] = pred_sum[:] / CFG.N_SPLITS
    pred_sum.flush()

    log_ok("Predicción externa del ensemble completada.")

    return pred_sum


external_pred_memmap = predict_external_ensemble_to_memmap()


# ------------------------------------------------------------
# 10.5B. Construcción de tabla de resultados externos
# ------------------------------------------------------------

def build_external_validation_results(pred_memmap, true_masks_memmap):
    """
    Construye una tabla por imagen con:
        - métricas de segmentación;
        - biomarcadores predichos;
        - biomarcadores reales derivados de máscara GT;
        - error de vCDR;
        - score de riesgo heurístico;
        - etiqueta clínica si está disponible.

    No usa create_dataset().
    """

    records = []

    n_samples = len(external_val_data)

    for global_index in range(n_samples):
        if global_index % 50 == 0:
            log_info(f"Construyendo resultados externos {global_index}/{n_samples}")

        image_path, mask_path = external_val_data[global_index]

        probability_map = np.array(
            pred_memmap[global_index],
            dtype=np.float32
        )

        raw_mask = np.argmax(
            probability_map,
            axis=-1
        ).astype(np.uint8)

        final_mask, postprocess_diagnostics = postprocess_segmentation_mask(
            raw_mask
        )

        true_mask = np.array(
            true_masks_memmap[global_index],
            dtype=np.uint8
        )

        seg_metrics = compute_segmentation_metrics_np(
            true_mask=true_mask,
            pred_mask=final_mask
        )

        pred_biomarkers = compute_biomarkers_from_mask(
            final_mask
        )

        raw_biomarkers = compute_biomarkers_from_mask(
            raw_mask
        )

        true_biomarkers = compute_biomarkers_from_mask(
            true_mask
        )

        confidence_maps = compute_confidence_maps(
            probability_map
        )

        quality_summary = summarize_prediction_quality(
            probability_map=probability_map,
            postprocess_diagnostics=postprocess_diagnostics
        )

        true_label = get_label_for_image_path(
            image_path,
            external_label_dict
        )

        record = {
            "index": global_index,
            "image_path": image_path,
            "mask_path": mask_path,
            "image_id": normalize_label_key(image_path),
            "true_glaucoma_label": true_label,
        }

        record.update(seg_metrics)

        for key, value in pred_biomarkers.items():
            record[f"pred_{key}"] = value

        for key, value in raw_biomarkers.items():
            record[f"raw_{key}"] = value

        for key, value in true_biomarkers.items():
            record[f"true_{key}"] = value

        if (
            not np.isnan(record.get("pred_vCDR", np.nan))
            and not np.isnan(record.get("true_vCDR", np.nan))
        ):
            record["abs_error_vCDR"] = abs(
                record["pred_vCDR"]
                - record["true_vCDR"]
            )
        else:
            record["abs_error_vCDR"] = np.nan

        # Score heurístico clínico.
        vcdr_score = np.clip(
            (record.get("pred_vCDR", np.nan) - 0.45) / 0.40,
            0.0,
            1.0
        )

        rcdr_score = np.clip(
            (record.get("pred_rCDR", np.nan) - 0.50) / 0.35,
            0.0,
            1.0
        )

        isnt_risk = record.get("pred_isnt_like_risk", np.nan)

        if isinstance(isnt_risk, str):
            try:
                isnt_risk = float(isnt_risk)
            except Exception:
                isnt_risk = np.nan

        if np.isnan(vcdr_score):
            vcdr_score = 0.0

        if np.isnan(rcdr_score):
            rcdr_score = 0.0

        if np.isnan(isnt_risk):
            isnt_risk = 0.5

        record["risk_score_vcdr_only"] = float(vcdr_score)

        record["risk_score_combined"] = float(
            0.70 * vcdr_score
            + 0.20 * rcdr_score
            + 0.10 * isnt_risk
        )

        for key, value in postprocess_diagnostics.items():
            record[f"post_{key}"] = value

        for key, value in quality_summary.items():
            record[f"quality_{key}"] = value

        records.append(record)

    results_df = pd.DataFrame(records)

    return results_df


external_results_df = build_external_validation_results(
    pred_memmap=external_pred_memmap,
    true_masks_memmap=external_true_masks_memmap
)

external_results_df.to_csv(
    EXTERNAL_RESULTS_PATH,
    index=False
)

log_ok(f"Resultados externos guardados en: {EXTERNAL_RESULTS_PATH}")

display(external_results_df.head())

# ------------------------------------------------------------
# 10.6. Resumen de métricas de segmentación y biomarcadores
# ------------------------------------------------------------

def summarize_external_segmentation(results_df):
    """
    Resume métricas de segmentación y error de biomarcadores.
    """

    metric_cols = [
        "seg_disc_iou",
        "seg_disc_dice",
        "seg_cup_iou",
        "seg_cup_dice",
        "seg_clinical_global_iou",
        "seg_clinical_global_dice",
        "abs_error_vCDR",
        "pred_vCDR",
        "true_vCDR",
        "pred_area_CDR",
        "true_area_CDR",
        "pred_rCDR",
        "true_rCDR",
    ]

    rows = []

    for col in metric_cols:
        if col not in results_df.columns:
            continue

        values = pd.to_numeric(
            results_df[col],
            errors="coerce"
        ).dropna()

        rows.append(
            {
                "metric": col,
                "mean": values.mean(),
                "std": values.std(),
                "median": values.median(),
                "min": values.min(),
                "max": values.max(),
            }
        )

    summary_df = pd.DataFrame(rows)

    return summary_df


external_segmentation_summary_df = summarize_external_segmentation(
    external_results_df
)

display(external_segmentation_summary_df)


# ------------------------------------------------------------
# 10.7. Curvas ROC y métricas de clasificación
# ------------------------------------------------------------

def compute_classification_metrics(results_df, score_col):
    """
    Calcula AUC, umbral óptimo por Youden y métricas de clasificación.

    Nota:
        El umbral se calibra sobre REFUGE-Validation400, por lo que debe
        reportarse como umbral calibrado en validación, no como evaluación
        independiente sobre test externo.
    """

    if "true_glaucoma_label" not in results_df.columns:
        log_warning("No existe columna true_glaucoma_label.")
        return None

    df = results_df.copy()

    df["true_glaucoma_label"] = pd.to_numeric(
        df["true_glaucoma_label"],
        errors="coerce"
    )

    df[score_col] = pd.to_numeric(
        df[score_col],
        errors="coerce"
    )

    df = df.dropna(
        subset=["true_glaucoma_label", score_col]
    )

    if len(df) == 0:
        log_warning(f"No hay filas válidas para calcular clasificación con {score_col}.")
        return None

    y_true = df["true_glaucoma_label"].astype(int).values
    y_score = df[score_col].astype(float).values

    unique_labels = np.unique(y_true)

    if len(unique_labels) < 2:
        log_warning(
            f"No hay suficientes clases para ROC con {score_col}. "
            f"Clases encontradas: {unique_labels}"
        )
        return None

    auc = roc_auc_score(y_true, y_score)

    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    youden = tpr - fpr
    best_idx = int(np.argmax(youden))
    best_threshold = float(thresholds[best_idx])

    y_pred = (y_score >= best_threshold).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    metrics = {
        "score_col": score_col,
        "n_samples": int(len(df)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
        "auc": float(auc),
        "best_threshold_youden": best_threshold,
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "accuracy": float(accuracy),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
    }

    return metrics


classification_metrics_vcdr = compute_classification_metrics(
    external_results_df,
    score_col="risk_score_vcdr_only"
)

classification_metrics_combined = compute_classification_metrics(
    external_results_df,
    score_col="risk_score_combined"
)


# ------------------------------------------------------------
# 10.8. Visualización ROC y matriz de confusión
# ------------------------------------------------------------

def plot_roc_metrics(metrics_list):
    """
    Grafica curvas ROC de una o varias puntuaciones.
    """

    valid_metrics = [
        item for item in metrics_list
        if item is not None
    ]

    if len(valid_metrics) == 0:
        log_warning("No hay métricas ROC disponibles para graficar.")
        return

    plt.figure(figsize=(7, 6))

    for metrics in valid_metrics:
        label = (
            f"{metrics['score_col']} "
            f"(AUC={metrics['auc']:.3f})"
        )

        plt.plot(
            metrics["fpr"],
            metrics["tpr"],
            label=label
        )

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

    plt.xlabel("1 - Especificidad")
    plt.ylabel("Sensibilidad")
    plt.title("Curva ROC en REFUGE-Validation400")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def plot_confusion_matrix_from_metrics(metrics):
    """
    Grafica matriz de confusión.
    """

    if metrics is None:
        return

    cm = np.array(
        [
            [metrics["tn"], metrics["fp"]],
            [metrics["fn"], metrics["tp"]],
        ]
    )

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred sano", "Pred glaucoma"],
        yticklabels=["Real sano", "Real glaucoma"]
    )

    plt.title(
        f"Matriz de confusión - {metrics['score_col']}\n"
        f"Umbral={metrics['best_threshold_youden']:.3f}"
    )
    plt.ylabel("Etiqueta real")
    plt.xlabel("Predicción")
    plt.tight_layout()
    plt.show()


plot_roc_metrics(
    [
        classification_metrics_vcdr,
        classification_metrics_combined,
    ]
)

plot_confusion_matrix_from_metrics(
    classification_metrics_combined
)


# ------------------------------------------------------------
# 10.9. Tabla resumen final de validación externa
# ------------------------------------------------------------

summary_records = []

for _, row in external_segmentation_summary_df.iterrows():
    summary_records.append(
        {
            "section": "segmentation_biomarkers",
            "metric": row["metric"],
            "value": row["mean"],
            "std": row["std"],
        }
    )

for metrics in [classification_metrics_vcdr, classification_metrics_combined]:
    if metrics is None:
        continue

    for key in [
        "auc",
        "best_threshold_youden",
        "sensitivity",
        "specificity",
        "precision",
        "recall",
        "f1",
        "accuracy",
        "n_samples",
        "n_positive",
        "n_negative",
        "tn",
        "fp",
        "fn",
        "tp",
    ]:
        summary_records.append(
            {
                "section": metrics["score_col"],
                "metric": key,
                "value": metrics[key],
                "std": np.nan,
            }
        )

external_summary_df = pd.DataFrame(summary_records)

external_summary_df.to_csv(
    EXTERNAL_SUMMARY_PATH,
    index=False
)

log_ok(f"Resumen externo guardado en: {EXTERNAL_SUMMARY_PATH}")

display(external_summary_df)


# ------------------------------------------------------------
# 10.10. Conclusión automática del bloque
# ------------------------------------------------------------

log_ok("Validación externa completada.")

if classification_metrics_combined is not None:
    log_info(
        f"AUC combinado: {classification_metrics_combined['auc']:.4f} | "
        f"Sensibilidad: {classification_metrics_combined['sensitivity']:.4f} | "
        f"Especificidad: {classification_metrics_combined['specificity']:.4f} | "
        f"F1: {classification_metrics_combined['f1']:.4f}"
    )

if classification_metrics_vcdr is not None:
    log_info(
        f"AUC vCDR solo: {classification_metrics_vcdr['auc']:.4f}"
    )

mean_vcdr_mae = external_results_df["abs_error_vCDR"].mean()
log_info(f"MAE externo de vCDR: {mean_vcdr_mae:.4f}")

# 11. Calibración clínica y estudio de ablación

Este bloque analiza los resultados de `REFUGE-Validation400` para decidir qué biomarcadores aportan valor y qué umbral de decisión es más defendible. Compara `vCDR`, `rCDR`, `area_CDR`, el indicador `ISNT-like` y el `score combinado`. También calcula umbrales alternativos orientados a sensibilidad, algo especialmente importante si el sistema se plantea como ayuda al cribado glaucomatoso. No reentrena modelos ni modifica las predicciones.

In [ ]:
# ============================================================
# 11. CALIBRACIÓN CLÍNICA Y ESTUDIO DE ABLACIÓN
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

CALIBRATION_DIR = os.path.join(CFG.SAVE_PATH, "calibration")
os.makedirs(CALIBRATION_DIR, exist_ok=True)

ABLATION_RESULTS_PATH = os.path.join(CALIBRATION_DIR, "ablation_results.csv")
THRESHOLD_RESULTS_PATH = os.path.join(CALIBRATION_DIR, "threshold_analysis.csv")
ERROR_CASES_PATH = os.path.join(CALIBRATION_DIR, "external_error_cases.csv")
SELECTED_OPERATING_POINTS_PATH = os.path.join(CALIBRATION_DIR, "selected_operating_points.csv")

log_info(f"Directorio de calibración: {CALIBRATION_DIR}")


# ------------------------------------------------------------
# 11.1. Carga segura de resultados externos
# ------------------------------------------------------------

if "external_results_df" not in globals():
    if os.path.exists(EXTERNAL_RESULTS_PATH):
        external_results_df = pd.read_csv(EXTERNAL_RESULTS_PATH)
        log_ok(f"Resultados externos cargados desde: {EXTERNAL_RESULTS_PATH}")
    else:
        raise FileNotFoundError(
            "No existe external_results_df en memoria ni se encuentra el CSV externo. "
            "Ejecuta primero el punto 10."
        )
else:
    log_ok("external_results_df encontrado en memoria.")

results_df = external_results_df.copy()

results_df["true_glaucoma_label"] = pd.to_numeric(
    results_df["true_glaucoma_label"],
    errors="coerce"
)

results_df = results_df.dropna(subset=["true_glaucoma_label"]).copy()
results_df["true_glaucoma_label"] = results_df["true_glaucoma_label"].astype(int)

log_ok(f"Filas válidas con etiqueta clínica: {len(results_df)}")
log_info(f"Distribución etiquetas: {results_df['true_glaucoma_label'].value_counts().to_dict()}")


# ------------------------------------------------------------
# 11.2. Limpieza y normalización de variables
# ------------------------------------------------------------

candidate_numeric_cols = [
    "pred_vCDR",
    "pred_hCDR",
    "pred_area_CDR",
    "pred_rCDR",
    "pred_rim_to_disc_ratio",
    "pred_cup_to_rim_ratio",
    "pred_isnt_like_risk",
    "pred_vertical_to_horizontal_rim_ratio",
    "risk_score_vcdr_only",
    "risk_score_combined",
    "true_vCDR",
    "true_area_CDR",
    "true_rCDR",
    "seg_disc_iou",
    "seg_cup_iou",
    "abs_error_vCDR",
]

for col in candidate_numeric_cols:
    if col in results_df.columns:
        results_df[col] = pd.to_numeric(results_df[col], errors="coerce")


# ------------------------------------------------------------
# 11.3. Scores alternativos para ablación
# ------------------------------------------------------------

def minmax_clip(series, low, high):
    """
    Escala una variable a 0-1 con límites clínicos/heurísticos.
    """
    values = pd.to_numeric(series, errors="coerce")
    scaled = (values - low) / (high - low)
    return np.clip(scaled, 0.0, 1.0)


# Scores principales ya existentes
if "risk_score_vcdr_only" not in results_df.columns:
    results_df["risk_score_vcdr_only"] = minmax_clip(results_df["pred_vCDR"], 0.45, 0.85)

if "risk_score_combined" not in results_df.columns:
    vcdr_score = minmax_clip(results_df["pred_vCDR"], 0.45, 0.85)
    rcdr_score = minmax_clip(results_df["pred_rCDR"], 0.50, 0.85)

    if "pred_isnt_like_risk" in results_df.columns:
        isnt_score = results_df["pred_isnt_like_risk"].fillna(0.5)
    else:
        isnt_score = 0.5

    results_df["risk_score_combined"] = (
        0.70 * vcdr_score
        + 0.20 * rcdr_score
        + 0.10 * isnt_score
    )


# Scores adicionales para comparar
results_df["score_pred_vCDR_raw"] = results_df["pred_vCDR"]
results_df["score_pred_rCDR_raw"] = results_df["pred_rCDR"]
results_df["score_pred_area_CDR_raw"] = results_df["pred_area_CDR"]

if "pred_isnt_like_risk" in results_df.columns:
    results_df["score_isnt_like"] = results_df["pred_isnt_like_risk"]
else:
    results_df["score_isnt_like"] = np.nan

# Variante más sensible: da todavía más peso al vCDR.
results_df["risk_score_vcdr_dominant"] = (
    0.85 * minmax_clip(results_df["pred_vCDR"], 0.45, 0.85)
    + 0.10 * minmax_clip(results_df["pred_rCDR"], 0.50, 0.85)
    + 0.05 * results_df["score_isnt_like"].fillna(0.5)
)

# Variante más conservadora: exige más apoyo de área/rCDR.
results_df["risk_score_structural_balanced"] = (
    0.50 * minmax_clip(results_df["pred_vCDR"], 0.45, 0.85)
    + 0.30 * minmax_clip(results_df["pred_rCDR"], 0.50, 0.85)
    + 0.15 * minmax_clip(results_df["pred_area_CDR"], 0.20, 0.55)
    + 0.05 * results_df["score_isnt_like"].fillna(0.5)
)

# Scores derivados de máscaras GT: sirven como techo aproximado del biomarcador,
# no como sistema real, porque usan la máscara verdadera.
results_df["score_true_vCDR_raw"] = results_df["true_vCDR"]
results_df["score_true_rCDR_raw"] = results_df["true_rCDR"]
results_df["score_true_area_CDR_raw"] = results_df["true_area_CDR"]


# ------------------------------------------------------------
# 11.4. Funciones de métricas y umbrales
# ------------------------------------------------------------

def metrics_at_threshold(y_true, y_score, threshold):
    """
    Calcula métricas de clasificación para un umbral concreto.
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    y_pred = (y_score >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    return {
        "threshold": float(threshold),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "accuracy": float(accuracy),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def get_valid_score_data(df, score_col):
    """
    Extrae y_true/y_score válidos para una columna de score.
    """
    if score_col not in df.columns:
        return None, None

    temp = df[["true_glaucoma_label", score_col]].copy()
    temp[score_col] = pd.to_numeric(temp[score_col], errors="coerce")
    temp = temp.dropna()

    if len(temp) == 0:
        return None, None

    y_true = temp["true_glaucoma_label"].astype(int).values
    y_score = temp[score_col].astype(float).values

    if len(np.unique(y_true)) < 2:
        return None, None

    if len(np.unique(y_score)) < 2:
        return None, None

    return y_true, y_score


def find_youden_threshold(y_true, y_score):
    """
    Selecciona umbral por índice de Youden: sensibilidad + especificidad - 1.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1.0 - fpr
    youden = tpr + specificity - 1.0

    best_idx = int(np.argmax(youden))

    return float(thresholds[best_idx])


def find_threshold_for_target_sensitivity(y_true, y_score, target_sensitivity):
    """
    Busca el umbral que consigue al menos una sensibilidad objetivo
    maximizando la especificidad.
    """
    thresholds = np.unique(y_score)
    thresholds = np.r_[
        thresholds.min() - 1e-8,
        thresholds,
        thresholds.max() + 1e-8
    ]

    candidates = []

    for threshold in thresholds:
        m = metrics_at_threshold(y_true, y_score, threshold)

        if m["sensitivity"] >= target_sensitivity:
            candidates.append(m)

    if len(candidates) == 0:
        return None

    candidates = sorted(
        candidates,
        key=lambda x: (x["specificity"], x["threshold"]),
        reverse=True
    )

    return candidates[0]


def find_threshold_for_target_specificity(y_true, y_score, target_specificity):
    """
    Busca el umbral que consigue al menos una especificidad objetivo
    maximizando la sensibilidad.
    """
    thresholds = np.unique(y_score)
    thresholds = np.r_[
        thresholds.min() - 1e-8,
        thresholds,
        thresholds.max() + 1e-8
    ]

    candidates = []

    for threshold in thresholds:
        m = metrics_at_threshold(y_true, y_score, threshold)

        if m["specificity"] >= target_specificity:
            candidates.append(m)

    if len(candidates) == 0:
        return None

    candidates = sorted(
        candidates,
        key=lambda x: (x["sensitivity"], -x["threshold"]),
        reverse=True
    )

    return candidates[0]


# ------------------------------------------------------------
# 11.5. Estudio de ablación
# ------------------------------------------------------------

score_columns = [
    "score_pred_vCDR_raw",
    "risk_score_vcdr_only",
    "score_pred_rCDR_raw",
    "score_pred_area_CDR_raw",
    "score_isnt_like",
    "risk_score_combined",
    "risk_score_vcdr_dominant",
    "risk_score_structural_balanced",
    "score_true_vCDR_raw",
    "score_true_rCDR_raw",
    "score_true_area_CDR_raw",
]

ablation_records = []
threshold_records = []

for score_col in score_columns:
    y_true, y_score = get_valid_score_data(results_df, score_col)

    if y_true is None:
        log_warning(f"No se puede evaluar {score_col}.")
        continue

    auc = roc_auc_score(y_true, y_score)

    youden_threshold = find_youden_threshold(y_true, y_score)
    youden_metrics = metrics_at_threshold(y_true, y_score, youden_threshold)

    record = {
        "score_col": score_col,
        "auc": float(auc),
        "youden_threshold": youden_threshold,
        "n_samples": int(len(y_true)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
    }

    record.update(
        {
            f"youden_{key}": value
            for key, value in youden_metrics.items()
        }
    )

    ablation_records.append(record)

    # Guardar estrategia Youden
    threshold_records.append(
        {
            "score_col": score_col,
            "strategy": "youden",
            **youden_metrics,
            "auc": float(auc),
        }
    )

    # Estrategias orientadas a cribado: sensibilidad alta
    for target_sensitivity in [0.85, 0.90, 0.95]:
        threshold_metrics = find_threshold_for_target_sensitivity(
            y_true,
            y_score,
            target_sensitivity=target_sensitivity
        )

        if threshold_metrics is not None:
            threshold_records.append(
                {
                    "score_col": score_col,
                    "strategy": f"sensitivity_at_least_{target_sensitivity:.2f}",
                    **threshold_metrics,
                    "auc": float(auc),
                }
            )

    # Estrategias orientadas a especificidad alta
    for target_specificity in [0.85, 0.90, 0.95]:
        threshold_metrics = find_threshold_for_target_specificity(
            y_true,
            y_score,
            target_specificity=target_specificity
        )

        if threshold_metrics is not None:
            threshold_records.append(
                {
                    "score_col": score_col,
                    "strategy": f"specificity_at_least_{target_specificity:.2f}",
                    **threshold_metrics,
                    "auc": float(auc),
                }
            )


ablation_df = pd.DataFrame(ablation_records)
threshold_df = pd.DataFrame(threshold_records)

ablation_df = ablation_df.sort_values(
    by="auc",
    ascending=False
).reset_index(drop=True)

ablation_df.to_csv(ABLATION_RESULTS_PATH, index=False)
threshold_df.to_csv(THRESHOLD_RESULTS_PATH, index=False)

log_ok(f"Resultados de ablación guardados en: {ABLATION_RESULTS_PATH}")
log_ok(f"Análisis de umbrales guardado en: {THRESHOLD_RESULTS_PATH}")

display(ablation_df)
display(threshold_df)


# ------------------------------------------------------------
# 11.6. Visualización de AUC por biomarcador
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

plot_df = ablation_df.copy()
plot_df = plot_df[~plot_df["score_col"].str.startswith("score_true")]

plt.barh(
    plot_df["score_col"],
    plot_df["auc"]
)

plt.xlabel("AUC")
plt.title("Comparación de biomarcadores y scores en REFUGE-Validation400")
plt.xlim(0.0, 1.0)
plt.grid(axis="x", alpha=0.3)
plt.gca().invert_yaxis()
plt.show()


# ------------------------------------------------------------
# 11.7. Curvas ROC comparativas
# ------------------------------------------------------------

def plot_selected_rocs(df, selected_scores):
    """
    Grafica curvas ROC de scores seleccionados.
    """
    plt.figure(figsize=(7, 6))

    for score_col in selected_scores:
        y_true, y_score = get_valid_score_data(df, score_col)

        if y_true is None:
            continue

        auc = roc_auc_score(y_true, y_score)
        fpr, tpr, thresholds = roc_curve(y_true, y_score)

        plt.plot(
            fpr,
            tpr,
            label=f"{score_col} (AUC={auc:.3f})"
        )

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

    plt.xlabel("1 - Especificidad")
    plt.ylabel("Sensibilidad")
    plt.title("ROC comparativa de biomarcadores")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


plot_selected_rocs(
    results_df,
    selected_scores=[
        "risk_score_combined",
        "risk_score_vcdr_only",
        "risk_score_vcdr_dominant",
        "risk_score_structural_balanced",
        "score_true_vCDR_raw",
    ]
)


# ------------------------------------------------------------
# 11.8. Sesgo y error de biomarcadores
# ------------------------------------------------------------

biomarker_error_records = []

for pred_col, true_col in [
    ("pred_vCDR", "true_vCDR"),
    ("pred_rCDR", "true_rCDR"),
    ("pred_area_CDR", "true_area_CDR"),
]:
    if pred_col in results_df.columns and true_col in results_df.columns:
        pred_values = pd.to_numeric(results_df[pred_col], errors="coerce")
        true_values = pd.to_numeric(results_df[true_col], errors="coerce")

        valid = ~(pred_values.isna() | true_values.isna())

        diff = pred_values[valid] - true_values[valid]
        abs_diff = diff.abs()

        biomarker_error_records.append(
            {
                "biomarker": pred_col.replace("pred_", ""),
                "mean_pred": float(pred_values[valid].mean()),
                "mean_true": float(true_values[valid].mean()),
                "mean_bias_pred_minus_true": float(diff.mean()),
                "mae": float(abs_diff.mean()),
                "median_abs_error": float(abs_diff.median()),
                "std_error": float(diff.std()),
                "max_abs_error": float(abs_diff.max()),
            }
        )

biomarker_error_df = pd.DataFrame(biomarker_error_records)

display(biomarker_error_df)

biomarker_error_df.to_csv(
    os.path.join(CALIBRATION_DIR, "biomarker_error_summary.csv"),
    index=False
)


# ------------------------------------------------------------
# 11.9. Análisis de errores: falsos positivos, falsos negativos y vCDR extremo
# ------------------------------------------------------------

def add_predictions_for_operating_point(df, score_col, threshold):
    """
    Añade predicción binaria para un score y umbral concretos.
    """
    out = df.copy()

    out[f"pred_label_{score_col}"] = (
        pd.to_numeric(out[score_col], errors="coerce") >= threshold
    ).astype(int)

    return out


# Usamos por defecto el score combinado con umbral Youden.
combined_row = ablation_df[ablation_df["score_col"] == "risk_score_combined"]

if len(combined_row) > 0:
    selected_threshold = float(combined_row.iloc[0]["youden_threshold"])
else:
    selected_threshold = CFG.CLINICAL_THRESHOLD

error_df = add_predictions_for_operating_point(
    results_df,
    score_col="risk_score_combined",
    threshold=selected_threshold
)

error_df["error_type"] = "correct"

error_df.loc[
    (error_df["true_glaucoma_label"] == 0)
    & (error_df["pred_label_risk_score_combined"] == 1),
    "error_type"
] = "false_positive"

error_df.loc[
    (error_df["true_glaucoma_label"] == 1)
    & (error_df["pred_label_risk_score_combined"] == 0),
    "error_type"
] = "false_negative"

error_df["abs_error_vCDR"] = pd.to_numeric(
    error_df["abs_error_vCDR"],
    errors="coerce"
)

# Marcar casos con errores fuertes de biomarcador.
error_df["large_vCDR_error"] = error_df["abs_error_vCDR"] >= 0.15

error_cases_df = error_df[
    (error_df["error_type"] != "correct")
    | (error_df["large_vCDR_error"])
].copy()

error_cases_df = error_cases_df.sort_values(
    by=["error_type", "abs_error_vCDR"],
    ascending=[True, False]
)

error_cases_df.to_csv(ERROR_CASES_PATH, index=False)

log_ok(f"Casos de error guardados en: {ERROR_CASES_PATH}")

display(
    error_cases_df[
        [
            "index",
            "image_id",
            "true_glaucoma_label",
            "pred_label_risk_score_combined",
            "error_type",
            "risk_score_combined",
            "pred_vCDR",
            "true_vCDR",
            "abs_error_vCDR",
            "seg_disc_iou",
            "seg_cup_iou",
            "quality_mean_confidence",
            "quality_mean_entropy",
        ]
    ].head(30)
)


# ------------------------------------------------------------
# 11.10. Selección preliminar de puntos operativos
# ------------------------------------------------------------

selected_strategies = [
    "youden",
    "sensitivity_at_least_0.85",
    "sensitivity_at_least_0.90",
    "specificity_at_least_0.85",
    "specificity_at_least_0.90",
]

selected_scores = [
    "risk_score_combined",
    "risk_score_vcdr_only",
    "risk_score_vcdr_dominant",
    "risk_score_structural_balanced",
]

selected_operating_points_df = threshold_df[
    threshold_df["score_col"].isin(selected_scores)
    & threshold_df["strategy"].isin(selected_strategies)
].copy()

selected_operating_points_df = selected_operating_points_df.sort_values(
    by=["score_col", "strategy"]
)

selected_operating_points_df.to_csv(
    SELECTED_OPERATING_POINTS_PATH,
    index=False
)

display(selected_operating_points_df)

log_ok(f"Puntos operativos seleccionados guardados en: {SELECTED_OPERATING_POINTS_PATH}")


# ------------------------------------------------------------
# 11.11. Conclusión automática del bloque
# ------------------------------------------------------------

best_auc_row = ablation_df.iloc[0]

log_ok("Calibración clínica y estudio de ablación completados.")

log_info(
    f"Mejor AUC observado: {best_auc_row['score_col']} "
    f"con AUC={best_auc_row['auc']:.4f}"
)

combined_auc = ablation_df.loc[
    ablation_df["score_col"] == "risk_score_combined",
    "auc"
]

vcdr_auc = ablation_df.loc[
    ablation_df["score_col"] == "risk_score_vcdr_only",
    "auc"
]

if len(combined_auc) > 0 and len(vcdr_auc) > 0:
    log_info(
        f"AUC combinado={float(combined_auc.iloc[0]):.4f} | "
        f"AUC vCDR solo={float(vcdr_auc.iloc[0]):.4f}"
    )

log_info(
    "Revisa selected_operating_points_df antes de pasar al test final. "
    "Para cribado, probablemente convenga priorizar sensibilidad sobre Youden."
)

# 12. Evaluación final en REFUGE-Test400

Este bloque evalúa el sistema final sobre `REFUGE-Test400`, usando el score seleccionado y el umbral calibrado previamente en `REFUGE-Validation400`. No recalibra el umbral, no reentrena modelos y no modifica el ensemble. Su objetivo es estimar el rendimiento final en un conjunto separado, calculando métricas de segmentación, error de `vCDR`, `AUC`, `sensibilidad`, `especificidad`, `precisión`, `F1` y `matriz de confusión`.

In [ ]:
# ============================================================
# 12. EVALUACIÓN FINAL EN REFUGE-TEST400
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ------------------------------------------------------------
# 12.1. Configuración del test final
# ------------------------------------------------------------

TEST_RESULTS_PATH = os.path.join(
    CFG.SAVE_PATH,
    "test_validation_results.csv"
)

TEST_SUMMARY_PATH = os.path.join(
    CFG.SAVE_PATH,
    "test_validation_summary.csv"
)

TEST_PREDICTION_MEMMAP_PATH = "/content/test_ensemble_predictions.dat"
TEST_IMAGES_MEMMAP_PATH = "/content/test_images_preprocessed.dat"
TEST_TRUE_MASKS_MEMMAP_PATH = "/content/test_true_masks.dat"

TEST_BATCH_SIZE = INFERENCE_BATCH_SIZE

# Score final seleccionado a partir de validación.
FINAL_SCORE_COL = "risk_score_combined"

# Estrategia recomendada:
# - Punto operativo PRINCIPAL (cribado): "sensitivity_at_least_0.85"
# - Alternativa equilibrada: "youden"
# Se descarta "sensitivity_at_least_0.90": solo gana 1 FN a costa de ~32 FP adicionales.
FINAL_THRESHOLD_STRATEGY = "sensitivity_at_least_0.85"

log_info("Configuración de evaluación final:")
log_info(f"Score final: {FINAL_SCORE_COL}")
log_info(f"Estrategia de umbral: {FINAL_THRESHOLD_STRATEGY}")
log_info(f"Batch size test: {TEST_BATCH_SIZE}")


# ------------------------------------------------------------
# 12.2. Selección del umbral desde validación
# ------------------------------------------------------------

def load_selected_operating_points():
    """
    Carga los puntos operativos calibrados en REFUGE-Validation400.
    """

    if "selected_operating_points_df" in globals():
        return selected_operating_points_df.copy()

    if os.path.exists(SELECTED_OPERATING_POINTS_PATH):
        df = pd.read_csv(SELECTED_OPERATING_POINTS_PATH)
        log_ok(f"Puntos operativos cargados desde: {SELECTED_OPERATING_POINTS_PATH}")
        return df

    if os.path.exists(THRESHOLD_RESULTS_PATH):
        df = pd.read_csv(THRESHOLD_RESULTS_PATH)
        log_ok(f"Análisis de umbrales cargado desde: {THRESHOLD_RESULTS_PATH}")
        return df

    raise FileNotFoundError(
        "No se encuentra la tabla de umbrales calibrados. "
        "Ejecuta primero el punto 11."
    )


def select_final_threshold(score_col=FINAL_SCORE_COL, strategy=FINAL_THRESHOLD_STRATEGY):
    """
    Selecciona el umbral final usando únicamente la calibración de Validation400.

    No se calibra nada con Test400.
    """

    operating_df = load_selected_operating_points()

    candidates = operating_df[
        (operating_df["score_col"] == score_col)
        & (operating_df["strategy"] == strategy)
    ].copy()

    if len(candidates) == 0:
        log_warning(
            f"No se ha encontrado estrategia {strategy} para {score_col}. "
            "Se usará Youden como fallback."
        )

        candidates = operating_df[
            (operating_df["score_col"] == score_col)
            & (operating_df["strategy"] == "youden")
        ].copy()

    if len(candidates) == 0:
        raise ValueError(
            f"No se ha podido seleccionar umbral para {score_col}."
        )

    row = candidates.iloc[0]

    threshold = float(row["threshold"])

    log_ok(
        f"Umbral final seleccionado desde validación: {threshold:.6f} "
        f"({score_col}, {row['strategy']})"
    )

    log_info(
        f"Rendimiento esperado en Validation400 con este umbral: "
        f"Sens={row['sensitivity']:.3f}, "
        f"Spec={row['specificity']:.3f}, "
        f"F1={row['f1']:.3f}"
    )

    return threshold, row


FINAL_THRESHOLD, FINAL_OPERATING_POINT = select_final_threshold(
    score_col=FINAL_SCORE_COL,
    strategy=FINAL_THRESHOLD_STRATEGY
)


# ------------------------------------------------------------
# 12.3. Localización e indexación de Test400
# ------------------------------------------------------------

def get_test_dataset_paths():
    """
    Devuelve rutas robustas para Test400 y su ground truth.
    """

    candidate_image_dirs = [
        os.path.join(CFG.BASE_PATH, "Test400"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Test400", "Test400"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Test400"),
    ]

    candidate_mask_dirs = [
        os.path.join(CFG.BASE_PATH, "REFUGE-Test-GT", "Disc_Cup_Masks"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Test-GT", "REFUGE-Test-GT", "Disc_Cup_Masks"),
    ]

    candidate_label_files = [
        os.path.join(CFG.BASE_PATH, "REFUGE-Test-GT", "Glaucoma_label_and_Fovea_location.xlsx"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Test-GT", "REFUGE-Test-GT", "Glaucoma_label_and_Fovea_location.xlsx"),
    ]

    test_images_dir = None
    test_masks_dir = None
    test_labels_file = None

    for path in candidate_image_dirs:
        if os.path.exists(path):
            n_images = count_files_by_extensions(
                path,
                [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
            )

            if n_images > 0:
                test_images_dir = path
                break

    for path in candidate_mask_dirs:
        if os.path.exists(path):
            n_masks = count_files_by_extensions(
                path,
                [".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"]
            )

            if n_masks > 0:
                test_masks_dir = path
                break

    for path in candidate_label_files:
        if os.path.exists(path):
            test_labels_file = path
            break

    if test_images_dir is None:
        raise FileNotFoundError("No se ha encontrado la carpeta de imágenes Test400.")

    if test_masks_dir is None:
        raise FileNotFoundError("No se ha encontrado la carpeta de máscaras de Test400.")

    if test_labels_file is None:
        raise FileNotFoundError("No se ha encontrado el archivo de etiquetas de Test400.")

    log_ok(f"Imágenes Test400: {test_images_dir}")
    log_ok(f"Máscaras Test400: {test_masks_dir}")
    log_ok(f"Etiquetas Test400: {test_labels_file}")

    return {
        "test_images": test_images_dir,
        "test_masks": test_masks_dir,
        "test_labels": test_labels_file,
    }


TEST_PATHS = get_test_dataset_paths()

TEST_DIRS = [
    (
        TEST_PATHS["test_images"],
        TEST_PATHS["test_masks"],
    )
]

test_data = get_all_pairs_robust(TEST_DIRS)

log_ok(f"Test400 indexado: {len(test_data)} pares imagen-máscara.")

if len(test_data) == 0:
    raise ValueError("No se han encontrado pares imagen-máscara en Test400.")


# ------------------------------------------------------------
# 12.4. Carga de etiquetas clínicas de test
# ------------------------------------------------------------

def load_test_glaucoma_labels(label_file):
    """
    Carga etiquetas clínicas de Test400.

    En REFUGE-Test-GT suele aparecer como:
        Label(Glaucoma=1)
    """

    df = pd.read_excel(label_file)

    log_info(f"Columnas disponibles en etiquetas test: {list(df.columns)}")

    possible_name_cols = [
        col for col in df.columns
        if (
            "img" in str(col).lower()
            or "image" in str(col).lower()
            or "name" in str(col).lower()
            or "filename" in str(col).lower()
            or "file" in str(col).lower()
        )
    ]

    possible_label_cols = [
        col for col in df.columns
        if (
            "glaucoma" in str(col).lower()
            or "label" in str(col).lower()
            or "diagnosis" in str(col).lower()
        )
    ]

    if len(possible_name_cols) == 0:
        raise ValueError("No se ha encontrado columna de nombre de imagen en etiquetas test.")

    if len(possible_label_cols) == 0:
        raise ValueError("No se ha encontrado columna de etiqueta clínica en etiquetas test.")

    name_col = possible_name_cols[0]

    glaucoma_candidates = [
        col for col in possible_label_cols
        if "glaucoma" in str(col).lower()
    ]

    if len(glaucoma_candidates) > 0:
        label_col = glaucoma_candidates[0]
    else:
        label_col = possible_label_cols[0]

    log_ok(f"Columna imagen test: {name_col}")
    log_ok(f"Columna etiqueta test: {label_col}")

    label_dict = {}

    for _, row in df.iterrows():
        image_key = normalize_label_key(row[name_col])

        try:
            label = int(row[label_col])
        except Exception:
            continue

        label_dict[image_key] = label

    log_ok(f"Etiquetas clínicas test cargadas: {len(label_dict)}")

    return label_dict, df


test_label_dict, test_label_df = load_test_glaucoma_labels(
    TEST_PATHS["test_labels"]
)


def get_test_label_for_image_path(image_path):
    """
    Recupera etiqueta clínica de test para una imagen.
    """

    stem_key = normalize_label_key(image_path)

    if stem_key in test_label_dict:
        return test_label_dict[stem_key]

    num_key = numeric_key(image_path)

    if num_key is not None:
        for key, value in test_label_dict.items():
            if numeric_key(key) == num_key:
                return value

    return np.nan


# ------------------------------------------------------------
# 12.5. Preprocesamiento de Test400 a memmap
# ------------------------------------------------------------

def preprocess_test_to_memmap(force_rebuild=False):
    """
    Preprocesa Test400 una vez y guarda:
        - imágenes preprocesadas;
        - máscaras reales 0/1/2.
    """

    n_samples = len(test_data)

    if force_rebuild:
        for path in [
            TEST_IMAGES_MEMMAP_PATH,
            TEST_TRUE_MASKS_MEMMAP_PATH,
        ]:
            if os.path.exists(path):
                os.remove(path)

    images_ready = os.path.exists(TEST_IMAGES_MEMMAP_PATH)
    masks_ready = os.path.exists(TEST_TRUE_MASKS_MEMMAP_PATH)

    images_memmap = np.memmap(
        TEST_IMAGES_MEMMAP_PATH,
        dtype="float32",
        mode="w+" if not images_ready else "r+",
        shape=(n_samples, CFG.IMG_SIZE, CFG.IMG_SIZE, 3)
    )

    masks_memmap = np.memmap(
        TEST_TRUE_MASKS_MEMMAP_PATH,
        dtype="uint8",
        mode="w+" if not masks_ready else "r+",
        shape=(n_samples, CFG.IMG_SIZE, CFG.IMG_SIZE)
    )

    if images_ready and masks_ready and not force_rebuild:
        log_info("Preprocesamiento de test ya existe en memmap. Se reutiliza.")
        return images_memmap, masks_memmap

    log_info("Preprocesando Test400.")

    for idx, (image_path, mask_path) in enumerate(test_data):
        if idx % 50 == 0:
            log_info(f"Preprocesando test {idx}/{n_samples}")

        image_preprocessed, mask_onehot = preprocess_image_and_mask(
            image_path,
            mask_path
        )

        mask_class = np.argmax(
            mask_onehot,
            axis=-1
        ).astype(np.uint8)

        images_memmap[idx] = image_preprocessed.astype(np.float32)
        masks_memmap[idx] = mask_class

    images_memmap.flush()
    masks_memmap.flush()

    log_ok("Preprocesamiento de test guardado en memmap.")

    return images_memmap, masks_memmap


# force_rebuild=True: regenerar el memmap de test tras cambiar locate_roi_center
# (evita reutilizar recortes de la ROI antigua en la misma sesion).
test_images_memmap, test_true_masks_memmap = preprocess_test_to_memmap(
    force_rebuild=True
)


# ------------------------------------------------------------
# 12.6. Predicción ensemble sobre Test400
# ------------------------------------------------------------

def iter_test_batches(images_memmap, batch_size):
    """
    Iterador de batches sobre imágenes test preprocesadas.
    """

    n_samples = images_memmap.shape[0]

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)

        batch = np.array(
            images_memmap[start:end],
            dtype=np.float32
        )

        yield start, end, batch


def predict_test_ensemble_to_memmap():
    """
    Predice Test400 con el ensemble.

    No calibra nada. Solo aplica los modelos entrenados.
    """

    n_samples = len(test_data)

    if os.path.exists(TEST_PREDICTION_MEMMAP_PATH):
        os.remove(TEST_PREDICTION_MEMMAP_PATH)

    pred_sum = np.memmap(
        TEST_PREDICTION_MEMMAP_PATH,
        dtype="float32",
        mode="w+",
        shape=(n_samples, CFG.IMG_SIZE, CFG.IMG_SIZE, CFG.CLASSES)
    )

    pred_sum[:] = 0.0
    pred_sum.flush()

    for fold_number in range(1, CFG.N_SPLITS + 1):
        print("\n" + "-" * 80)
        log_info(f"Predicción test con fold {fold_number}/{CFG.N_SPLITS}")
        print("-" * 80)

        model = load_single_model(
            fold_number,
            compile_model=False
        )

        for start, end, batch_images in iter_test_batches(
            test_images_memmap,
            TEST_BATCH_SIZE
        ):
            pred = predict_model_with_tta(
                model=model,
                images_np=batch_images,
                use_tta=USE_TTA_INFERENCE
            )

            pred_sum[start:end] += pred.astype(np.float32)

        pred_sum.flush()

        del model
        tf.keras.backend.clear_session()
        gc.collect()

        log_ok(f"Fold {fold_number} acumulado correctamente en test.")

    pred_sum[:] = pred_sum[:] / CFG.N_SPLITS
    pred_sum.flush()

    log_ok("Predicción del ensemble sobre Test400 completada.")

    return pred_sum


test_pred_memmap = predict_test_ensemble_to_memmap()


# ------------------------------------------------------------
# 12.7. Construcción de resultados de Test400
# ------------------------------------------------------------

def build_test_results(pred_memmap, true_masks_memmap):
    """
    Construye tabla final de resultados sobre Test400.
    """

    records = []

    n_samples = len(test_data)

    for idx in range(n_samples):
        if idx % 50 == 0:
            log_info(f"Construyendo resultados test {idx}/{n_samples}")

        image_path, mask_path = test_data[idx]

        probability_map = np.array(
            pred_memmap[idx],
            dtype=np.float32
        )

        raw_mask = np.argmax(
            probability_map,
            axis=-1
        ).astype(np.uint8)

        final_mask, postprocess_diagnostics = postprocess_segmentation_mask(
            raw_mask
        )

        true_mask = np.array(
            true_masks_memmap[idx],
            dtype=np.uint8
        )

        seg_metrics = compute_segmentation_metrics_np(
            true_mask=true_mask,
            pred_mask=final_mask
        )

        pred_biomarkers = compute_biomarkers_from_mask(
            final_mask
        )

        raw_biomarkers = compute_biomarkers_from_mask(
            raw_mask
        )

        true_biomarkers = compute_biomarkers_from_mask(
            true_mask
        )

        confidence_maps = compute_confidence_maps(
            probability_map
        )

        quality_summary = summarize_prediction_quality(
            probability_map=probability_map,
            postprocess_diagnostics=postprocess_diagnostics
        )

        true_label = get_test_label_for_image_path(
            image_path
        )

        record = {
            "index": idx,
            "image_path": image_path,
            "mask_path": mask_path,
            "image_id": normalize_label_key(image_path),
            "true_glaucoma_label": true_label,
        }

        record.update(seg_metrics)

        for key, value in pred_biomarkers.items():
            record[f"pred_{key}"] = value

        for key, value in raw_biomarkers.items():
            record[f"raw_{key}"] = value

        for key, value in true_biomarkers.items():
            record[f"true_{key}"] = value

        if (
            not np.isnan(record.get("pred_vCDR", np.nan))
            and not np.isnan(record.get("true_vCDR", np.nan))
        ):
            record["abs_error_vCDR"] = abs(
                record["pred_vCDR"]
                - record["true_vCDR"]
            )
        else:
            record["abs_error_vCDR"] = np.nan

        vcdr_score = np.clip(
            (record.get("pred_vCDR", np.nan) - 0.45) / 0.40,
            0.0,
            1.0
        )

        rcdr_score = np.clip(
            (record.get("pred_rCDR", np.nan) - 0.50) / 0.35,
            0.0,
            1.0
        )

        isnt_risk = record.get("pred_isnt_like_risk", np.nan)

        if isinstance(isnt_risk, str):
            try:
                isnt_risk = float(isnt_risk)
            except Exception:
                isnt_risk = np.nan

        if np.isnan(vcdr_score):
            vcdr_score = 0.0

        if np.isnan(rcdr_score):
            rcdr_score = 0.0

        if np.isnan(isnt_risk):
            isnt_risk = 0.5

        record["risk_score_vcdr_only"] = float(vcdr_score)

        record["risk_score_combined"] = float(
            0.70 * vcdr_score
            + 0.20 * rcdr_score
            + 0.10 * isnt_risk
        )

        record["final_score_col"] = FINAL_SCORE_COL
        record["final_threshold"] = FINAL_THRESHOLD

        record["final_score"] = record[FINAL_SCORE_COL]
        record["final_pred_label"] = int(
            record["final_score"] >= FINAL_THRESHOLD
        )

        for key, value in postprocess_diagnostics.items():
            record[f"post_{key}"] = value

        for key, value in quality_summary.items():
            record[f"quality_{key}"] = value

        records.append(record)

    results_df = pd.DataFrame(records)

    return results_df


test_results_df = build_test_results(
    pred_memmap=test_pred_memmap,
    true_masks_memmap=test_true_masks_memmap
)

test_results_df.to_csv(
    TEST_RESULTS_PATH,
    index=False
)

log_ok(f"Resultados de Test400 guardados en: {TEST_RESULTS_PATH}")

display(test_results_df.head())


# ------------------------------------------------------------
# 12.8. Resumen de segmentación y biomarcadores en test
# ------------------------------------------------------------

def summarize_test_segmentation(results_df):
    """
    Resume métricas de segmentación y biomarcadores en test.
    """

    metric_cols = [
        "seg_disc_iou",
        "seg_disc_dice",
        "seg_cup_iou",
        "seg_cup_dice",
        "seg_clinical_global_iou",
        "seg_clinical_global_dice",
        "abs_error_vCDR",
        "pred_vCDR",
        "true_vCDR",
        "pred_area_CDR",
        "true_area_CDR",
        "pred_rCDR",
        "true_rCDR",
    ]

    rows = []

    for col in metric_cols:
        if col not in results_df.columns:
            continue

        values = pd.to_numeric(
            results_df[col],
            errors="coerce"
        ).dropna()

        rows.append(
            {
                "metric": col,
                "mean": values.mean(),
                "std": values.std(),
                "median": values.median(),
                "min": values.min(),
                "max": values.max(),
            }
        )

    return pd.DataFrame(rows)


test_segmentation_summary_df = summarize_test_segmentation(
    test_results_df
)

display(test_segmentation_summary_df)


# ------------------------------------------------------------
# 12.9. Métricas finales de clasificación en test
# ------------------------------------------------------------

def compute_final_test_classification(results_df):
    """
    Calcula métricas finales usando el umbral fijado en Validation400.
    """

    df = results_df.copy()

    df["true_glaucoma_label"] = pd.to_numeric(
        df["true_glaucoma_label"],
        errors="coerce"
    )

    df["final_score"] = pd.to_numeric(
        df["final_score"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["true_glaucoma_label", "final_score"]
    )

    y_true = df["true_glaucoma_label"].astype(int).values
    y_score = df["final_score"].astype(float).values
    y_pred = (y_score >= FINAL_THRESHOLD).astype(int)

    auc = roc_auc_score(y_true, y_score)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    metrics = {
        "score_col": FINAL_SCORE_COL,
        "threshold_strategy": FINAL_THRESHOLD_STRATEGY,
        "threshold": float(FINAL_THRESHOLD),
        "n_samples": int(len(df)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
        "auc": float(auc),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "accuracy": float(accuracy),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
    }

    return metrics


test_classification_metrics = compute_final_test_classification(
    test_results_df
)


# ------------------------------------------------------------
# 12.10. Visualización ROC y matriz final
# ------------------------------------------------------------

plt.figure(figsize=(7, 6))
plt.plot(
    test_classification_metrics["fpr"],
    test_classification_metrics["tpr"],
    label=f"{FINAL_SCORE_COL} (AUC={test_classification_metrics['auc']:.3f})"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("1 - Especificidad")
plt.ylabel("Sensibilidad")
plt.title("Curva ROC final en REFUGE-Test400")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


cm = np.array(
    [
        [test_classification_metrics["tn"], test_classification_metrics["fp"]],
        [test_classification_metrics["fn"], test_classification_metrics["tp"]],
    ]
)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred sano", "Pred glaucoma"],
    yticklabels=["Real sano", "Real glaucoma"]
)
plt.title(
    f"Matriz de confusión final - {FINAL_SCORE_COL}\n"
    f"Umbral={FINAL_THRESHOLD:.3f}"
)
plt.ylabel("Etiqueta real")
plt.xlabel("Predicción")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 12.11. Tabla resumen final
# ------------------------------------------------------------

summary_records = []

for _, row in test_segmentation_summary_df.iterrows():
    summary_records.append(
        {
            "section": "test_segmentation_biomarkers",
            "metric": row["metric"],
            "value": row["mean"],
            "std": row["std"],
        }
    )

for key in [
    "auc",
    "threshold",
    "sensitivity",
    "specificity",
    "precision",
    "recall",
    "f1",
    "accuracy",
    "n_samples",
    "n_positive",
    "n_negative",
    "tn",
    "fp",
    "fn",
    "tp",
]:
    summary_records.append(
        {
            "section": "test_classification",
            "metric": key,
            "value": test_classification_metrics[key],
            "std": np.nan,
        }
    )

test_summary_df = pd.DataFrame(summary_records)

test_summary_df.to_csv(
    TEST_SUMMARY_PATH,
    index=False
)

log_ok(f"Resumen final de Test400 guardado en: {TEST_SUMMARY_PATH}")

display(test_summary_df)


# ------------------------------------------------------------
# 12.12. Conclusión automática del test final
# ------------------------------------------------------------

log_ok("Evaluación final en REFUGE-Test400 completada.")

log_info(
    f"AUC test: {test_classification_metrics['auc']:.4f} | "
    f"Sensibilidad: {test_classification_metrics['sensitivity']:.4f} | "
    f"Especificidad: {test_classification_metrics['specificity']:.4f} | "
    f"F1: {test_classification_metrics['f1']:.4f}"
)

mean_test_vcdr_mae = test_results_df["abs_error_vCDR"].mean()

log_info(f"MAE test de vCDR: {mean_test_vcdr_mae:.4f}")

# ============================================================
# 12 (cont.). COMPARACIÓN DE PUNTOS OPERATIVOS EN REFUGE-TEST400
# ============================================================

# ------------------------------------------------------------
# 12.13. Carga segura de resultados y umbrales
# ------------------------------------------------------------

if "test_results_df" not in globals():
    if os.path.exists(TEST_RESULTS_PATH):
        test_results_df = pd.read_csv(TEST_RESULTS_PATH)
        log_ok(f"Resultados test cargados desde: {TEST_RESULTS_PATH}")
    else:
        raise FileNotFoundError("No se encuentra test_results_df ni TEST_RESULTS_PATH.")

if "threshold_df" not in globals():
    if os.path.exists(THRESHOLD_RESULTS_PATH):
        threshold_df = pd.read_csv(THRESHOLD_RESULTS_PATH)
        log_ok(f"Umbrales de validación cargados desde: {THRESHOLD_RESULTS_PATH}")
    else:
        raise FileNotFoundError("No se encuentra threshold_df ni THRESHOLD_RESULTS_PATH.")
# ------------------------------------------------------------
# 12.14. Reconstrucción de scores derivados en Test400
# ------------------------------------------------------------

def minmax_clip_local(series, low, high):
    """
    Escala una variable a 0-1 usando límites fijos.
    Es la misma lógica usada en la calibración del punto 11.
    """
    values = pd.to_numeric(series, errors="coerce")
    scaled = (values - low) / (high - low)
    return np.clip(scaled, 0.0, 1.0)


# Asegurar columnas numéricas necesarias
for col in [
    "pred_vCDR",
    "pred_rCDR",
    "pred_area_CDR",
    "pred_isnt_like_risk",
    "risk_score_vcdr_only",
    "risk_score_combined",
]:
    if col in test_results_df.columns:
        test_results_df[col] = pd.to_numeric(
            test_results_df[col],
            errors="coerce"
        )


# Score vCDR solo
if "risk_score_vcdr_only" not in test_results_df.columns:
    test_results_df["risk_score_vcdr_only"] = minmax_clip_local(
        test_results_df["pred_vCDR"],
        0.45,
        0.85
    )


# Score ISNT-like
if "pred_isnt_like_risk" in test_results_df.columns:
    test_results_df["score_isnt_like"] = pd.to_numeric(
        test_results_df["pred_isnt_like_risk"],
        errors="coerce"
    )
else:
    test_results_df["score_isnt_like"] = np.nan


# Score combinado original
if "risk_score_combined" not in test_results_df.columns:
    vcdr_score = minmax_clip_local(test_results_df["pred_vCDR"], 0.45, 0.85)
    rcdr_score = minmax_clip_local(test_results_df["pred_rCDR"], 0.50, 0.85)
    isnt_score = test_results_df["score_isnt_like"].fillna(0.5)

    test_results_df["risk_score_combined"] = (
        0.70 * vcdr_score
        + 0.20 * rcdr_score
        + 0.10 * isnt_score
    )


# Score dominante en vCDR
test_results_df["risk_score_vcdr_dominant"] = (
    0.85 * minmax_clip_local(test_results_df["pred_vCDR"], 0.45, 0.85)
    + 0.10 * minmax_clip_local(test_results_df["pred_rCDR"], 0.50, 0.85)
    + 0.05 * test_results_df["score_isnt_like"].fillna(0.5)
)


# Score estructural balanceado
test_results_df["risk_score_structural_balanced"] = (
    0.50 * minmax_clip_local(test_results_df["pred_vCDR"], 0.45, 0.85)
    + 0.30 * minmax_clip_local(test_results_df["pred_rCDR"], 0.50, 0.85)
    + 0.15 * minmax_clip_local(test_results_df["pred_area_CDR"], 0.20, 0.55)
    + 0.05 * test_results_df["score_isnt_like"].fillna(0.5)
)

log_ok("Scores derivados reconstruidos en test_results_df.")

log_info(
    "Columnas de score disponibles en test: "
    f"{[col for col in test_results_df.columns if 'score' in col or 'risk' in col]}"
)

# ------------------------------------------------------------
# 12.15. Función para evaluar umbrales en test
# ------------------------------------------------------------

def evaluate_score_threshold_on_test(results_df, score_col, threshold, strategy_name):
    """
    Evalúa un score y un umbral fijo sobre Test400.

    Importante:
        El umbral viene de Validation400. No se recalibra con Test400.
    """

    df = results_df.copy()

    df["true_glaucoma_label"] = pd.to_numeric(
        df["true_glaucoma_label"],
        errors="coerce"
    )

    df[score_col] = pd.to_numeric(
        df[score_col],
        errors="coerce"
    )

    df = df.dropna(
        subset=["true_glaucoma_label", score_col]
    )

    y_true = df["true_glaucoma_label"].astype(int).values
    y_score = df[score_col].astype(float).values
    y_pred = (y_score >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_score)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    return {
        "score_col": score_col,
        "strategy": strategy_name,
        "threshold": float(threshold),
        "auc": float(auc),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "accuracy": float(accuracy),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "n_samples": int(len(df)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
    }


# ------------------------------------------------------------
# 12.16. Evaluación de estrategias relevantes
# ------------------------------------------------------------

strategies_to_compare = [
    "youden",
    "sensitivity_at_least_0.85",
    "sensitivity_at_least_0.90",
    "sensitivity_at_least_0.95",
    "specificity_at_least_0.85",
    "specificity_at_least_0.90",
]

scores_to_compare = [
    score
    for score in [
        "risk_score_combined",
        "risk_score_vcdr_only",
        "risk_score_vcdr_dominant",
        "risk_score_structural_balanced",
    ]
    if score in test_results_df.columns
]

log_info(f"Scores que se evaluarán en test: {scores_to_compare}")


test_operating_records = []

for score_col in scores_to_compare:
    for strategy in strategies_to_compare:
        row = threshold_df[
            (threshold_df["score_col"] == score_col)
            & (threshold_df["strategy"] == strategy)
        ]

        if len(row) == 0:
            continue

        threshold = float(row.iloc[0]["threshold"])

        metrics = evaluate_score_threshold_on_test(
            results_df=test_results_df,
            score_col=score_col,
            threshold=threshold,
            strategy_name=strategy
        )

        test_operating_records.append(metrics)

test_operating_points_df = pd.DataFrame(test_operating_records)

test_operating_points_df = test_operating_points_df.sort_values(
    by=["score_col", "strategy"]
).reset_index(drop=True)

TEST_OPERATING_POINTS_PATH = os.path.join(
    CFG.SAVE_PATH,
    "test_operating_points_comparison.csv"
)

test_operating_points_df.to_csv(
    TEST_OPERATING_POINTS_PATH,
    index=False
)

log_ok(f"Comparación de puntos operativos en test guardada en: {TEST_OPERATING_POINTS_PATH}")

display(test_operating_points_df)


# ------------------------------------------------------------
# 12.17. Selección de candidatos recomendables
# ------------------------------------------------------------

recommended_candidates = test_operating_points_df[
    (
        (test_operating_points_df["score_col"] == "risk_score_combined")
        & (
            test_operating_points_df["strategy"].isin(
                [
                    "youden",
                    "sensitivity_at_least_0.85",
                    "sensitivity_at_least_0.90",
                    "specificity_at_least_0.85",
                ]
            )
        )
    )
].copy()

display(recommended_candidates)


# ------------------------------------------------------------
# 12.18. Visualización sensibilidad/especificidad
# ------------------------------------------------------------

plot_df = recommended_candidates.copy()

plt.figure(figsize=(9, 5))

x = np.arange(len(plot_df))

plt.bar(
    x - 0.2,
    plot_df["sensitivity"],
    width=0.4,
    label="Sensibilidad"
)

plt.bar(
    x + 0.2,
    plot_df["specificity"],
    width=0.4,
    label="Especificidad"
)

plt.xticks(
    x,
    plot_df["strategy"],
    rotation=30,
    ha="right"
)

plt.ylim(0, 1)
plt.ylabel("Valor")
plt.title("Comparación de puntos operativos en REFUGE-Test400")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 12.19. Conclusión automática
# ------------------------------------------------------------

log_ok("Comparación de puntos operativos completada.")

best_f1_row = test_operating_points_df.sort_values(
    by="f1",
    ascending=False
).iloc[0]

log_info(
    f"Mejor F1 en test: {best_f1_row['score_col']} | "
    f"{best_f1_row['strategy']} | "
    f"F1={best_f1_row['f1']:.3f}, "
    f"Sens={best_f1_row['sensitivity']:.3f}, "
    f"Spec={best_f1_row['specificity']:.3f}"
)

# 12.b Significancia estadística (intervalos de confianza bootstrap)

Las métricas puntuales de Test400 (AUC, Dice de disco y copa, MAE de vCDR) se acompañan de **intervalos de confianza bootstrap al 95 %** (percentil; B=2000 remuestreos con reemplazo sobre las 400 imágenes de test). El objetivo es cuantificar la incertidumbre muestral: con solo 40 glaucomas en el test, diferencias de unas pocas centésimas de AUC no son distinguibles del ruido, y reportar el intervalo es más honesto que reportar un único número. No se comparan versiones; se describe únicamente la versión final. No reentrena ni recalibra.

In [ ]:
# ============================================================
# 12.b SIGNIFICANCIA ESTADÍSTICA: INTERVALOS DE CONFIANZA BOOTSTRAP
# ============================================================
# IC bootstrap (percentil) al 95% sobre Test400. No reentrena ni recalibra.

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

CI_BOOTSTRAP_N = 2000
CI_ALPHA = 0.05
CI_SEED = CFG.SEED

# Carga segura de resultados de test.
if "test_results_df" not in globals():
    if os.path.exists(TEST_RESULTS_PATH):
        test_results_df = pd.read_csv(TEST_RESULTS_PATH)
        log_ok(f"Resultados test cargados desde: {TEST_RESULTS_PATH}")
    else:
        raise FileNotFoundError(
            "No se encuentra test_results_df ni TEST_RESULTS_PATH. Ejecuta primero la Sección 12."
        )

ci_df = test_results_df.copy()
ci_df["true_glaucoma_label"] = pd.to_numeric(ci_df["true_glaucoma_label"], errors="coerce")


def bootstrap_ci_mean(values, n_boot=CI_BOOTSTRAP_N, alpha=CI_ALPHA, seed=CI_SEED):
    """IC percentil para la media de una métrica por imagen (ignora NaN)."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    n = v.size
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot[i] = v[idx].mean()
    lo = float(np.percentile(boot, 100 * alpha / 2))
    hi = float(np.percentile(boot, 100 * (1 - alpha / 2)))
    return (float(v.mean()), lo, hi)


def bootstrap_ci_auc(y, score, n_boot=CI_BOOTSTRAP_N, alpha=CI_ALPHA, seed=CI_SEED):
    """IC percentil para el AUC. Descarta remuestras con una sola clase."""
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(y) & np.isfinite(score)
    y, score = y[mask].astype(int), score[mask]
    if y.size == 0 or len(np.unique(y)) < 2:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    n = y.size
    point = float(roc_auc_score(y, score))
    boot = []
    while len(boot) < n_boot:
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        boot.append(roc_auc_score(y[idx], score[idx]))
    lo = float(np.percentile(boot, 100 * alpha / 2))
    hi = float(np.percentile(boot, 100 * (1 - alpha / 2)))
    return (point, lo, hi)


def bootstrap_ci_rate(y, score, threshold, kind, n_boot=CI_BOOTSTRAP_N, alpha=CI_ALPHA, seed=CI_SEED):
    """IC percentil para sensibilidad ('sens') o especificidad ('spec') a umbral fijo."""
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(y) & np.isfinite(score)
    y, score = y[mask].astype(int), score[mask]
    rng = np.random.default_rng(seed)
    n = y.size

    def _rate(yy, ss):
        pred = (ss >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(yy, pred, labels=[0, 1]).ravel()
        if kind == "sens":
            return tp / (tp + fn) if (tp + fn) > 0 else np.nan
        return tn / (tn + fp) if (tn + fp) > 0 else np.nan

    point = _rate(y, score)
    boot = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r = _rate(y[idx], score[idx])
        if np.isfinite(r):
            boot.append(r)
    lo = float(np.percentile(boot, 100 * alpha / 2)) if boot else np.nan
    hi = float(np.percentile(boot, 100 * (1 - alpha / 2))) if boot else np.nan
    return (float(point) if np.isfinite(point) else np.nan, lo, hi)


rows = []

# Segmentación y biomarcador (media por imagen).
for col, label in [
    ("seg_disc_dice", "Disco Dice (test)"),
    ("seg_cup_dice", "Copa Dice (test)"),
    ("seg_disc_iou", "Disco IoU (test)"),
    ("seg_cup_iou", "Copa IoU (test)"),
    ("abs_error_vCDR", "MAE vCDR (test)"),
]:
    if col in ci_df.columns:
        m, lo, hi = bootstrap_ci_mean(ci_df[col].values)
        rows.append({"métrica": label, "valor": m, "IC95_low": lo, "IC95_high": hi})

# AUC del score combinado.
if "risk_score_combined" in ci_df.columns:
    m, lo, hi = bootstrap_ci_auc(ci_df["true_glaucoma_label"].values, ci_df["risk_score_combined"].values)
    rows.append({"métrica": "AUC combinado (test)", "valor": m, "IC95_low": lo, "IC95_high": hi})

# Sensibilidad / especificidad en el umbral calibrado en Validation400.
_thr = float(globals().get("FINAL_THRESHOLD", 0.288396))
if "risk_score_combined" in ci_df.columns:
    y_ci = ci_df["true_glaucoma_label"].values
    s_ci = ci_df["risk_score_combined"].values
    sens, s_lo, s_hi = bootstrap_ci_rate(y_ci, s_ci, _thr, "sens")
    spec, sp_lo, sp_hi = bootstrap_ci_rate(y_ci, s_ci, _thr, "spec")
    rows.append({"métrica": f"Sensibilidad @{_thr:.3f}", "valor": sens, "IC95_low": s_lo, "IC95_high": s_hi})
    rows.append({"métrica": f"Especificidad @{_thr:.3f}", "valor": spec, "IC95_low": sp_lo, "IC95_high": sp_hi})

ci_table = pd.DataFrame(rows)
ci_table_path = os.path.join(CFG.SAVE_PATH, "test_metrics_ci.csv")
ci_table.to_csv(ci_table_path, index=False)

log_ok(f"IC bootstrap (B={CI_BOOTSTRAP_N}, percentil 95%) calculados sobre Test400.")
print(ci_table.round(4).to_string(index=False))
log_info(f"Tabla de IC guardada en: {ci_table_path}")

# 13. Análisis visual de errores y casos representativos

Este bloque selecciona ejemplos relevantes de REFUGE-Test400: verdaderos positivos, verdaderos negativos, falsos positivos, falsos negativos y casos con mayor error de vCDR. Visualiza imagen, máscara real, predicción, overlay, biomarcadores y score final. No reentrena modelos ni recalibra umbrales; sirve para interpretar clínicamente por qué el sistema acierta o falla.

In [ ]:
# ============================================================
# 13. ANÁLISIS VISUAL DE ERRORES Y CASOS REPRESENTATIVOS
# ============================================================

# ------------------------------------------------------------
# 13.1. Configuración del punto operativo de análisis
# ------------------------------------------------------------

FINAL_ANALYSIS_SCORE_COL = "risk_score_combined"
FINAL_ANALYSIS_STRATEGY = "sensitivity_at_least_0.85"

ERROR_ANALYSIS_DIR = os.path.join(
    CFG.SAVE_PATH,
    "error_analysis"
)

os.makedirs(ERROR_ANALYSIS_DIR, exist_ok=True)

ERROR_ANALYSIS_TABLE_PATH = os.path.join(
    ERROR_ANALYSIS_DIR,
    "test_error_analysis_cases.csv"
)

log_info(f"Score de análisis: {FINAL_ANALYSIS_SCORE_COL}")
log_info(f"Estrategia de análisis: {FINAL_ANALYSIS_STRATEGY}")


# ------------------------------------------------------------
# 13.2. Carga segura de resultados de test y umbrales
# ------------------------------------------------------------

if "test_results_df" not in globals():
    if os.path.exists(TEST_RESULTS_PATH):
        test_results_df = pd.read_csv(TEST_RESULTS_PATH)
        log_ok(f"Resultados test cargados desde: {TEST_RESULTS_PATH}")
    else:
        raise FileNotFoundError("No se encuentra test_results_df ni TEST_RESULTS_PATH.")

if "threshold_df" not in globals():
    if os.path.exists(THRESHOLD_RESULTS_PATH):
        threshold_df = pd.read_csv(THRESHOLD_RESULTS_PATH)
        log_ok(f"Umbrales cargados desde: {THRESHOLD_RESULTS_PATH}")
    else:
        raise FileNotFoundError("No se encuentra threshold_df ni THRESHOLD_RESULTS_PATH.")


def get_threshold_from_validation(score_col, strategy):
    """
    Recupera el umbral calibrado en Validation400.
    """
    row = threshold_df[
        (threshold_df["score_col"] == score_col)
        & (threshold_df["strategy"] == strategy)
    ]

    if len(row) == 0:
        raise ValueError(
            f"No se encuentra umbral para {score_col} con estrategia {strategy}."
        )

    return float(row.iloc[0]["threshold"])


FINAL_ANALYSIS_THRESHOLD = get_threshold_from_validation(
    FINAL_ANALYSIS_SCORE_COL,
    FINAL_ANALYSIS_STRATEGY
)

log_ok(
    f"Umbral de análisis seleccionado: {FINAL_ANALYSIS_THRESHOLD:.6f}"
)


# ------------------------------------------------------------
# 13.3. Asegurar scores derivados en test
# ------------------------------------------------------------

def minmax_clip_analysis(series, low, high):
    values = pd.to_numeric(series, errors="coerce")
    scaled = (values - low) / (high - low)
    return np.clip(scaled, 0.0, 1.0)


test_analysis_df = test_results_df.copy()

for col in [
    "true_glaucoma_label",
    "pred_vCDR",
    "pred_rCDR",
    "pred_area_CDR",
    "pred_isnt_like_risk",
    "true_vCDR",
    "abs_error_vCDR",
    "seg_disc_iou",
    "seg_cup_iou",
    "quality_mean_confidence",
    "quality_mean_entropy",
]:
    if col in test_analysis_df.columns:
        test_analysis_df[col] = pd.to_numeric(
            test_analysis_df[col],
            errors="coerce"
        )

if "score_isnt_like" not in test_analysis_df.columns:
    if "pred_isnt_like_risk" in test_analysis_df.columns:
        test_analysis_df["score_isnt_like"] = pd.to_numeric(
            test_analysis_df["pred_isnt_like_risk"],
            errors="coerce"
        )
    else:
        test_analysis_df["score_isnt_like"] = np.nan

if "risk_score_vcdr_only" not in test_analysis_df.columns:
    test_analysis_df["risk_score_vcdr_only"] = minmax_clip_analysis(
        test_analysis_df["pred_vCDR"],
        0.45,
        0.85
    )

if "risk_score_combined" not in test_analysis_df.columns:
    vcdr_score = minmax_clip_analysis(test_analysis_df["pred_vCDR"], 0.45, 0.85)
    rcdr_score = minmax_clip_analysis(test_analysis_df["pred_rCDR"], 0.50, 0.85)
    isnt_score = test_analysis_df["score_isnt_like"].fillna(0.5)

    test_analysis_df["risk_score_combined"] = (
        0.70 * vcdr_score
        + 0.20 * rcdr_score
        + 0.10 * isnt_score
    )


# ------------------------------------------------------------
# 13.4. Etiquetado de aciertos y errores
# ------------------------------------------------------------

test_analysis_df["analysis_score"] = pd.to_numeric(
    test_analysis_df[FINAL_ANALYSIS_SCORE_COL],
    errors="coerce"
)

test_analysis_df["analysis_pred_label"] = (
    test_analysis_df["analysis_score"] >= FINAL_ANALYSIS_THRESHOLD
).astype(int)

test_analysis_df["error_type"] = "correct"

test_analysis_df.loc[
    (test_analysis_df["true_glaucoma_label"] == 0)
    & (test_analysis_df["analysis_pred_label"] == 0),
    "error_type"
] = "true_negative"

test_analysis_df.loc[
    (test_analysis_df["true_glaucoma_label"] == 1)
    & (test_analysis_df["analysis_pred_label"] == 1),
    "error_type"
] = "true_positive"

test_analysis_df.loc[
    (test_analysis_df["true_glaucoma_label"] == 0)
    & (test_analysis_df["analysis_pred_label"] == 1),
    "error_type"
] = "false_positive"

test_analysis_df.loc[
    (test_analysis_df["true_glaucoma_label"] == 1)
    & (test_analysis_df["analysis_pred_label"] == 0),
    "error_type"
] = "false_negative"

test_analysis_df["large_vcdr_error"] = (
    pd.to_numeric(test_analysis_df["abs_error_vCDR"], errors="coerce") >= 0.15
)

log_info("Distribución de tipos de caso:")
display(test_analysis_df["error_type"].value_counts())


# ------------------------------------------------------------
# 13.5. Selección de casos representativos
# ------------------------------------------------------------

def select_case_indices(df):
    """
    Selecciona casos representativos:
        - verdaderos positivos;
        - verdaderos negativos;
        - falsos positivos;
        - falsos negativos;
        - mayor error de vCDR;
        - mayor incertidumbre.
    """

    selected = []

    def add_cases(sub_df, case_group, n=3, sort_col=None, ascending=False):
        if len(sub_df) == 0:
            return

        temp = sub_df.copy()

        if sort_col is not None and sort_col in temp.columns:
            temp = temp.sort_values(
                by=sort_col,
                ascending=ascending
            )

        temp = temp.head(n).copy()
        temp["case_group"] = case_group

        selected.append(temp)

    add_cases(
        test_analysis_df[test_analysis_df["error_type"] == "true_positive"],
        "true_positive_high_score",
        n=3,
        sort_col="analysis_score",
        ascending=False
    )

    add_cases(
        test_analysis_df[test_analysis_df["error_type"] == "true_negative"],
        "true_negative_low_score",
        n=3,
        sort_col="analysis_score",
        ascending=True
    )

    add_cases(
        test_analysis_df[test_analysis_df["error_type"] == "false_positive"],
        "false_positive_high_score",
        n=5,
        sort_col="analysis_score",
        ascending=False
    )

    add_cases(
        test_analysis_df[test_analysis_df["error_type"] == "false_negative"],
        "false_negative_low_score",
        n=5,
        sort_col="analysis_score",
        ascending=True
    )

    add_cases(
        test_analysis_df,
        "largest_vcdr_error",
        n=5,
        sort_col="abs_error_vCDR",
        ascending=False
    )

    add_cases(
        test_analysis_df,
        "highest_entropy",
        n=5,
        sort_col="quality_mean_entropy",
        ascending=False
    )

    if len(selected) == 0:
        return pd.DataFrame()

    selected_df = pd.concat(selected, ignore_index=True)

    selected_df = selected_df.drop_duplicates(
        subset=["index", "case_group"]
    ).reset_index(drop=True)

    return selected_df


selected_cases_df = select_case_indices(test_analysis_df)

selected_cases_df.to_csv(
    ERROR_ANALYSIS_TABLE_PATH,
    index=False
)

log_ok(f"Tabla de casos seleccionados guardada en: {ERROR_ANALYSIS_TABLE_PATH}")

display(
    selected_cases_df[
        [
            "case_group",
            "index",
            "image_id",
            "true_glaucoma_label",
            "analysis_pred_label",
            "error_type",
            "analysis_score",
            "pred_vCDR",
            "true_vCDR",
            "abs_error_vCDR",
            "seg_disc_iou",
            "seg_cup_iou",
            "quality_mean_confidence",
            "quality_mean_entropy",
        ]
    ]
)


# ------------------------------------------------------------
# 13.6. Reconstrucción de resultado visual desde índice
# ------------------------------------------------------------

def get_test_result_visual_by_index(index):
    """
    Reconstruye los elementos visuales para un caso de Test400.

    Usa:
        - test_data;
        - test_pred_memmap;
        - test_true_masks_memmap.
    """

    index = int(index)

    image_path, mask_path = test_data[index]

    image_preprocessed, image_resized, metadata = preprocess_image_only(
        image_path
    )

    probability_map = np.array(
        test_pred_memmap[index],
        dtype=np.float32
    )

    raw_mask = np.argmax(
        probability_map,
        axis=-1
    ).astype(np.uint8)

    final_mask, postprocess_diagnostics = postprocess_segmentation_mask(
        raw_mask
    )

    true_mask = np.array(
        test_true_masks_memmap[index],
        dtype=np.uint8
    )

    confidence_maps = compute_confidence_maps(
        probability_map
    )

    quality_summary = summarize_prediction_quality(
        probability_map=probability_map,
        postprocess_diagnostics=postprocess_diagnostics
    )

    result = {
        "image_path": image_path,
        "mask_path": mask_path,
        "image_resized": image_resized,
        "image_preprocessed": image_preprocessed,
        "probability_map": probability_map,
        "raw_mask": raw_mask,
        "final_mask": final_mask,
        "true_mask": true_mask,
        "metadata": metadata,
        "postprocess_diagnostics": postprocess_diagnostics,
        "confidence_maps": confidence_maps,
        "quality_summary": quality_summary,
    }

    return result


# ------------------------------------------------------------
# 13.7. Visualización enriquecida de casos
# ------------------------------------------------------------

def visualize_case_analysis(row):
    """
    Visualiza un caso representativo con biomarcadores y predicción clínica.
    """

    result = get_test_result_visual_by_index(
        row["index"]
    )

    image = result["image_resized"]
    true_mask = result["true_mask"]
    final_mask = result["final_mask"]
    entropy = result["confidence_maps"]["entropy"]

    overlay_pred = create_overlay(
        image,
        final_mask,
        alpha=0.35
    )

    overlay_true = create_overlay(
        image,
        true_mask,
        alpha=0.35
    )

    plt.figure(figsize=(20, 5))

    plt.subplot(1, 5, 1)
    plt.imshow(image)
    plt.title("Imagen ROI")
    plt.axis("off")

    plt.subplot(1, 5, 2)
    plt.imshow(overlay_true)
    plt.title("GT overlay")
    plt.axis("off")

    plt.subplot(1, 5, 3)
    plt.imshow(overlay_pred)
    plt.title("Pred overlay")
    plt.axis("off")

    plt.subplot(1, 5, 4)
    plt.imshow(final_mask, cmap="viridis", vmin=0, vmax=2)
    plt.title("Pred máscara")
    plt.axis("off")

    plt.subplot(1, 5, 5)
    plt.imshow(entropy, cmap="magma", vmin=0, vmax=1)
    plt.title("Incertidumbre")
    plt.axis("off")

    plt.suptitle(
        f"{row['case_group']} | {row['image_id']} | "
        f"real={int(row['true_glaucoma_label'])} pred={int(row['analysis_pred_label'])} | "
        f"score={row['analysis_score']:.3f} | "
        f"vCDR pred={row['pred_vCDR']:.3f} true={row['true_vCDR']:.3f} | "
        f"err={row['abs_error_vCDR']:.3f}",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()

    log_info(
        f"Tipo: {row['error_type']} | "
        f"Score: {row['analysis_score']:.4f} | "
        f"Umbral: {FINAL_ANALYSIS_THRESHOLD:.4f}"
    )

    log_info(
        f"Disc IoU: {row['seg_disc_iou']:.4f} | "
        f"Cup IoU: {row['seg_cup_iou']:.4f} | "
        f"Confianza media: {row['quality_mean_confidence']:.4f} | "
        f"Entropía media: {row['quality_mean_entropy']:.4f}"
    )


# ------------------------------------------------------------
# 13.8. Mostrar casos seleccionados
# ------------------------------------------------------------

MAX_CASES_TO_VISUALIZE = 12

for _, row in selected_cases_df.head(MAX_CASES_TO_VISUALIZE).iterrows():
    visualize_case_analysis(row)


# ------------------------------------------------------------
# 13.9. Resumen textual automático
# ------------------------------------------------------------

error_summary = test_analysis_df["error_type"].value_counts().to_dict()

mean_fp_score = test_analysis_df.loc[
    test_analysis_df["error_type"] == "false_positive",
    "analysis_score"
].mean()

mean_fn_score = test_analysis_df.loc[
    test_analysis_df["error_type"] == "false_negative",
    "analysis_score"
].mean()

mean_fp_vcdr = test_analysis_df.loc[
    test_analysis_df["error_type"] == "false_positive",
    "pred_vCDR"
].mean()

mean_fn_vcdr = test_analysis_df.loc[
    test_analysis_df["error_type"] == "false_negative",
    "pred_vCDR"
].mean()

log_ok("Análisis visual de errores completado.")

log_info(f"Resumen de errores: {error_summary}")

log_info(
    f"Score medio falsos positivos: {mean_fp_score:.4f} | "
    f"vCDR pred medio FP: {mean_fp_vcdr:.4f}"
)

log_info(
    f"Score medio falsos negativos: {mean_fn_score:.4f} | "
    f"vCDR pred medio FN: {mean_fn_vcdr:.4f}"
)
# ============================================================
# 13 (cont.). AUDITORÍA TÉCNICA DE ROI/GT
# ============================================================

AUDIT_DIR = os.path.join(
    CFG.SAVE_PATH,
    "gt_audit"
)

os.makedirs(AUDIT_DIR, exist_ok=True)

AUDIT_RESULTS_PATH = os.path.join(
    AUDIT_DIR,
    "test_gt_audit_results.csv"
)

log_info(f"Directorio de auditoría GT: {AUDIT_DIR}")


# ------------------------------------------------------------
# 13.10. Auditoría ROI de cobertura total (Training400 / Validation400 / Test400)
# ------------------------------------------------------------
# Recorre TODAS las imágenes de los tres conjuntos y mide, usando la máscara GT
# SOLO para verificar (nunca para localizar), si el recorte ROI captura disco y copa.
# Además compara el centro nuevo (con guarda) frente al centro del algoritmo antiguo
# (`_legacy_locate_roi_center`) para cuantificar el distribution shift y decidir, de
# forma objetiva, si hace falta reentrenar.

def compute_roi_mask_areas(mask_gray, cx, cy):
    """Áreas de disco y copa dentro del recorte ROI centrado en (cx, cy)."""
    mask_crop = crop_square_with_padding(
        mask_gray, cx, cy, CFG.ROI_RADIUS, is_mask=True
    )
    mask_resized = cv2.resize(
        mask_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )
    roi_class = decode_refuge_mask(mask_resized)

    roi_disc = int(np.logical_or(roi_class == 1, roi_class == 2).sum())
    roi_cup = int((roi_class == 2).sum())

    return roi_disc, roi_cup


def audit_roi_coverage(pairs, split_name):
    """
    Auditoría ROI de cobertura total sobre una lista de pares (imagen, máscara).
    Para cada caso registra: áreas GT completas, áreas GT dentro de la ROI,
    validez full/roi, centro antiguo, centro nuevo y desplazamiento entre ambos.
    """
    records = []

    for idx in range(len(pairs)):
        image_path, mask_path = pairs[idx]

        try:
            image_rgb = read_rgb_image(image_path)
            mask_gray = read_mask_grayscale(mask_path)
        except Exception as exc:
            records.append(
                {
                    "split": split_name,
                    "index": idx,
                    "image_id": os.path.basename(str(image_path)),
                    "error": str(exc),
                }
            )
            continue

        full_class = decode_refuge_mask(mask_gray)
        full_disc = int(np.logical_or(full_class == 1, full_class == 2).sum())
        full_cup = int((full_class == 2).sum())

        cx_old, cy_old = _legacy_locate_roi_center(image_rgb)   # algoritmo antiguo
        cx_new, cy_new = locate_roi_center(image_rgb)           # nuevo con guarda

        roi_disc, roi_cup = compute_roi_mask_areas(mask_gray, cx_new, cy_new)

        displacement = float(np.hypot(cx_new - cx_old, cy_new - cy_old))

        records.append(
            {
                "split": split_name,
                "index": idx,
                "image_id": os.path.basename(str(image_path)),
                "full_disc_area": full_disc,
                "full_cup_area": full_cup,
                "roi_disc_area": roi_disc,
                "roi_cup_area": roi_cup,
                "gt_valid_full": bool(full_disc > 0 and full_cup > 0),
                "gt_valid_roi": bool(roi_disc > 0 and roi_cup > 0),
                "cx_old": int(cx_old),
                "cy_old": int(cy_old),
                "cx_new": int(cx_new),
                "cy_new": int(cy_new),
                "center_displacement_px": displacement,
                "center_changed": bool(displacement > 1e-6),
            }
        )

    return pd.DataFrame(records)


def summarize_roi_audit(df, split_name):
    """Resumen por conjunto: casos perdidos por la ROI y desplazamiento de centros."""
    if "gt_valid_full" not in df.columns or len(df) == 0:
        return {"split": split_name, "n_images": int(len(df))}, df.iloc[0:0]

    valid_full = df["gt_valid_full"].fillna(False)
    missed = df[valid_full & (~df["gt_valid_roi"].fillna(False))]

    disp = pd.to_numeric(df["center_displacement_px"], errors="coerce").dropna()
    tol = 0.05 * CFG.ROI_RADIUS  # 10 px con ROI_RADIUS = 200

    summary = {
        "split": split_name,
        "n_images": int(len(df)),
        "n_gt_valid_full": int(valid_full.sum()),
        "n_roi_crop_missed_gt_disc": int(len(missed)),
        "pct_center_identical": float((disp <= 1e-6).mean() * 100) if len(disp) else float("nan"),
        "median_displacement_px": float(disp.median()) if len(disp) else float("nan"),
        "p95_displacement_px": float(disp.quantile(0.95)) if len(disp) else float("nan"),
        "pct_displacement_gt_tol": float((disp > tol).mean() * 100) if len(disp) else float("nan"),
    }

    return summary, missed


# Conjuntos disponibles (Training400, Validation400, Test400).
roi_audit_inputs = []

if "train_data" in globals():
    roi_audit_inputs.append(("Training400", train_data))

if "external_val_data" in globals():
    roi_audit_inputs.append(("Validation400", external_val_data))

if "test_data" in globals():
    roi_audit_inputs.append(("Test400", test_data))

roi_audit_frames = []
roi_audit_summaries = []
roi_audit_missed = {}

for split_name, pairs in roi_audit_inputs:
    log_info(f"Auditando ROI en {split_name} ({len(pairs)} imágenes)...")

    df_split = audit_roi_coverage(pairs, split_name)
    summary, missed = summarize_roi_audit(df_split, split_name)

    roi_audit_frames.append(df_split)
    roi_audit_summaries.append(summary)
    roi_audit_missed[split_name] = missed

    log_ok(
        f"{split_name}: roi_crop_missed_gt_disc={summary.get('n_roi_crop_missed_gt_disc')}, "
        f"centros idénticos={summary.get('pct_center_identical', float('nan')):.1f}%, "
        f"desplazamiento p95={summary.get('p95_displacement_px', float('nan')):.1f}px"
    )

if roi_audit_frames:
    roi_coverage_audit_df = pd.concat(roi_audit_frames, ignore_index=True)
else:
    roi_coverage_audit_df = pd.DataFrame()

roi_audit_summary_df = pd.DataFrame(roi_audit_summaries)

ROI_AUDIT_FULL_PATH = os.path.join(AUDIT_DIR, "roi_coverage_audit_full.csv")
ROI_AUDIT_SUMMARY_PATH = os.path.join(AUDIT_DIR, "roi_coverage_audit_summary.csv")

roi_coverage_audit_df.to_csv(ROI_AUDIT_FULL_PATH, index=False)
roi_audit_summary_df.to_csv(ROI_AUDIT_SUMMARY_PATH, index=False)

log_ok(f"Auditoría ROI de cobertura total guardada en: {ROI_AUDIT_FULL_PATH}")
display(roi_audit_summary_df)


# ------------------------------------------------------------
# 13.10.1. Visualización antiguo-vs-nuevo de los casos corregidos por la nueva ROI
# ------------------------------------------------------------
def visualize_roi_correction(row):
    """Muestra centro antiguo (rojo) y nuevo (verde) + recuadros ROI de un caso."""
    split_name = row["split"]
    idx = int(row["index"])

    pairs = dict(roi_audit_inputs)[split_name]
    image_path, _ = pairs[idx]
    image_rgb = read_rgb_image(image_path)

    r = CFG.ROI_RADIUS
    plt.figure(figsize=(7, 7))
    plt.imshow(image_rgb)
    ax = plt.gca()

    for (cx, cy, color, label) in [
        (row["cx_old"], row["cy_old"], "red", "ROI antigua"),
        (row["cx_new"], row["cy_new"], "lime", "ROI nueva"),
    ]:
        plt.scatter([cx], [cy], c=color, s=40)
        ax.add_patch(
            plt.Rectangle(
                (cx - r, cy - r), 2 * r, 2 * r,
                fill=False, edgecolor=color, linewidth=2, label=label
            )
        )

    plt.legend(loc="lower right")
    plt.title(f"{split_name} | {row['image_id']} | desp={row['center_displacement_px']:.0f}px")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# Casos donde la nueva ROI corrigió un recorte que antes perdía el disco.
corrected_rows = []

for split_name, pairs in roi_audit_inputs:
    df_split = roi_coverage_audit_df[roi_coverage_audit_df["split"] == split_name]
    if "gt_valid_full" not in df_split.columns:
        continue

    # Caso corregido = GT válida completa, centro cambió y ahora la ROI sí es válida.
    fixed = df_split[
        df_split["gt_valid_full"].fillna(False)
        & df_split["center_changed"].fillna(False)
        & df_split["gt_valid_roi"].fillna(False)
    ]
    for _, r in fixed.iterrows():
        corrected_rows.append(r)

log_info(f"Casos con ROI modificada y GT ahora capturada: {len(corrected_rows)}")

MAX_ROI_CORRECTION_VISUALS = 8
for r in corrected_rows[:MAX_ROI_CORRECTION_VISUALS]:
    visualize_roi_correction(r)


# ------------------------------------------------------------
# 13.10.2. Decisión objetiva sobre reentrenamiento
# ------------------------------------------------------------
def decide_retraining(summary_df):
    """
    Criterio: NO reentrenar si la nueva ROI corrige los fallos sin mover la
    distribución de recortes (la mayoría de centros idénticos al antiguo).
    """
    test_row = summary_df[summary_df["split"] == "Test400"]
    if len(test_row) == 0:
        log_warning("No hay resumen de Test400; ejecuta antes el punto 12.")
        return None

    test_row = test_row.iloc[0]
    n_missed = test_row.get("n_roi_crop_missed_gt_disc", np.nan)
    pct_shift = test_row.get("pct_displacement_gt_tol", np.nan)

    no_retrain = (n_missed == 0) and (pct_shift < 5.0)

    log_info("Criterio de reentrenamiento:")
    log_info(f"  - Test400 roi_crop_missed_gt_disc = {n_missed} (objetivo: 0)")
    log_info(f"  - Test400 % imágenes con desplazamiento > tolerancia = {pct_shift:.2f}% (objetivo: < 5%)")

    if no_retrain:
        log_ok(
            "DECISIÓN: NO es necesario reentrenar. La nueva ROI corrige los fallos sin "
            "alterar la distribución de recortes; los modelos actuales siguen siendo válidos."
        )
    else:
        log_warning(
            "DECISIÓN: revisar reentrenamiento. La ROI cambió la distribución de recortes o "
            "persisten fallos; si las métricas de test empeoran, reejecutar el punto 6 con la nueva ROI."
        )

    return no_retrain


roi_retrain_decision = decide_retraining(roi_audit_summary_df)




# ------------------------------------------------------------
# 13.11. Carga segura de resultados de test
# ------------------------------------------------------------

if "test_results_df" not in globals():
    if os.path.exists(TEST_RESULTS_PATH):
        test_results_df = pd.read_csv(TEST_RESULTS_PATH)
        log_ok(f"Resultados test cargados desde: {TEST_RESULTS_PATH}")
    else:
        raise FileNotFoundError("No se encuentra test_results_df ni TEST_RESULTS_PATH.")

if "test_data" not in globals():
    raise NameError("No existe test_data. Ejecuta antes el punto 12.")

if "test_pred_memmap" not in globals():
    log_warning("test_pred_memmap no está en memoria. Algunas visualizaciones de predicción pueden no estar disponibles.")

if "test_true_masks_memmap" not in globals():
    log_warning("test_true_masks_memmap no está en memoria. Se reconstruirán las máscaras desde archivo.")


# ------------------------------------------------------------
# 13.12. Selección de casos sospechosos
# ------------------------------------------------------------

audit_df = test_results_df.copy()

for col in [
    "true_vCDR",
    "pred_vCDR",
    "abs_error_vCDR",
    "seg_disc_iou",
    "seg_cup_iou",
    "quality_mean_entropy",
    "quality_mean_confidence",
]:
    if col in audit_df.columns:
        audit_df[col] = pd.to_numeric(
            audit_df[col],
            errors="coerce"
        )

suspicious_mask = (
    audit_df["true_vCDR"].isna()
    | (audit_df["seg_disc_iou"] < 0.05)
    | (audit_df["seg_cup_iou"] < 0.05)
    | (audit_df["abs_error_vCDR"] >= 0.25)
    | (audit_df["pred_vCDR"] >= 0.95)
)

suspicious_cases_df = audit_df[suspicious_mask].copy()

suspicious_cases_df = suspicious_cases_df.sort_values(
    by=["true_vCDR", "abs_error_vCDR", "seg_disc_iou"],
    ascending=[True, False, True]
)

log_info(f"Casos sospechosos detectados: {len(suspicious_cases_df)}")

display(
    suspicious_cases_df[
        [
            "index",
            "image_id",
            "true_glaucoma_label",
            "pred_vCDR",
            "true_vCDR",
            "abs_error_vCDR",
            "seg_disc_iou",
            "seg_cup_iou",
            "risk_score_combined",
            "quality_mean_confidence",
            "quality_mean_entropy",
        ]
    ].head(30)
)


# ------------------------------------------------------------
# 13.13. Inspección de máscaras originales y recorte ROI
# ------------------------------------------------------------

def inspect_test_pair_integrity(index):
    """
    Inspecciona la integridad de imagen, máscara GT y ROI para un caso de Test400.
    """

    index = int(index)

    image_path, mask_path = test_data[index]

    image_id = normalize_label_key(image_path)
    mask_id = normalize_label_key(mask_path)

    image_num = numeric_key(image_path)
    mask_num = numeric_key(mask_path)

    image_exists = os.path.exists(image_path)
    mask_exists = os.path.exists(mask_path)

    record = {
        "index": index,
        "image_path": image_path,
        "mask_path": mask_path,
        "image_id": image_id,
        "mask_id": mask_id,
        "image_num": image_num,
        "mask_num": mask_num,
        "image_exists": image_exists,
        "mask_exists": mask_exists,
        "numeric_match": image_num == mask_num,
    }

    if not image_exists or not mask_exists:
        record.update(
            {
                "raw_mask_unique_values": None,
                "full_disc_area": np.nan,
                "full_cup_area": np.nan,
                "roi_disc_area": np.nan,
                "roi_cup_area": np.nan,
                "roi_center_x": np.nan,
                "roi_center_y": np.nan,
                "gt_valid_full": False,
                "gt_valid_roi": False,
            }
        )

        return record

    image_rgb = read_rgb_image(image_path)
    mask_gray = read_mask_grayscale(mask_path)

    raw_unique_values = np.unique(mask_gray)

    full_mask_class = decode_refuge_mask(mask_gray)

    full_disc = np.logical_or(
        full_mask_class == 1,
        full_mask_class == 2
    )

    full_cup = full_mask_class == 2

    full_disc_area = int(full_disc.sum())
    full_cup_area = int(full_cup.sum())

    cx, cy = locate_roi_center(image_rgb)

    mask_crop = crop_square_with_padding(
        mask_gray,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=True
    )

    mask_resized = cv2.resize(
        mask_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    roi_mask_class = decode_refuge_mask(mask_resized)

    roi_disc = np.logical_or(
        roi_mask_class == 1,
        roi_mask_class == 2
    )

    roi_cup = roi_mask_class == 2

    roi_disc_area = int(roi_disc.sum())
    roi_cup_area = int(roi_cup.sum())

    record.update(
        {
            "raw_mask_unique_values": str(raw_unique_values.tolist()),
            "image_shape": str(image_rgb.shape),
            "mask_shape": str(mask_gray.shape),
            "roi_center_x": int(cx),
            "roi_center_y": int(cy),
            "full_disc_area": full_disc_area,
            "full_cup_area": full_cup_area,
            "roi_disc_area": roi_disc_area,
            "roi_cup_area": roi_cup_area,
            "gt_valid_full": bool(full_disc_area > 0 and full_cup_area > 0),
            "gt_valid_roi": bool(roi_disc_area > 0 and roi_cup_area > 0),
        }
    )

    return record


audit_records = []

for idx in suspicious_cases_df["index"].tolist():
    audit_records.append(
        inspect_test_pair_integrity(idx)
    )

gt_audit_df = pd.DataFrame(audit_records)

gt_audit_df.to_csv(
    AUDIT_RESULTS_PATH,
    index=False
)

log_ok(f"Auditoría GT guardada en: {AUDIT_RESULTS_PATH}")

display(gt_audit_df)


# ------------------------------------------------------------
# 13.14. Resumen de causas probables
# ------------------------------------------------------------

def classify_audit_issue(row):
    """
    Clasifica causa probable de anomalía.
    """

    if not row["image_exists"] or not row["mask_exists"]:
        return "missing_file"

    if not row["numeric_match"]:
        return "possible_pairing_mismatch"

    if not row["gt_valid_full"]:
        return "empty_or_invalid_full_gt_mask"

    if row["gt_valid_full"] and not row["gt_valid_roi"]:
        return "roi_crop_missed_gt_disc"

    if row["gt_valid_roi"]:
        return "valid_gt_possible_model_error"

    return "unknown"


if len(gt_audit_df) > 0:
    gt_audit_df["probable_issue"] = gt_audit_df.apply(
        classify_audit_issue,
        axis=1
    )

    display(
        gt_audit_df[
            [
                "index",
                "image_id",
                "mask_id",
                "numeric_match",
                "full_disc_area",
                "full_cup_area",
                "roi_disc_area",
                "roi_cup_area",
                "gt_valid_full",
                "gt_valid_roi",
                "probable_issue",
            ]
        ]
    )

    log_info("Distribución de causas probables:")
    display(gt_audit_df["probable_issue"].value_counts())


# ------------------------------------------------------------
# 13.15. Visualización manual de casos sospechosos
# ------------------------------------------------------------

def visualize_gt_audit_case(index):
    """
    Visualiza imagen, máscara GT original, ROI GT y predicción para auditar un caso.
    """

    index = int(index)

    image_path, mask_path = test_data[index]

    image_rgb = read_rgb_image(image_path)
    mask_gray = read_mask_grayscale(mask_path)

    cx, cy = locate_roi_center(image_rgb)

    image_crop = crop_square_with_padding(
        image_rgb,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=False
    )

    image_resized = cv2.resize(
        image_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    mask_crop = crop_square_with_padding(
        mask_gray,
        cx,
        cy,
        CFG.ROI_RADIUS,
        is_mask=True
    )

    mask_resized = cv2.resize(
        mask_crop,
        (CFG.IMG_SIZE, CFG.IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    true_mask_roi = decode_refuge_mask(mask_resized)

    if "test_pred_memmap" in globals():
        probability_map = np.array(
            test_pred_memmap[index],
            dtype=np.float32
        )

        raw_mask = np.argmax(
            probability_map,
            axis=-1
        ).astype(np.uint8)

        pred_mask, _ = postprocess_segmentation_mask(raw_mask)

        pred_available = True
    else:
        pred_mask = np.zeros_like(true_mask_roi)
        pred_available = False

    overlay_true = create_overlay(
        image_resized,
        true_mask_roi,
        alpha=0.35
    )

    overlay_pred = create_overlay(
        image_resized,
        pred_mask,
        alpha=0.35
    )

    row = test_results_df[test_results_df["index"] == index]

    if len(row) > 0:
        row = row.iloc[0]
        title_info = (
            f"{row.get('image_id', index)} | "
            f"label={row.get('true_glaucoma_label', np.nan)} | "
            f"pred_vCDR={row.get('pred_vCDR', np.nan):.3f} | "
            f"true_vCDR={row.get('true_vCDR', np.nan)} | "
            f"discIoU={row.get('seg_disc_iou', np.nan):.3f} | "
            f"cupIoU={row.get('seg_cup_iou', np.nan):.3f}"
        )
    else:
        title_info = f"Index {index}"

    plt.figure(figsize=(22, 5))

    plt.subplot(1, 5, 1)
    plt.imshow(image_rgb)
    plt.scatter([cx], [cy], c="red", s=30)
    plt.title("Imagen original + centro ROI")
    plt.axis("off")

    plt.subplot(1, 5, 2)
    plt.imshow(mask_gray, cmap="gray")
    plt.title(f"GT original\nvalores={np.unique(mask_gray)[:10]}")
    plt.axis("off")

    plt.subplot(1, 5, 3)
    plt.imshow(overlay_true)
    plt.title("GT ROI overlay")
    plt.axis("off")

    plt.subplot(1, 5, 4)
    plt.imshow(overlay_pred)
    plt.title("Predicción overlay" if pred_available else "Pred no disponible")
    plt.axis("off")

    plt.subplot(1, 5, 5)
    plt.imshow(true_mask_roi, cmap="viridis", vmin=0, vmax=2)
    plt.title("GT ROI clase 0/1/2")
    plt.axis("off")

    plt.suptitle(
        title_info,
        fontsize=12
    )

    plt.tight_layout()
    plt.show()


# Visualizar los casos más problemáticos
MAX_AUDIT_VISUAL_CASES = 12

for idx in suspicious_cases_df["index"].head(MAX_AUDIT_VISUAL_CASES).tolist():
    visualize_gt_audit_case(idx)


# ------------------------------------------------------------
# 13.16. Recalcular resumen de segmentación excluyendo GT inválidas
# ------------------------------------------------------------

if len(gt_audit_df) > 0:
    invalid_gt_indices = gt_audit_df.loc[
        ~gt_audit_df["gt_valid_roi"],
        "index"
    ].astype(int).tolist()
else:
    invalid_gt_indices = []

log_info(f"Índices con GT ROI inválida: {invalid_gt_indices}")

test_results_valid_gt_df = test_results_df[
    ~test_results_df["index"].isin(invalid_gt_indices)
].copy()

metric_cols = [
    "seg_disc_iou",
    "seg_disc_dice",
    "seg_cup_iou",
    "seg_cup_dice",
    "seg_clinical_global_iou",
    "seg_clinical_global_dice",
    "abs_error_vCDR",
]

valid_gt_summary_records = []

for col in metric_cols:
    if col in test_results_valid_gt_df.columns:
        values = pd.to_numeric(
            test_results_valid_gt_df[col],
            errors="coerce"
        ).dropna()

        valid_gt_summary_records.append(
            {
                "metric": col,
                "mean_valid_gt": values.mean(),
                "std_valid_gt": values.std(),
                "median_valid_gt": values.median(),
                "n": len(values),
            }
        )

valid_gt_summary_df = pd.DataFrame(valid_gt_summary_records)

display(valid_gt_summary_df)

VALID_GT_SUMMARY_PATH = os.path.join(
    AUDIT_DIR,
    "test_segmentation_summary_valid_gt_only.csv"
)

valid_gt_summary_df.to_csv(
    VALID_GT_SUMMARY_PATH,
    index=False
)

log_ok(f"Resumen con GT válida guardado en: {VALID_GT_SUMMARY_PATH}")


# ------------------------------------------------------------
# 13.17. Conclusión automática de auditoría
# ------------------------------------------------------------

log_ok("Auditoría de casos anómalos completada.")

if len(gt_audit_df) > 0:
    issue_counts = gt_audit_df["probable_issue"].value_counts().to_dict()
    log_info(f"Causas probables detectadas: {issue_counts}")

    n_invalid_roi = int((~gt_audit_df["gt_valid_roi"]).sum())
    log_info(f"Casos sospechosos con GT ROI inválida: {n_invalid_roi}")

# 14. Resumen final del experimento y exportación de resultados

Este bloque cierra el notebook. Consolida en una única tabla los resultados de segmentación disco-copa, los biomarcadores (vCDR y derivados), las métricas de clasificación en validación y test, el punto operativo principal (`sensitivity_at_least_0.85`) frente a la alternativa equilibrada (Youden) y el resumen de la auditoría ROI. Exporta el informe final y deja explícitas las limitaciones y el encuadre clínico: se trata de un sistema de segmentación y estimación de biomarcadores morfológicos asociados a *sospecha* glaucomatosa, no de un diagnóstico automático de glaucoma. No reentrena, no recalibra y no modifica el ensemble.

In [ ]:
# ============================================================
# 14. RESUMEN FINAL DEL EXPERIMENTO Y EXPORTACIÓN DE RESULTADOS
# ============================================================
#
# Este bloque cierra el notebook. NO reentrena, NO recalibra y NO modifica el
# ensemble. Consolida en una sola tabla los resultados de segmentación,
# biomarcadores, clasificación y auditoría ROI, exporta el resumen y deja
# explícitas las limitaciones y el encuadre clínico del sistema.

FINAL_SUMMARY_DIR = os.path.join(CFG.SAVE_PATH, "final_summary")
os.makedirs(FINAL_SUMMARY_DIR, exist_ok=True)


# ------------------------------------------------------------
# 14.1. Utilidades de recuperación segura
# ------------------------------------------------------------
def _get_df(name, path=None):
    """Devuelve un DataFrame desde globals() o, si no, desde CSV. None si no existe."""
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        return globals()[name]
    if path and os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception as exc:
            log_warning(f"No se pudo leer {path}: {exc}")
    return None


def _mean_metric(df, col):
    if df is None or col not in df.columns:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").dropna().mean())


# ------------------------------------------------------------
# 14.2. Consolidación de métricas finales
# ------------------------------------------------------------
test_df = _get_df("test_results_df", globals().get("TEST_RESULTS_PATH"))
external_df = _get_df("external_results_df", globals().get("EXTERNAL_RESULTS_PATH"))
test_ops_df = _get_df("test_operating_points_df", globals().get("TEST_OPERATING_POINTS_PATH"))
roi_summary_df = _get_df("roi_audit_summary_df", globals().get("ROI_AUDIT_SUMMARY_PATH"))

final_rows = []

# Segmentación y biomarcadores en Test400.
final_rows.append({"bloque": "Segmentación", "métrica": "Disco IoU (test, media)", "valor": _mean_metric(test_df, "seg_disc_iou")})
final_rows.append({"bloque": "Segmentación", "métrica": "Copa IoU (test, media)", "valor": _mean_metric(test_df, "seg_cup_iou")})
final_rows.append({"bloque": "Segmentación", "métrica": "Disco Dice (test, media)", "valor": _mean_metric(test_df, "seg_disc_dice")})
final_rows.append({"bloque": "Segmentación", "métrica": "Copa Dice (test, media)", "valor": _mean_metric(test_df, "seg_cup_dice")})
final_rows.append({"bloque": "Biomarcador", "métrica": "MAE vCDR (test)", "valor": _mean_metric(test_df, "abs_error_vCDR")})

# AUC de clasificación (validación y test) si están disponibles en memoria.
def _auc_from(name):
    m = globals().get(name)
    if isinstance(m, dict):
        try:
            return float(m.get("auc", np.nan))
        except Exception:
            return np.nan
    return np.nan

# En validación, el score combinado se evalúa en classification_metrics_combined (punto 10).
final_rows.append({"bloque": "Clasificación", "métrica": "AUC combinado (validación)", "valor": _auc_from("classification_metrics_combined")})
final_rows.append({"bloque": "Clasificación", "métrica": "AUC combinado (test)", "valor": _auc_from("test_classification_metrics")})

final_metrics_df = pd.DataFrame(final_rows)
final_metrics_df["valor"] = pd.to_numeric(final_metrics_df["valor"], errors="coerce").round(4)

log_info("Resumen de métricas finales (Test400):")
display(final_metrics_df)

FINAL_METRICS_PATH = os.path.join(FINAL_SUMMARY_DIR, "final_metrics_summary.csv")
final_metrics_df.to_csv(FINAL_METRICS_PATH, index=False)


# ------------------------------------------------------------
# 14.3. Punto operativo principal y alternativa
# ------------------------------------------------------------
operating_summary_df = None

if test_ops_df is not None and "strategy" in test_ops_df.columns:
    wanted = test_ops_df[
        (test_ops_df.get("score_col") == "risk_score_combined")
        & (test_ops_df["strategy"].isin(["sensitivity_at_least_0.85", "youden"]))
    ].copy()

    show_cols = [c for c in [
        "score_col", "strategy", "threshold",
        "sensitivity", "specificity", "fp", "fn", "tp", "tn", "f1", "auc",
    ] if c in wanted.columns]

    operating_summary_df = wanted[show_cols] if show_cols else wanted

    log_info("Puntos operativos en Test400 (principal = sensitivity_at_least_0.85; alternativa = youden):")
    display(operating_summary_df)

    OPERATING_SUMMARY_PATH = os.path.join(FINAL_SUMMARY_DIR, "final_operating_points.csv")
    operating_summary_df.to_csv(OPERATING_SUMMARY_PATH, index=False)
else:
    log_warning("No hay test_operating_points_df; ejecuta antes el punto 12.")


# ------------------------------------------------------------
# 14.4. Resumen de la auditoría ROI (antes/después)
# ------------------------------------------------------------
if roi_summary_df is not None:
    log_info("Resumen de la auditoría ROI por conjunto:")
    display(roi_summary_df)
else:
    log_warning("No hay roi_audit_summary_df; ejecuta antes la auditoría del punto 13.")


# ------------------------------------------------------------
# 14.5. Limitaciones y encuadre clínico
# ------------------------------------------------------------
ENCUADRE_CLINICO = (
    "Este trabajo es un SISTEMA DE SEGMENTACIÓN DISCO-COPA Y ESTIMACIÓN DE "
    "BIOMARCADORES MORFOLÓGICOS (vCDR, hCDR, área-CDR, rCDR) ASOCIADOS A SOSPECHA "
    "GLAUCOMATOSA. NO es un sistema automático de diagnóstico de glaucoma."
)

LIMITACIONES = [
    "El glaucoma no se decide solo con la foto de papila ni solo con el vCDR: requiere campo "
    "visual, OCT, presión intraocular, historia clínica y exploración oftalmológica.",
    "Los falsos negativos observados sugieren casos glaucomatosos no explicables solo por la "
    "relación copa/disco.",
    "El punto operativo principal (sensitivity_at_least_0.85) prioriza la sensibilidad (uso de "
    "cribado), aceptando más falsos positivos; Youden es la alternativa más equilibrada.",
    "Validación externa en REFUGE-Validation400/Test400; no se ha validado en otras cámaras, "
    "poblaciones ni condiciones de adquisición.",
    "La localización ROI se corrigió con un localizador robusto con guarda que solo interviene "
    "cuando el método original falla, evitando reentrenar; su efecto está cuantificado en la "
    "auditoría ROI del punto 13.",
]

log_ok("ENCUADRE CLÍNICO:")
print(ENCUADRE_CLINICO)
log_info("LIMITACIONES:")
for i, lim in enumerate(LIMITACIONES, 1):
    print(f"  {i}. {lim}")


# ------------------------------------------------------------
# 14.6. Exportación de conclusiones en Markdown
# ------------------------------------------------------------
FINAL_REPORT_PATH = os.path.join(FINAL_SUMMARY_DIR, "final_report.md")


def _df_to_text(df):
    """Tabla en texto: usa to_markdown si hay tabulate; si no, to_string."""
    try:
        return df.to_markdown(index=False).split("\n")
    except Exception:
        return df.to_string(index=False).split("\n")


report_lines = ["# Resumen final del experimento", "", "## Métricas finales (Test400)", ""]
report_lines += _df_to_text(final_metrics_df)
report_lines += ["", "## Punto operativo", ""]
if operating_summary_df is not None:
    report_lines += _df_to_text(operating_summary_df)
report_lines += ["", "## Auditoría ROI", ""]
if roi_summary_df is not None:
    report_lines += _df_to_text(roi_summary_df)
report_lines += ["", "## Encuadre clínico", "", ENCUADRE_CLINICO, "", "## Limitaciones", ""]
report_lines += [f"{i}. {lim}" for i, lim in enumerate(LIMITACIONES, 1)]

try:
    with open(FINAL_REPORT_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(report_lines))
    log_ok(f"Informe final exportado en: {FINAL_REPORT_PATH}")
except Exception as exc:
    log_warning(f"No se pudo exportar el informe final: {exc}")

log_ok("Punto 14 completado. Notebook cerrado.")

# 15. Refinamiento clínico post-hoc y calibración

Este bloque **no reentrena ni reinfiere**: opera sobre los DataFrames ya calculados (`external_results_df` = Validation400, `test_results_df` = Test400). Regla metodológica estricta: **todo se ajusta en Validation400 y se aplica a Test400**; Test400 nunca se usa para ajustar.

Incluye: **(C.1)** corrección afín del sesgo residual de vCDR — la solución oficial del sesgo +0.077 detectado en la auditoría, ya que el reentreno no lo corrigió (ver sección 6); **(C.2)** score simplificado vCDR+rCDR sin ISNT, con ablación sobre la versión final; **(C.3)** calibración de probabilidad (Platt) con **diagrama de fiabilidad, Brier y ECE**; **(C.4)** gate de incertidumbre / predicción selectiva.

Nota de rigor: las correcciones C.1 y C.3 son **transformaciones monótonas** del vCDR/score, por lo que **no cambian el AUC** (mejoran MAE, interpretabilidad y calibración). La única palanca post-hoc que afecta a la discriminación operativa es el gate selectivo (C.4).

In [ ]:
# ============================================================
# 15. REFINAMIENTO CLÍNICO POST-HOC (C.1 - C.3)
# Ajuste en Validation400, aplicación a Test400. No reinfiere.
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, confusion_matrix

# Directorio de calibración (definido en la Sección 11; fallback por robustez).
CALIBRATION_DIR = globals().get("CALIBRATION_DIR", os.path.join(CFG.SAVE_PATH, "calibration"))
os.makedirs(CALIBRATION_DIR, exist_ok=True)


def _load_df(name, path):
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        return globals()[name].copy()
    return pd.read_csv(path)


val_df = _load_df("external_results_df", os.path.join(CFG.SAVE_PATH, "external_validation_results.csv"))
test_df = _load_df("test_results_df", os.path.join(CFG.SAVE_PATH, "test_validation_results.csv"))

LABEL = "true_glaucoma_label"
for df in (val_df, test_df):
    df.dropna(subset=[LABEL], inplace=True)
    df[LABEL] = df[LABEL].astype(int)


def safe_auc(y, score):
    """AUC robusto a NaN: enmascara pares no finitos antes de evaluar."""
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(y) & np.isfinite(score)
    if mask.sum() == 0 or len(np.unique(y[mask].astype(int))) < 2:
        return float("nan")
    return float(roc_auc_score(y[mask].astype(int), score[mask]))


def minmax_clip_v5(s, low, high):
    return np.clip((np.asarray(s, dtype=float) - low) / (high - low), 0.0, 1.0)


def threshold_at_sensitivity(y, score, target=0.85):
    """Umbral con sensibilidad >= target y la mayor especificidad posible (fijado en Val)."""
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(y) & np.isfinite(score)
    y, score = y[mask], score[mask]
    fpr, tpr, thr = roc_curve(y, score)
    idx = np.where(tpr >= target)[0]
    best = idx[np.argmin(fpr[idx])]
    return float(thr[best])


def clf_metrics(y, score, threshold):
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(y) & np.isfinite(score)
    y, score = y[mask].astype(int), score[mask]
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return dict(threshold=threshold, sensitivity=sens, specificity=spec,
                tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
                auc=safe_auc(y, score))


def mae(df, col):
    m = df.dropna(subset=[col, "true_vCDR"])
    return float(np.mean(np.abs(m[col] - m["true_vCDR"])))


# ----- C.1. Corrección afín del bias de vCDR (fit en Val400) -----
# Esta es la solución OFICIAL del sesgo +0.077 detectado en la auditoría:
# el reentreno Tversky no lo corrigió (la copa usa pesos recall-oriented; ver
# sección 6), por lo que el sesgo se corrige aquí post-hoc, ajustando en
# Validation400 y aplicando a Test400. Es una transformación monótona => mejora
# MAE/interpretabilidad/calibración, NO el AUC.
v = val_df.dropna(subset=["pred_vCDR", "true_vCDR"])
reg = LinearRegression().fit(v[["pred_vCDR"]].values, v["true_vCDR"].values)
a = float(reg.coef_[0]); b = float(reg.intercept_)
for df in (val_df, test_df):
    df["pred_vCDR_corr"] = a * df["pred_vCDR"].values + b

print("[C.1] Corrección afín de vCDR (fit Val400): pred_corr = %.4f * pred + %.4f" % (a, b))
print("      Bias medio (pred-true)  Val: %+.4f -> corr: %+.4f" % (
    float((val_df["pred_vCDR"] - val_df["true_vCDR"]).mean()),
    float((val_df["pred_vCDR_corr"] - val_df["true_vCDR"]).mean())))
print("      MAE vCDR Val : %.4f -> %.4f" % (mae(val_df, "pred_vCDR"), mae(val_df, "pred_vCDR_corr")))
print("      MAE vCDR Test: %.4f -> %.4f" % (mae(test_df, "pred_vCDR"), mae(test_df, "pred_vCDR_corr")))
print("      AUC test vCDR crudo=%.4f | corregido=%.4f  (≈invariante: corrección monótona)" % (
    safe_auc(test_df[LABEL], test_df["pred_vCDR"]),
    safe_auc(test_df[LABEL], test_df["pred_vCDR_corr"])))


# ----- C.2. Score simplificado vCDR+rCDR (sin ISNT) + ablación -----
def build_scores(df):
    vc = minmax_clip_v5(df["pred_vCDR_corr"], 0.45, 0.85)
    out = {"vcdr_only_corr": vc}
    if "pred_rCDR" in df.columns:
        rc = minmax_clip_v5(df["pred_rCDR"], 0.50, 0.85)
        out["simplified_vcdr_rcdr"] = 0.78 * vc + 0.22 * rc  # renormaliza 0.70/0.20 quitando ISNT
    return out


val_scores = build_scores(val_df)
test_scores = build_scores(test_df)

print("\n[C.2] Ablación de scores en la versión final (AUC):")
for name in val_scores:
    print("      %-22s Val=%.4f | Test=%.4f" % (
        name, safe_auc(val_df[LABEL], val_scores[name]), safe_auc(test_df[LABEL], test_scores[name])))
if "risk_score_combined" in test_df.columns:
    print("      %-22s Test=%.4f (score combinado actual, con ISNT)" % (
        "risk_score_combined", safe_auc(test_df[LABEL], test_df["risk_score_combined"])))

FINAL = "simplified_vcdr_rcdr" if "simplified_vcdr_rcdr" in val_scores else "vcdr_only_corr"
thr = threshold_at_sensitivity(val_df[LABEL].values, val_scores[FINAL], 0.85)
m_val = clf_metrics(val_df[LABEL].values, val_scores[FINAL], thr)
m_test = clf_metrics(test_df[LABEL].values, test_scores[FINAL], thr)
print("\n      Score final: %s | umbral(sens>=0.85 en Val)=%.4f" % (FINAL, thr))
print("      Val : sens=%.3f spec=%.3f" % (m_val["sensitivity"], m_val["specificity"]))
print("      Test: sens=%.3f spec=%.3f (AUC=%.4f)" % (
    m_test["sensitivity"], m_test["specificity"], m_test["auc"]))


# ----- C.3. Calibración de probabilidad (Platt, fit Val400) + fiabilidad/Brier/ECE -----
# Robustez a NaN: algunas imágenes pueden no producir vCDR/rCDR (segmentación
# fallida), lo que propaga NaN al score. Se ajusta Platt solo con muestras
# finitas de Val400 y se evalúa solo con muestras finitas de Test400.
_val_score = np.asarray(val_scores[FINAL], dtype=float)
_val_y = val_df[LABEL].values.astype(int)
_val_fit_mask = np.isfinite(_val_score)
platt = LogisticRegression().fit(_val_score[_val_fit_mask].reshape(-1, 1), _val_y[_val_fit_mask])

_test_score = np.asarray(test_scores[FINAL], dtype=float)
_test_y = test_df[LABEL].values.astype(int)
_test_mask = np.isfinite(_test_score)
test_proba = np.full(_test_score.shape, np.nan, dtype=float)
test_proba[_test_mask] = platt.predict_proba(_test_score[_test_mask].reshape(-1, 1))[:, 1]
test_df["final_proba_calibrated"] = test_proba
_n_nan_test = int((~_test_mask).sum())
if _n_nan_test:
    log_warning(f"[C.3] {_n_nan_test} imágenes de test con score NaN excluidas de la calibración.")


def reliability_bins(y, p, bins=10):
    """Devuelve (conf, acc, count) por bin de probabilidad para el diagrama de fiabilidad."""
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    conf, acc, cnt = [], [], []
    for i in range(bins):
        m = (p >= edges[i]) & ((p < edges[i + 1]) if i < bins - 1 else (p <= edges[i + 1]))
        if m.sum() > 0:
            conf.append(p[m].mean()); acc.append(y[m].mean()); cnt.append(int(m.sum()))
        else:
            conf.append(np.nan); acc.append(np.nan); cnt.append(0)
    return np.array(conf), np.array(acc), np.array(cnt)


def ece(y, p, bins=10):
    conf, acc, cnt = reliability_bins(y, p, bins)
    total = cnt.sum()
    if total == 0:
        return float("nan")
    valid = cnt > 0
    return float(np.sum(np.abs(conf[valid] - acc[valid]) * cnt[valid]) / total)


# Evaluación de calibración solo sobre muestras finitas de Test400.
y_eval = _test_y[_test_mask]
raw_eval = np.clip(_test_score[_test_mask], 0.0, 1.0)
cal_eval = test_proba[_test_mask]
brier_raw = brier_score_loss(y_eval, raw_eval)
brier_cal = brier_score_loss(y_eval, cal_eval)
ece_raw = ece(y_eval, raw_eval)
ece_cal = ece(y_eval, cal_eval)
print("\n[C.3] Calibración Platt (fit Val400):")
print("      Brier test crudo=%.4f -> calibrado=%.4f" % (brier_raw, brier_cal))
print("      ECE   test crudo=%.4f -> calibrado=%.4f  (AUC invariante: calibración monótona)" % (ece_raw, ece_cal))

# Diagrama de fiabilidad (reliability diagram).
conf_r, acc_r, cnt_r = reliability_bins(y_eval, raw_eval)
conf_c, acc_c, cnt_c = reliability_bins(y_eval, cal_eval)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Calibración perfecta")
ax.plot(conf_r, acc_r, "o-", color="tab:orange", label="Score crudo (Brier=%.3f, ECE=%.3f)" % (brier_raw, ece_raw))
ax.plot(conf_c, acc_c, "s-", color="tab:blue", label="Platt calibrado (Brier=%.3f, ECE=%.3f)" % (brier_cal, ece_cal))
ax.set_xlabel("Probabilidad media predicha")
ax.set_ylabel("Frecuencia observada de glaucoma")
ax.set_title("Diagrama de fiabilidad — Test400")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.3)
reliability_path = os.path.join(CALIBRATION_DIR, "reliability_diagram_test.png")
fig.tight_layout(); fig.savefig(reliability_path, dpi=120); plt.show(); plt.close(fig)

reliability_table = pd.DataFrame({
    "bin_conf_raw": conf_r, "bin_acc_raw": acc_r, "bin_count_raw": cnt_r,
    "bin_conf_cal": conf_c, "bin_acc_cal": acc_c, "bin_count_cal": cnt_c,
})
reliability_table.to_csv(os.path.join(CALIBRATION_DIR, "reliability_bins_test.csv"), index=False)
calibration_summary = pd.DataFrame([
    {"metric": "Brier", "crudo": brier_raw, "calibrado": brier_cal},
    {"metric": "ECE", "crudo": ece_raw, "calibrado": ece_cal},
])
calibration_summary.to_csv(os.path.join(CALIBRATION_DIR, "calibration_summary_test.csv"), index=False)
print("      Diagrama de fiabilidad guardado en:", reliability_path)

# ============================================================
# 15. (C.4) GATE DE INCERTIDUMBRE + GUARDADO DEL REFINAMIENTO
# ============================================================
# ----- C.4. Predicción selectiva (umbral de entropía fijado en Val400) -----
ENT = "quality_mean_entropy"
if ENT in val_df.columns and ENT in test_df.columns:
    ent_thr = float(val_df[ENT].quantile(0.90))  # abstener el ~10% más incierto
    retained = (test_df[ENT] <= ent_thr).values
    yv = test_df[LABEL].values
    sc = test_scores[FINAL]
    m_all = clf_metrics(yv, sc, thr)
    m_ret = clf_metrics(yv[retained], sc[retained], thr)
    print("[C.4] Gate de incertidumbre (entropía > %.4f -> revisión manual; umbral fijado en Val400)" % ent_thr)
    print("      Cobertura auto-decidida Test: %.1f%% (%d/%d) | abstenidos=%d" % (
        100 * retained.mean(), int(retained.sum()), len(test_df), int((~retained).sum())))
    print("      Test completo     : sens=%.3f spec=%.3f" % (m_all["sensitivity"], m_all["specificity"]))
    print("      Test auto-decidido: sens=%.3f spec=%.3f" % (m_ret["sensitivity"], m_ret["specificity"]))
    abst = test_df[~retained]
    print("      Composición abstenidos: glaucoma=%d, sanos=%d" % (
        int((abst[LABEL] == 1).sum()), int((abst[LABEL] == 0).sum())))
else:
    print("[C.4] Sin columna de entropía; gate omitido.")

# ----- Resumen del refinamiento post-hoc de la versión final (sin comparación de versiones) -----
posthoc_summary = {
    "score_final": FINAL,
    "umbral_sens>=0.85_val": float(thr),
    "AUC_test_final": float(m_test["auc"]),
    "sens_test_final": float(m_test["sensitivity"]),
    "spec_test_final": float(m_test["specificity"]),
    "MAE_vCDR_test_crudo": mae(test_df, "pred_vCDR"),
    "MAE_vCDR_test_corregido_C1": mae(test_df, "pred_vCDR_corr"),
    "brier_test_crudo": float(brier_raw),
    "brier_test_calibrado": float(brier_cal),
    "ece_test_crudo": float(ece_raw),
    "ece_test_calibrado": float(ece_cal),
}
posthoc_df = pd.DataFrame([posthoc_summary]).T.reset_index()
posthoc_df.columns = ["métrica", "valor"]
print("\n=== Resumen del refinamiento post-hoc (versión final) ===")
print(posthoc_df.to_string(index=False))

out_dir = os.path.join(CFG.SAVE_PATH, "posthoc_refinement")
os.makedirs(out_dir, exist_ok=True)
posthoc_df.to_csv(os.path.join(out_dir, "posthoc_summary.csv"), index=False)
test_df.to_csv(os.path.join(out_dir, "test_results_posthoc.csv"), index=False)
print("\n[OK] Refinamiento post-hoc guardado en:", out_dir)


# 16. Figuras para la memoria del TFG

Este bloque final genera y guarda en alta resolución (300 dpi) las figuras de la memoria, en la carpeta `figures/` dentro del proyecto en Drive (`CFG.DRIVE_BASE`). No reentrena ni reinfiere salvo la **Figura 2**, que re-ejecuta la inferencia visual sobre tres casos representativos reutilizando los memmaps de test y `get_test_result_visual_by_index()` de la sección 13.

Se generan cuatro figuras:

- **Figura 1 — Curva ROC con techo GT-vCDR** (sección 4.8.2): score combinado frente al techo interno calculado con el vCDR de la anotación experta, con el punto operativo seleccionado en REFUGE-Validation400.
- **Figura 2 — Panel de segmentación** (sección 4.8.1): tres casos representativos (mejor verdadero positivo, falso negativo y mayor error de vCDR) con retinografía, anotación experta y predicción.
- **Figura 3 — Sesgo del vCDR y corrección afín** (sección 4.7.2): dispersión predicho/real y distribución de residuos antes y después de la corrección afín.
- **Figura 7 — Curvas de aprendizaje por pliegue** (sección 4.5): `clinical_selection_score` de entrenamiento y validación interna por fold, con el mejor checkpoint marcado.

Cada figura está aislada en su propio bloque `try/except`: si una falla (por ejemplo, por falta de los memmaps en memoria), las demás se siguen generando.


In [ ]:
# ============================================================
# 16. FIGURAS PARA LA MEMORIA DEL TFG
# ============================================================
# Genera y guarda en alta resolucion (300 dpi) las figuras de la memoria.
# Reutiliza los DataFrames y funciones ya definidos en el notebook:
#   - test_results_df / external_results_df (etiqueta real: true_glaucoma_label)
#   - get_test_result_visual_by_index() y create_overlay() (seccion 13)
#   - CFG.SAVE_PATH (logs de entrenamiento) y FINAL_THRESHOLD (seccion 12)
# Cada figura va en su propio bloque try/except: si una falla, el resto
# se siguen generando.

import os
import glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# ------------------------------------------------------------
# Carpeta de figuras (en Drive, junto al resto del proyecto)
# ------------------------------------------------------------
FIGURES_DIR = os.path.join(CFG.DRIVE_BASE, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
log_ok(f"Carpeta de figuras: {FIGURES_DIR}")

# Estilo y paleta consistentes para toda la memoria.
matplotlib.rcParams.update({"font.size": 11, "font.family": "DejaVu Sans"})
C_GREEN  = "#1D9E75"   # sistema implementado
C_GRAY   = "#888780"   # referencia / techo
C_ORANGE = "#D85A30"
C_BLUE   = "#3B8BD4"
C_PURPLE = "#7F77DD"

LABEL_COL = "true_glaucoma_label"   # nombre real de la etiqueta en este notebook


def _save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    log_ok(f"{name} guardada en {path}")


# ------------------------------------------------------------
# Figura 1 - Curva ROC con techo GT-vCDR superpuesto (seccion 4.8.2)
# ------------------------------------------------------------
try:
    df_roc = test_results_df.dropna(
        subset=[LABEL_COL, "risk_score_combined", "true_vCDR"]
    ).copy()
    y_true  = df_roc[LABEL_COL].astype(int).values
    y_score = df_roc["risk_score_combined"].astype(float).values
    y_vcdr  = df_roc["true_vCDR"].astype(float).values

    fpr_s, tpr_s, _ = roc_curve(y_true, y_score)
    fpr_g, tpr_g, _ = roc_curve(y_true, y_vcdr)
    auc_s = roc_auc_score(y_true, y_score)
    auc_g = roc_auc_score(y_true, y_vcdr)

    # Punto operativo seleccionado en REFUGE-Validation400 (documentado).
    op_sens, op_spec = 0.800, 0.753
    op_fpr = 1 - op_spec

    fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=300)
    ax.plot([0, 1], [0, 1], "k--", lw=0.7, alpha=0.35, label="Azar (AUC = 0,500)")
    ax.plot(fpr_g, tpr_g, color=C_GRAY, lw=1.5, linestyle="--",
            label=f"Techo GT-vCDR (AUC = {auc_g:.3f})")
    ax.plot(fpr_s, tpr_s, color=C_GREEN, lw=2,
            label=f"Score combinado (AUC = {auc_s:.3f})")
    ax.scatter([op_fpr], [op_sens], color=C_GREEN, s=70, zorder=6,
               label=f"Punto operativo (sens {op_sens:.3f} / spec {op_spec:.3f})")

    ax.set_xlabel("Tasa de falsos positivos (1 - especificidad)")
    ax.set_ylabel("Tasa de verdaderos positivos (sensibilidad)")
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1]); ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.12, lw=0.5)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)

    fig.tight_layout()
    _save_fig(fig, "fig1_roc_curve.png")
    plt.show()
except Exception as exc:
    log_error(f"[Figura 1] No se pudo generar: {exc}")


# ------------------------------------------------------------
# Figura 3 - Sesgo del vCDR y correccion afin (seccion 4.7.2)
# ------------------------------------------------------------
try:
    pred = external_results_df["pred_vCDR"].values.astype(float)
    true = external_results_df["true_vCDR"].values.astype(float)
    mask = ~(np.isnan(pred) | np.isnan(true))
    pred, true = pred[mask], true[mask]

    # Reutiliza los coeficientes del refinamiento post-hoc (seccion 15) si ya
    # se ejecuto; si no, los recalcula sobre Validation400 (fit true ~ a*pred + b).
    if "a" in globals() and "b" in globals():
        a_fit, b_fit = float(a), float(b)
    else:
        _slope, _intercept = np.polyfit(pred, true, 1)
        a_fit, b_fit = float(_slope), float(_intercept)

    pred_corr = a_fit * pred + b_fit
    res_raw  = pred - true
    res_corr = pred_corr - true

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5), dpi=300)
    fig.suptitle("Sesgo del vCDR y correccion afin - REFUGE-Validation400", fontsize=11)

    # Panel izquierdo: dispersion + recta de correccion.
    ax1.scatter(pred, true, alpha=0.35, s=12, color=C_GRAY,
                label=f"Predicciones (n={len(pred)})")
    ax1.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4, label="Prediccion perfecta")
    x_line = np.array([0.2, 0.9])
    ax1.plot(x_line, a_fit * x_line + b_fit, color=C_GREEN, lw=2,
             label=f"Correccion afin\ny = {a_fit:.4f}*x + {b_fit:.4f}")
    ax1.axvline(pred.mean(), color=C_ORANGE, lw=0.8, linestyle=":", alpha=0.6,
                label=f"Media pred ({pred.mean():.3f})")
    ax1.axhline(true.mean(), color=C_BLUE, lw=0.8, linestyle=":", alpha=0.6,
                label=f"Media real ({true.mean():.3f})")
    ax1.set_xlabel("vCDR predicho"); ax1.set_ylabel("vCDR real (GT)")
    ax1.set_xlim([0, 1]); ax1.set_ylim([0, 1]); ax1.set_aspect("equal")
    ax1.legend(fontsize=7.5, loc="upper left"); ax1.grid(alpha=0.1, lw=0.5)
    for s in ["top", "right"]:
        ax1.spines[s].set_visible(False)

    # Panel derecho: distribucion de residuos antes/despues.
    bins = 30
    ax2.hist(res_raw,  bins=bins, alpha=0.6, color=C_GRAY,
             label=f"Crudo (sesgo = {res_raw.mean():+.3f})")
    ax2.hist(res_corr, bins=bins, alpha=0.6, color=C_GREEN,
             label=f"Corregido (sesgo = {res_corr.mean():+.3f})")
    ax2.axvline(0, color="k", lw=1, linestyle="--", alpha=0.5)
    ax2.set_xlabel("Residuo (predicho - real)"); ax2.set_ylabel("Frecuencia")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.1, lw=0.5)
    for s in ["top", "right"]:
        ax2.spines[s].set_visible(False)

    fig.tight_layout()
    _save_fig(fig, "fig3_vcdr_scatter.png")
    plt.show()
except Exception as exc:
    log_error(f"[Figura 3] No se pudo generar: {exc}")


# ------------------------------------------------------------
# Figura 7 - Curvas de aprendizaje por pliegue (seccion 4.5)
# ------------------------------------------------------------
try:
    fig, axes = plt.subplots(1, CFG.N_SPLITS, figsize=(15, 3.8), dpi=300, sharey=True)
    fig.suptitle("Curvas de aprendizaje por pliegue - clinical_selection_score", fontsize=11)
    colors = [C_GREEN, C_BLUE, C_ORANGE, C_PURPLE, C_GRAY]

    for fold in range(1, CFG.N_SPLITS + 1):
        ax = axes[fold - 1]
        color = colors[(fold - 1) % len(colors)]

        # El CSVLogger del notebook guarda log_fold_{n}.csv en CFG.SAVE_PATH.
        patterns = [
            os.path.join(CFG.SAVE_PATH, f"log_fold_{fold}.csv"),
            os.path.join(CFG.SAVE_PATH, f"fold_{fold}", "training_log.csv"),
            os.path.join(CFG.SAVE_PATH, f"training_log_fold{fold}.csv"),
        ]
        csv_path = None
        for pat in patterns:
            matches = glob.glob(pat)
            if matches:
                csv_path = matches[0]
                break

        if csv_path and os.path.exists(csv_path):
            dfl = pd.read_csv(csv_path)
            epoch_col = "epoch" if "epoch" in dfl.columns else dfl.columns[0]
            val_cols = [c for c in dfl.columns
                        if "val" in c and "clinical" in c and "selection" in c]
            trn_cols = [c for c in dfl.columns
                        if "clinical" in c and "selection" in c and "val" not in c]

            if val_cols:
                ax.plot(dfl[epoch_col], dfl[val_cols[0]], color=color, lw=2,
                        label="Val. interna")
            if trn_cols:
                ax.plot(dfl[epoch_col], dfl[trn_cols[0]], color=color, lw=1,
                        linestyle="--", alpha=0.55, label="Entrenamiento")

            # Mejor checkpoint = epoca de maximo val_clinical_selection_score
            # (ModelCheckpoint usa save_best_only con mode=max).
            if val_cols:
                best_i = int(dfl[val_cols[0]].idxmax())
                row = dfl.iloc[best_i]
                ax.scatter([row[epoch_col]], [row[val_cols[0]]], color=color,
                           s=50, zorder=6)
                ax.axvline(row[epoch_col], color="gray", lw=0.6, linestyle=":", alpha=0.5)
        else:
            ax.text(0.5, 0.5, f"CSV no encontrado\nFold {fold}", ha="center",
                    va="center", transform=ax.transAxes, fontsize=9, color="gray")

        ax.set_title(f"Fold {fold}", fontsize=10)
        ax.set_xlabel("Epoca", fontsize=9)
        if fold == 1:
            ax.set_ylabel("clinical_selection_score", fontsize=9)
        ax.set_xlim([0, CFG.EPOCHS + 2]); ax.grid(alpha=0.1, lw=0.5)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right", fontsize=9, ncol=2,
                   bbox_to_anchor=(1.0, 1.0))
    fig.tight_layout()
    _save_fig(fig, "fig7_learning_curves.png")
    plt.show()
except Exception as exc:
    log_error(f"[Figura 7] No se pudo generar: {exc}")


# ------------------------------------------------------------
# Figura 2 - Panel de segmentacion de casos representativos (seccion 4.8.1)
# ------------------------------------------------------------
# Requiere los memmaps de test en memoria (seccion 12) y la funcion
# get_test_result_visual_by_index() (seccion 13). Re-ejecuta la inferencia
# visual sobre los tres casos seleccionados.
try:
    _need = ["test_data", "test_pred_memmap", "test_true_masks_memmap",
             "get_test_result_visual_by_index", "create_overlay"]
    _missing = [g for g in _need if g not in globals()]
    if _missing:
        raise RuntimeError(
            f"Faltan elementos en memoria: {_missing}. "
            "Ejecuta las secciones 12 y 13 antes de esta celda."
        )

    # Umbral final del sistema.
    if "FINAL_THRESHOLD" in globals():
        thr = float(FINAL_THRESHOLD)
    else:
        thr = float(test_results_df["final_threshold"].iloc[0])

    base = test_results_df.dropna(
        subset=["risk_score_combined", "abs_error_vCDR", "true_vCDR", "pred_vCDR"]
    ).copy()

    # Verdadero positivo con menor error de vCDR.
    tp_df = base[(base[LABEL_COL] == 1) & (base["risk_score_combined"] >= thr)]
    best_tp = tp_df.loc[tp_df["abs_error_vCDR"].idxmin()]
    # Falso negativo con el score mas bajo (error mas confiado).
    fn_df = base[(base[LABEL_COL] == 1) & (base["risk_score_combined"] < thr)]
    worst_fn = fn_df.loc[fn_df["risk_score_combined"].idxmin()] if len(fn_df) else best_tp
    # Mayor error de vCDR en cualquier clase.
    worst_vcdr = base.loc[base["abs_error_vCDR"].idxmax()]

    panel = [
        ("Verdadero positivo\n(min. error vCDR)", best_tp),
        ("Falso negativo\n(glaucoma no detectado)", worst_fn),
        ("Mayor error de vCDR", worst_vcdr),
    ]

    def _memmap_index(row):
        return int(row["index"]) if "index" in row.index else int(row.name)

    fig, axes = plt.subplots(3, 3, figsize=(10.5, 11), dpi=300)
    col_titles = ["Retinografia (ROI)", "Anotacion experta (GT)", "Prediccion del sistema"]
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontsize=11)

    for i, (label, row) in enumerate(panel):
        res = get_test_result_visual_by_index(_memmap_index(row))
        image = res["image_resized"]
        overlay_true = create_overlay(image, res["true_mask"], alpha=0.35)
        overlay_pred = create_overlay(image, res["final_mask"], alpha=0.35)

        for j, im in enumerate([image, overlay_true, overlay_pred]):
            axes[i, j].imshow(im)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

        # Etiqueta del caso a la izquierda de la fila.
        axes[i, 0].set_ylabel(label, fontsize=9, rotation=0, ha="right",
                              va="center", labelpad=18)

        # Cifras clinicas bajo la prediccion.
        suspect = "SOSPECHA" if row["risk_score_combined"] >= thr else "no sospecha"
        axes[i, 2].set_xlabel(
            f"vCDR pred {row['pred_vCDR']:.3f} / real {row['true_vCDR']:.3f}\n"
            f"score {row['risk_score_combined']:.3f} -> {suspect}",
            fontsize=8.5,
        )

    fig.suptitle("Segmentacion en casos representativos - REFUGE-Test400", fontsize=12)
    fig.tight_layout(rect=[0.07, 0, 1, 0.97])
    _save_fig(fig, "fig2_segmentation_panel.png")
    plt.show()

    # Resumen numerico de los casos del panel.
    for label, row in panel:
        log_info(
            f"{label.splitlines()[0]} | idx {_memmap_index(row)} | "
            f"vCDR pred/real {row['pred_vCDR']:.3f}/{row['true_vCDR']:.3f} | "
            f"err {row['abs_error_vCDR']:.3f} | score {row['risk_score_combined']:.3f} | "
            f"clase {'Glaucoma' if int(row[LABEL_COL]) == 1 else 'Sano'}"
        )
except Exception as exc:
    log_error(f"[Figura 2] No se pudo generar: {exc}")


log_ok(f"Proceso de figuras finalizado. Disponibles en: {FIGURES_DIR}")
